# Phase 4 — Historical Feature Engineering

## Purpose

This notebook provides reproducible evidence for the historical feature-engineering phase of the Student Learning State Identification component.

The validated processed datasets from Phase 3 are used as the input foundation:

- `data/processed/base_interactions.csv`
- `data/processed/concept_interactions.csv`

The objective of this phase is to transform interaction history into meaningful learning-state evidence.

Candidate feature categories include:

- recent correctness and accuracy,
- correct and incorrect counts,
- hint behaviour,
- attempt behaviour,
- response-time behaviour,
- concept-specific performance,
- general student performance,
- previous-vs-recent trend information,
- short-term versus broader recent performance,
- concept-history length,
- overall student-history length,
- and evidence-availability indicators.

The main design principle is:

**Features for a current interaction must be calculated only from information that would have been available before that interaction.**

This prevents future information from leaking into the student's current-state prediction.

No learning-state labels, thresholds, classifier training, or final state rules are created in this notebook until the historical features themselves are validated.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd().parent

BASE_INTERACTIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "base_interactions.csv"
)

CONCEPT_INTERACTIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "concept_interactions.csv"
)

base_df = pd.read_csv(
    BASE_INTERACTIONS_PATH,
    low_memory=False
)

concept_df = pd.read_csv(
    CONCEPT_INTERACTIONS_PATH,
    low_memory=False
)

print("Base Interaction Dataset")
print("Shape:", base_df.shape)
print(
    "Unique students:",
    base_df["user_id"].nunique()
)

print()

print("Concept Interaction Dataset")
print("Shape:", concept_df.shape)
print(
    "Unique students:",
    concept_df["user_id"].nunique()
)
print(
    "Unique skills:",
    concept_df["skill_id"].nunique()
)

Base Interaction Dataset
Shape: (346860, 22)
Unique students: 4217

Concept Interaction Dataset
Shape: (338001, 27)
Unique students: 4163
Unique skills: 123


## 1. Historical Interaction Ordering

Historical features must be calculated using only information available before the current interaction.

Therefore, a reliable ordering of student interactions must be established before rolling or historical features are constructed.

The processed dataset does not contain a conventional timestamp for every interaction. Candidate ordering information available in the dataset includes:

- `order_id`
- `sequence_id`
- `base_sequence_id`
- `position`
- `opportunity_start`
- `opportunity_end`

This section investigates whether `order_id` provides a consistent interaction sequence within each student and how it relates to the other structural ordering fields.

No historical features are calculated until the ordering strategy is validated.

In [2]:
print("BASE ORDERING FIELDS")

for column in [
    "order_id",
    "sequence_id",
    "base_sequence_id",
    "position"
]:
    print(
        f"{column}: "
        f"missing={base_df[column].isna().sum()}, "
        f"unique={base_df[column].nunique()}"
    )

print()
print("CONCEPT ORDERING FIELDS")

for column in [
    "order_id",
    "sequence_id",
    "base_sequence_id",
    "position",
    "opportunity_start",
    "opportunity_end"
]:
    print(
        f"{column}: "
        f"missing={concept_df[column].isna().sum()}, "
        f"unique={concept_df[column].nunique()}"
    )

print()

# Check whether order_id is unique inside each student's
# Base Interaction history.
student_order_duplicates = (
    base_df.duplicated(
        subset=["user_id", "order_id"]
    ).sum()
)

print(
    "Duplicate user_id + order_id in Base:",
    student_order_duplicates
)

print()

# Number of interactions per student
student_history_lengths = (
    base_df.groupby("user_id").size()
)

print("Base student-history length summary:")
print(
    student_history_lengths
    .describe()
    .round(2)
)

print()

# Show several example histories sorted by order_id.
example_students = (
    student_history_lengths
    .sort_values(ascending=False)
    .head(3)
    .index
)

for user_id in example_students:

    print("=" * 70)
    print("Student:", user_id)

    example_history = (
        base_df[
            base_df["user_id"] == user_id
        ][
            [
                "order_id",
                "assignment_id",
                "problem_id",
                "sequence_id",
                "base_sequence_id",
                "position",
                "correct"
            ]
        ]
        .sort_values("order_id")
        .head(20)
    )

    print(
        example_history.to_string(
            index=False
        )
    )
    print()

BASE ORDERING FIELDS
order_id: missing=0, unique=346860
sequence_id: missing=0, unique=677
base_sequence_id: missing=0, unique=366
position: missing=0, unique=283

CONCEPT ORDERING FIELDS
order_id: missing=0, unique=283105
sequence_id: missing=0, unique=620
base_sequence_id: missing=0, unique=311
position: missing=0, unique=283
opportunity_start: missing=0, unique=932
opportunity_end: missing=0, unique=957

Duplicate user_id + order_id in Base: 0

Base student-history length summary:
count    4217.00
mean       82.25
std       163.68
min         1.00
25%        10.00
50%        23.00
75%        74.00
max      1383.00
dtype: float64

Student: 78978
 order_id  assignment_id  problem_id  sequence_id  base_sequence_id  position  correct
 20857587         262758       49280         5924              5924         8        1
 20863960         262782       84907         7012              7012         9        1
 20864017         262779       53975         5961              5961        10      

### 1.1 Validation of `order_id` as a Historical Ordering Key

Example student histories suggest that increasing `order_id` follows the progression of interactions through the dataset.

However, example histories alone are insufficient to establish an ordering rule.

To validate `order_id` as a historical ordering key, every student's Base Interaction history is sorted by `order_id`. The corresponding `position` values are then checked for backward movement.

Repeated `position` values are allowed because a student may complete multiple problem interactions within the same structural position.

The important condition is whether increasing `order_id` causes `position` to decrease within a student's history.

In [3]:
ordered_base = (
    base_df
    .sort_values(
        ["user_id", "order_id"]
    )
    .copy()
)

ordered_base["previous_position"] = (
    ordered_base
    .groupby("user_id")["position"]
    .shift(1)
)

ordered_base["position_change"] = (
    ordered_base["position"]
    - ordered_base["previous_position"]
)

backward_position = (
    ordered_base[
        ordered_base["position_change"] < 0
    ]
)

same_position = (
    ordered_base[
        ordered_base["position_change"] == 0
    ]
)

forward_position = (
    ordered_base[
        ordered_base["position_change"] > 0
    ]
)

comparable_rows = (
    ordered_base["previous_position"]
    .notna()
    .sum()
)

print(
    "Comparable consecutive interactions:",
    comparable_rows
)

print()

print(
    "Forward position transitions:",
    len(forward_position)
)

print(
    "Same-position transitions:",
    len(same_position)
)

print(
    "Backward position transitions:",
    len(backward_position)
)

print()

print(
    "Students with at least one backward position transition:",
    backward_position["user_id"].nunique()
)

print()

print("Position-change summary:")
print(
    ordered_base.loc[
        ordered_base["previous_position"].notna(),
        "position_change"
    ]
    .describe()
    .round(2)
)

print()

if len(backward_position) > 0:

    print("Example backward transitions:")

    print(
        backward_position[
            [
                "user_id",
                "order_id",
                "assignment_id",
                "problem_id",
                "previous_position",
                "position",
                "position_change"
            ]
        ]
        .head(20)
        .to_string(index=False)
    )

Comparable consecutive interactions: 342643

Forward position transitions: 53984
Same-position transitions: 271329
Backward position transitions: 17330

Students with at least one backward position transition: 2006

Position-change summary:
count    342643.00
mean          0.19
std          22.72
min        -276.00
25%           0.00
50%           0.00
75%           0.00
max         255.00
Name: position_change, dtype: float64

Example backward transitions:
 user_id  order_id  assignment_id  problem_id  previous_position  position  position_change
   52613  29994521         220683       66746              162.0        31           -131.0
   52613  36259010         285317      161807               31.0         1            -30.0
   53167  24605927         263326       90437               16.0         4            -12.0
   53167  24606065         263329       60006               17.0         5            -12.0
   53167  24606174         263320       54134               14.0        12    

### 1.2 Context of Backward Position Transitions

Sorting interactions by `order_id` revealed backward movements in `position` for many students.

This does not necessarily invalidate `order_id` as an ordering key because `position` may represent a position within an assignment, sequence, or other curriculum structure rather than a global student timeline.

Therefore, backward position transitions are examined to determine whether they coincide with changes in:

- assignment,
- sequence,
- or base sequence.

If position commonly decreases when these structural identifiers change, `position` should not be used as the primary global historical ordering field.

In [4]:
ordered_base["previous_assignment_id"] = (
    ordered_base
    .groupby("user_id")["assignment_id"]
    .shift(1)
)

ordered_base["previous_sequence_id"] = (
    ordered_base
    .groupby("user_id")["sequence_id"]
    .shift(1)
)

ordered_base["previous_base_sequence_id"] = (
    ordered_base
    .groupby("user_id")["base_sequence_id"]
    .shift(1)
)


backward_position = (
    ordered_base[
        ordered_base["position_change"] < 0
    ]
    .copy()
)


backward_position["assignment_changed"] = (
    backward_position["assignment_id"]
    != backward_position["previous_assignment_id"]
)

backward_position["sequence_changed"] = (
    backward_position["sequence_id"]
    != backward_position["previous_sequence_id"]
)

backward_position["base_sequence_changed"] = (
    backward_position["base_sequence_id"]
    != backward_position["previous_base_sequence_id"]
)


print(
    "Backward position transitions:",
    len(backward_position)
)

print()

for column, label in [
    ("assignment_changed", "Assignment changed"),
    ("sequence_changed", "Sequence changed"),
    ("base_sequence_changed", "Base sequence changed")
]:

    count = backward_position[column].sum()

    print(
        f"{label}:",
        count,
        f"({count / len(backward_position) * 100:.2f}%)"
    )


print()

any_context_change = (
    backward_position[
        [
            "assignment_changed",
            "sequence_changed",
            "base_sequence_changed"
        ]
    ]
    .any(axis=1)
)

print(
    "Backward transitions with any context change:",
    any_context_change.sum()
)

print(
    "Percentage with any context change:",
    round(
        any_context_change.mean() * 100,
        2
    ),
    "%"
)

print()

same_context_backward = (
    backward_position[
        ~any_context_change
    ]
)

print(
    "Backward transitions with SAME assignment/sequence context:",
    len(same_context_backward)
)

print()

if len(same_context_backward) > 0:

    print("Examples of same-context backward transitions:")

    print(
        same_context_backward[
            [
                "user_id",
                "order_id",
                "assignment_id",
                "sequence_id",
                "base_sequence_id",
                "previous_position",
                "position",
                "position_change"
            ]
        ]
        .head(20)
        .to_string(index=False)
    )

Backward position transitions: 17330

Assignment changed: 17330 (100.00%)
Sequence changed: 17281 (99.72%)
Base sequence changed: 17007 (98.14%)

Backward transitions with any context change: 17330
Percentage with any context change: 100.0 %

Backward transitions with SAME assignment/sequence context: 0



### 1.3 Relationship Between Raw Dataset Order and `order_id`

`position` was shown to represent local assignment/sequence structure rather than a global student timeline.

The next candidate for historical ordering is `order_id`.

Because the dataset does not provide a conventional interaction timestamp, the relationship between `order_id` and the original raw-file ordering is examined.

If Base Interactions appear in increasing `order_id` order within student histories, this provides evidence that `order_id` can serve as the available interaction-order proxy.

This analysis does not assume that `order_id` is an actual timestamp. It evaluates whether it provides a consistent sequential ordering for historical feature construction.

In [5]:
raw_path = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "skill_builder_data.csv"
)

raw_order_df = pd.read_csv(
    raw_path,
    encoding="latin1",
    low_memory=False,
    usecols=[
        "order_id",
        "user_id"
    ]
)

# Preserve the original physical row order.
raw_order_df["raw_row_position"] = range(
    len(raw_order_df)
)

# One Base Interaction per order_id.
raw_base_order = (
    raw_order_df
    .groupby(
        "order_id",
        as_index=False
    )
    .agg(
        user_id=("user_id", "first"),
        first_raw_row=("raw_row_position", "min"),
        last_raw_row=("raw_row_position", "max")
    )
)

# Examine interactions in their original raw-file appearance.
raw_base_order = (
    raw_base_order
    .sort_values(
        ["user_id", "first_raw_row"]
    )
    .copy()
)

raw_base_order["previous_order_id"] = (
    raw_base_order
    .groupby("user_id")["order_id"]
    .shift(1)
)

raw_base_order["order_id_change"] = (
    raw_base_order["order_id"]
    - raw_base_order["previous_order_id"]
)

comparable = (
    raw_base_order["previous_order_id"]
    .notna()
)

increasing = (
    raw_base_order.loc[
        comparable,
        "order_id_change"
    ] > 0
)

equal = (
    raw_base_order.loc[
        comparable,
        "order_id_change"
    ] == 0
)

decreasing = (
    raw_base_order.loc[
        comparable,
        "order_id_change"
    ] < 0
)

print(
    "Comparable consecutive Base Interactions:",
    comparable.sum()
)

print()

print(
    "Increasing order_id transitions:",
    increasing.sum()
)

print(
    "Equal order_id transitions:",
    equal.sum()
)

print(
    "Decreasing order_id transitions:",
    decreasing.sum()
)

print()

decreasing_rows = (
    raw_base_order.loc[
        comparable
    ][
        raw_base_order.loc[
            comparable,
            "order_id_change"
        ] < 0
    ]
)

print(
    "Students with at least one decreasing order_id transition:",
    decreasing_rows["user_id"].nunique()
)

print()

print(
    "Percentage of transitions with increasing order_id:",
    round(
        increasing.mean() * 100,
        4
    ),
    "%"
)

if len(decreasing_rows) > 0:

    print()
    print("Example decreasing order_id transitions:")

    print(
        decreasing_rows[
            [
                "user_id",
                "previous_order_id",
                "order_id",
                "first_raw_row",
                "order_id_change"
            ]
        ]
        .head(20)
        .to_string(index=False)
    )

Comparable consecutive Base Interactions: 342643

Increasing order_id transitions: 319536
Equal order_id transitions: 0
Decreasing order_id transitions: 23107

Students with at least one decreasing order_id transition: 2907

Percentage of transitions with increasing order_id: 93.2562 %

Example decreasing order_id transitions:
 user_id  previous_order_id  order_id  first_raw_row  order_id_change
      14         21617962.0  21617915          55308            -47.0
      14         21617942.0  21617718         459208           -224.0
   21825         23944981.0  21616426         459214       -2328555.0
   52613         29994521.0  29417405         244692        -577116.0
   52613         29417405.0  21351565         288303       -8065840.0
   53167         35507298.0  26451064          19558       -9056234.0
   53167         35129136.0  35128920          47610           -216.0
   53167         35128920.0  26451143          72173       -8677777.0
   53167         26451162.0  24606012    

### 1.4 Context of Raw-Order / `order_id` Disagreement

The original raw-file order and increasing `order_id` agree for most consecutive student interactions, but 23,107 transitions show a decrease in `order_id`.

Therefore, raw-file position cannot be used as independent proof that `order_id` represents chronological time.

The disagreement is investigated further by examining whether decreasing `order_id` transitions coincide with changes in assignment, sequence, or curriculum position.

This helps determine whether the raw dataset is organized partly by structural grouping rather than purely by student interaction time.

In [6]:
ordering_context = (
    raw_base_order[
        [
            "order_id",
            "user_id",
            "first_raw_row",
            "previous_order_id",
            "order_id_change"
        ]
    ]
    .merge(
        base_df[
            [
                "order_id",
                "assignment_id",
                "sequence_id",
                "base_sequence_id",
                "position"
            ]
        ],
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        ["user_id", "first_raw_row"]
    )
    .copy()
)

for column in [
    "assignment_id",
    "sequence_id",
    "base_sequence_id",
    "position"
]:
    ordering_context[f"previous_{column}"] = (
        ordering_context
        .groupby("user_id")[column]
        .shift(1)
    )

decreasing_context = (
    ordering_context[
        ordering_context["order_id_change"] < 0
    ]
    .copy()
)

decreasing_context["assignment_changed"] = (
    decreasing_context["assignment_id"]
    != decreasing_context["previous_assignment_id"]
)

decreasing_context["sequence_changed"] = (
    decreasing_context["sequence_id"]
    != decreasing_context["previous_sequence_id"]
)

decreasing_context["base_sequence_changed"] = (
    decreasing_context["base_sequence_id"]
    != decreasing_context["previous_base_sequence_id"]
)

decreasing_context["position_changed"] = (
    decreasing_context["position"]
    != decreasing_context["previous_position"]
)

print(
    "Decreasing order_id transitions:",
    len(decreasing_context)
)

print()

for column, label in [
    ("assignment_changed", "Assignment changed"),
    ("sequence_changed", "Sequence changed"),
    ("base_sequence_changed", "Base sequence changed"),
    ("position_changed", "Position changed")
]:

    count = decreasing_context[column].sum()

    print(
        f"{label}:",
        count,
        f"({count / len(decreasing_context) * 100:.2f}%)"
    )

print()

same_assignment = (
    decreasing_context[
        ~decreasing_context["assignment_changed"]
    ]
)

print(
    "Decreasing order_id transitions within SAME assignment:",
    len(same_assignment)
)

print()

if len(same_assignment) > 0:

    print("Examples within the same assignment:")

    print(
        same_assignment[
            [
                "user_id",
                "previous_order_id",
                "order_id",
                "assignment_id",
                "previous_position",
                "position",
                "first_raw_row",
                "order_id_change"
            ]
        ]
        .head(20)
        .to_string(index=False)
    )

Decreasing order_id transitions: 23107

Assignment changed: 21395 (92.59%)
Sequence changed: 21212 (91.80%)
Base sequence changed: 20689 (89.54%)
Position changed: 21365 (92.46%)

Decreasing order_id transitions within SAME assignment: 1712

Examples within the same assignment:
 user_id  previous_order_id  order_id  assignment_id  previous_position  position  first_raw_row  order_id_change
      14         21617962.0  21617915         263601                3.0         3          55308            -47.0
   53167         35129136.0  35128920         277629              137.0       137          47610           -216.0
   54318         28025243.0  28025073         217923               24.0        24         426200           -170.0
   64525         32426427.0  32426227         277570               78.0        78         318267           -200.0
   64531         33653854.0  33653607         279793                2.0         2         237490           -247.0
   64532         34803683.0  34802963

### 1.5 `order_id` Ordering Within Assignments

Raw-file row position cannot be treated as a reliable student timeline because the file contains structural ordering differences.

A stronger test of `order_id` is therefore performed within individual student-assignment histories.

Within the same assignment, `position` provides local progression information. If interactions are sorted by `order_id`, position should not move backward within the same student-assignment history.

Repeated positions are valid because multiple problem interactions may occur at the same assignment position.

This test evaluates whether `order_id` provides a consistent local interaction sequence within assignments.

In [7]:
within_assignment = (
    base_df
    .sort_values(
        [
            "user_id",
            "assignment_id",
            "order_id"
        ]
    )
    .copy()
)

within_assignment["previous_position"] = (
    within_assignment
    .groupby(
        [
            "user_id",
            "assignment_id"
        ]
    )["position"]
    .shift(1)
)

within_assignment["position_change"] = (
    within_assignment["position"]
    - within_assignment["previous_position"]
)

comparable_within_assignment = (
    within_assignment["previous_position"]
    .notna()
)

within_backward = (
    within_assignment[
        within_assignment["position_change"] < 0
    ]
)

within_same = (
    within_assignment[
        within_assignment["position_change"] == 0
    ]
)

within_forward = (
    within_assignment[
        within_assignment["position_change"] > 0
    ]
)

print(
    "Comparable consecutive interactions within student-assignment:",
    comparable_within_assignment.sum()
)

print()

print(
    "Forward position transitions:",
    len(within_forward)
)

print(
    "Same-position transitions:",
    len(within_same)
)

print(
    "Backward position transitions:",
    len(within_backward)
)

print()

print(
    "Student-assignment histories with backward transitions:",
    within_backward[
        ["user_id", "assignment_id"]
    ]
    .drop_duplicates()
    .shape[0]
)

print()

print(
    "Percentage non-decreasing:",
    round(
        (
            (
                len(within_forward)
                + len(within_same)
            )
            / comparable_within_assignment.sum()
        )
        * 100,
        4
    ),
    "%"
)

if len(within_backward) > 0:

    print()
    print("Examples of backward transitions:")

    print(
        within_backward[
            [
                "user_id",
                "assignment_id",
                "order_id",
                "previous_position",
                "position",
                "position_change",
                "sequence_id",
                "base_sequence_id"
            ]
        ]
        .head(20)
        .to_string(index=False)
    )

Comparable consecutive interactions within student-assignment:

 280621

Forward position transitions: 0
Same-position transitions: 280621
Backward position transitions: 0

Student-assignment histories with backward transitions: 0

Percentage non-decreasing: 100.0 %


### 1.6 Global Properties of `order_id`

Previous analyses established that:

- raw CSV row position is not a reliable student chronology,
- `position` represents structural assignment/sequence information rather than a global timeline,
- and sorting by `order_id` produces no backward position movement within student-assignment histories.

Before adopting `order_id` as the available historical ordering proxy, its global structural properties are examined.

This analysis checks whether:

- every `order_id` belongs to exactly one student,
- `order_id` values are globally unique in the Base Interaction Dataset,
- and the magnitude/range of `order_id` is consistent with an interaction-level sequencing identifier.

The analysis does not claim that `order_id` is a timestamp. The goal is to determine whether it is the strongest available deterministic ordering key for leakage-safe historical feature construction.

In [8]:
print("ORDER_ID GLOBAL STRUCTURE")
print("=" * 60)

print(
    "Base interactions:",
    len(base_df)
)

print(
    "Unique order_id:",
    base_df["order_id"].nunique()
)

print(
    "Duplicate order_id:",
    base_df["order_id"].duplicated().sum()
)

print()

order_student_counts = (
    base_df
    .groupby("order_id")["user_id"]
    .nunique()
)

print(
    "order_id linked to multiple students:",
    (order_student_counts > 1).sum()
)

print()

print(
    "Minimum order_id:",
    base_df["order_id"].min()
)

print(
    "Maximum order_id:",
    base_df["order_id"].max()
)

print()

sorted_order_ids = (
    base_df["order_id"]
    .sort_values()
    .reset_index(drop=True)
)

order_id_gaps = (
    sorted_order_ids
    .diff()
    .dropna()
)

print("Consecutive sorted order_id gap summary:")
print(
    order_id_gaps
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .round(2)
)

print()

print(
    "Strictly increasing after global sort:",
    (order_id_gaps > 0).all()
)

ORDER_ID GLOBAL STRUCTURE
Base interactions: 346860
Unique order_id: 346860
Duplicate order_id: 0

order_id linked to multiple students: 0

Minimum order_id: 20224085
Maximum order_id: 38310202

Consecutive sorted order_id gap summary:
count     346859.00
mean          52.14
std         2501.92
min            1.00
25%            4.00
50%           11.00
75%           26.00
90%           58.00
95%          104.00
99%          483.00
max      1445286.00
Name: order_id, dtype: float64

Strictly increasing after global sort: True


### 1.7 Historical Ordering Decision

The historical-ordering investigation produced the following findings:

- The processed dataset does not provide a conventional timestamp that can be used to establish exact interaction time.
- `position` cannot be used as a global student-history ordering field because it resets when assignment and sequence context changes.
- All 17,330 backward `position` transitions observed after sorting student histories by `order_id` occurred when the assignment changed.
- No backward position transition occurred within the same assignment/sequence context.
- Original raw-file row position also cannot be treated as an authoritative chronology. When interactions were examined in raw-file appearance order, `order_id` decreased in 23,107 consecutive student transitions.
- Most of these raw-order disagreements occurred when assignment or sequence context changed, showing that raw-file organization contains structural grouping effects.
- Within student-assignment histories sorted by `order_id`, all 280,621 comparable transitions had the same assignment `position`, with no backward transition.
- The Base Interaction Dataset contains 346,860 globally unique `order_id` values.
- No `order_id` is associated with multiple students.
- Therefore, `order_id` provides a unique and deterministic interaction-level ordering key.

Based on the available evidence, `order_id` will be used as the historical interaction-order proxy for feature engineering.

This decision does **not** interpret `order_id` as a timestamp or claim that its numeric differences represent elapsed time. It is used only to establish a deterministic sequence of interactions when constructing student histories.

To prevent future-information leakage:

1. each student's interactions will be sorted by `order_id`;
2. features for a current interaction will use only interactions with earlier `order_id` values for that student;
3. the current interaction's outcome will not be included in its own historical features;
4. rolling and cumulative features will therefore be shifted so that they represent only previously available evidence.

This is the strongest ordering strategy supported by the available dataset fields and the empirical ordering checks performed in this notebook.

## 2. Student-Level Historical Features

### 2.1 Previous Interaction Count

The first historical feature represents the amount of student evidence available before the current interaction.

For each student, Base Interactions are sorted using the validated `order_id` ordering proxy.

The feature:

`previous_interaction_count`

represents the number of Base Interactions belonging to that student that occurred earlier in the ordered history.

The first interaction for every student therefore has a value of zero.

This feature contains no information from the current interaction outcome and can later be used as an evidence-strength indicator when interpreting learning-state predictions.

In [9]:
student_history_df = (
    base_df
    .sort_values(
        ["user_id", "order_id"]
    )
    .copy()
)

student_history_df[
    "previous_interaction_count"
] = (
    student_history_df
    .groupby("user_id")
    .cumcount()
)


print(
    "Feature rows:",
    len(student_history_df)
)

print()

print(
    "Minimum previous_interaction_count:",
    student_history_df[
        "previous_interaction_count"
    ].min()
)

print(
    "Maximum previous_interaction_count:",
    student_history_df[
        "previous_interaction_count"
    ].max()
)

print()

first_interactions = (
    student_history_df[
        "previous_interaction_count"
    ] == 0
)

print(
    "Interactions with previous_interaction_count = 0:",
    first_interactions.sum()
)

print(
    "Unique students:",
    student_history_df["user_id"].nunique()
)

print(
    "One zero-history interaction per student:",
    first_interactions.sum()
    == student_history_df["user_id"].nunique()
)

print()

print("Previous-interaction-count summary:")

print(
    student_history_df[
        "previous_interaction_count"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .round(2)
)

print()

print("Example student history:")

example_user = (
    student_history_df
    .groupby("user_id")
    .size()
    .idxmax()
)

print(
    student_history_df[
        student_history_df["user_id"]
        == example_user
    ][
        [
            "user_id",
            "order_id",
            "correct",
            "previous_interaction_count"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

Feature rows: 346860

Minimum previous_interaction_count: 0
Maximum previous_interaction_count: 1382

Interactions with previous_interaction_count = 0: 4217
Unique students: 4217
One zero-history interaction per student: True

Previous-interaction-count summary:
count    346860.00
mean        203.44
std         231.33
min           0.00
25%          31.00
50%         108.00
75%         305.00
90%         557.00
95%         693.00
99%         968.00
max        1382.00
Name: previous_interaction_count, dtype: float64

Example student history:
 user_id  order_id  correct  previous_interaction_count
   78978  20857587        1                           0
   78978  20863960        1                           1
   78978  20864017        1                           2
   78978  20960845        1                           3
   78978  20960896        1                           4
   78978  20960936        1                           5
   78978  21039220        1                           6
   78

### 2.2 Previous Correct and Incorrect Counts

The student's previous interaction count indicates the amount of available historical evidence but does not describe the quality of that evidence.

Two additional cumulative historical features are therefore constructed:

- `previous_correct_count`
- `previous_incorrect_count`

For each current interaction, these features count the number of correct and incorrect Base Interactions that occurred earlier in that student's ordered history.

The current interaction's correctness is excluded from both features.

The following identity should therefore hold for every interaction:

`previous_correct_count + previous_incorrect_count = previous_interaction_count`

The first interaction of every student should have zero previous correct and zero previous incorrect interactions.

In [10]:
student_history_df[
    "previous_correct_count"
] = (
    student_history_df
    .groupby("user_id")["correct"]
    .cumsum()
    - student_history_df["correct"]
)

student_history_df[
    "previous_incorrect_count"
] = (
    student_history_df[
        "previous_interaction_count"
    ]
    - student_history_df[
        "previous_correct_count"
    ]
)


print(
    "Minimum previous_correct_count:",
    student_history_df[
        "previous_correct_count"
    ].min()
)

print(
    "Minimum previous_incorrect_count:",
    student_history_df[
        "previous_incorrect_count"
    ].min()
)

print()

count_identity_valid = (
    (
        student_history_df[
            "previous_correct_count"
        ]
        +
        student_history_df[
            "previous_incorrect_count"
        ]
    )
    ==
    student_history_df[
        "previous_interaction_count"
    ]
).all()

print(
    "Correct + incorrect = previous interactions:",
    count_identity_valid
)

print()

first_rows = (
    student_history_df[
        student_history_df[
            "previous_interaction_count"
        ] == 0
    ]
)

print(
    "First interactions with previous_correct_count = 0:",
    (
        first_rows[
            "previous_correct_count"
        ] == 0
    ).all()
)

print(
    "First interactions with previous_incorrect_count = 0:",
    (
        first_rows[
            "previous_incorrect_count"
        ] == 0
    ).all()
)

print()

print("Previous correct-count summary:")
print(
    student_history_df[
        "previous_correct_count"
    ]
    .describe()
    .round(2)
)

print()

print("Previous incorrect-count summary:")
print(
    student_history_df[
        "previous_incorrect_count"
    ]
    .describe()
    .round(2)
)

print()

print("Example student history:")

print(
    student_history_df[
        student_history_df["user_id"]
        == example_user
    ][
        [
            "user_id",
            "order_id",
            "correct",
            "previous_interaction_count",
            "previous_correct_count",
            "previous_incorrect_count"
        ]
    ]
    .head(12)
    .to_string(index=False)
)

Minimum previous_correct_count: 0
Minimum previous_incorrect_count: 0

Correct + incorrect = previous interactions: True

First interactions with previous_correct_count = 0: True
First interactions with previous_incorrect_count = 0: True

Previous correct-count summary:
count    346860.00
mean        137.47
std         158.66
min           0.00
25%          18.00
50%          68.00
75%         211.00
max         888.00
Name: previous_correct_count, dtype: float64

Previous incorrect-count summary:
count    346860.00
mean         65.97
std          83.47
min           0.00
25%          10.00
50%          35.00
75%          91.00
max         686.00
Name: previous_incorrect_count, dtype: float64

Example student history:
 user_id  order_id  correct  previous_interaction_count  previous_correct_count  previous_incorrect_count
   78978  20857587        1                           0                       0                         0
   78978  20863960        1                           1     

### 2.3 Historical Accuracy

Previous correct and incorrect counts can be converted into a normalized measure of the student's historical performance.

The feature:

`historical_accuracy`

is defined as:

`previous_correct_count / previous_interaction_count`

Only interactions that occurred before the current interaction contribute to this value.

For the first interaction of a student, `previous_interaction_count = 0`. Historical accuracy is therefore unavailable rather than being assigned a value of zero.

This distinction is important because:

- `historical_accuracy = 0` means that previous evidence exists and all previous interactions were incorrect;
- missing historical accuracy means that no previous student evidence exists yet.

This feature represents the student's broader accumulated performance before the current interaction.

In [11]:
student_history_df[
    "historical_accuracy"
] = (
    student_history_df[
        "previous_correct_count"
    ]
    /
    student_history_df[
        "previous_interaction_count"
    ]
)

print(
    "Historical accuracy available:",
    student_history_df[
        "historical_accuracy"
    ].notna().sum()
)

print(
    "Historical accuracy unavailable:",
    student_history_df[
        "historical_accuracy"
    ].isna().sum()
)

print()

print(
    "Expected unavailable count (one per student):",
    student_history_df["user_id"].nunique()
)

print(
    "Unavailable count correct:",
    student_history_df[
        "historical_accuracy"
    ].isna().sum()
    == student_history_df["user_id"].nunique()
)

print()

available_accuracy = (
    student_history_df[
        "historical_accuracy"
    ].dropna()
)

print(
    "Minimum historical accuracy:",
    available_accuracy.min()
)

print(
    "Maximum historical accuracy:",
    available_accuracy.max()
)

print(
    "All available accuracy values within [0, 1]:",
    available_accuracy.between(
        0,
        1
    ).all()
)

print()

print("Historical-accuracy summary:")
print(
    available_accuracy
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .round(3)
)

print()

print("Example student history:")

print(
    student_history_df[
        student_history_df["user_id"]
        == example_user
    ][
        [
            "user_id",
            "order_id",
            "correct",
            "previous_interaction_count",
            "previous_correct_count",
            "previous_incorrect_count",
            "historical_accuracy"
        ]
    ]
    .head(15)
    .round(
        {
            "historical_accuracy": 3
        }
    )
    .to_string(index=False)
)

Historical accuracy available: 342643
Historical accuracy unavailable: 4217

Expected unavailable count (one per student): 4217
Unavailable count correct: True

Minimum historical accuracy: 0.0
Maximum historical accuracy: 1.0
All available accuracy values within [0, 1]: True

Historical-accuracy summary:
count    342643.000
mean          0.652
std           0.197
min           0.000
10%           0.406
25%           0.556
50%           0.690
75%           0.780
90%           0.850
95%           0.904
99%           1.000
max           1.000
Name: historical_accuracy, dtype: float64

Example student history:
 user_id  order_id  correct  previous_interaction_count  previous_correct_count  previous_incorrect_count  historical_accuracy
   78978  20857587        1                           0                       0                         0                  NaN
   78978  20863960        1                           1                       1                         0                1.000
   7

## 3. Recent Student-Level Performance

### 3.1 Candidate Recent-History Window Coverage

Lifetime historical accuracy summarizes all previous student interactions, but it may respond slowly to recent changes in learning behaviour.

A recent-history accuracy feature is therefore required to represent the student's more immediate performance.

Before selecting a rolling-history window, the amount of historical evidence available for different candidate window sizes is examined.

Candidate windows of 3, 5, 10, 20, and 30 previous interactions are evaluated.

For each window, two quantities are measured:

- interactions having at least one previous observation available;
- interactions having a complete history equal to the requested window size.

This analysis helps distinguish between small windows that are widely available but potentially unstable and larger windows that provide more evidence but may be unavailable for students with shorter histories.

No final recent-history window is selected at this stage.

In [12]:
candidate_windows = [
    3,
    5,
    10,
    20,
    30
]

window_coverage_rows = []

total_rows = len(student_history_df)

for window in candidate_windows:

    at_least_one = (
        student_history_df[
            "previous_interaction_count"
        ] >= 1
    ).sum()

    full_window = (
        student_history_df[
            "previous_interaction_count"
        ] >= window
    ).sum()

    students_with_full_window = (
        student_history_df.loc[
            student_history_df[
                "previous_interaction_count"
            ] >= window,
            "user_id"
        ]
        .nunique()
    )

    window_coverage_rows.append(
        {
            "window_size": window,

            "interactions_with_any_history":
                at_least_one,

            "any_history_percentage":
                round(
                    at_least_one
                    / total_rows
                    * 100,
                    2
                ),

            "interactions_with_full_window":
                full_window,

            "full_window_percentage":
                round(
                    full_window
                    / total_rows
                    * 100,
                    2
                ),

            "students_reaching_full_window":
                students_with_full_window,

            "student_percentage_reaching_window":
                round(
                    students_with_full_window
                    / student_history_df[
                        "user_id"
                    ].nunique()
                    * 100,
                    2
                )
        }
    )


window_coverage_table = pd.DataFrame(
    window_coverage_rows
)


print("Candidate recent-history window coverage:")
print(
    window_coverage_table.to_string(
        index=False
    )
)

print()

print("Student maximum available history summary:")

student_max_history = (
    student_history_df
    .groupby("user_id")[
        "previous_interaction_count"
    ]
    .max()
)

print(
    student_max_history
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .round(2)
)

Candidate recent-history window coverage:
 window_size  interactions_with_any_history  any_history_percentage  interactions_with_full_window  full_window_percentage  students_reaching_full_window  student_percentage_reaching_window
           3                         342643                   98.78                         334616                   96.47                           3840                               91.06
           5                         342643                   98.78                         327032                   94.28                           3591                               85.16
          10                         342643                   98.78                         310199                   89.43                           3014                               71.47
          20                         342643                   98.78                         283836                   81.83                           2242                               53.17
        

### 3.2 Comparison of Candidate Recent-Accuracy Windows

Coverage alone is insufficient for selecting a recent-history window.

Smaller windows react more quickly to changes in student performance but may be unstable because a single correct or incorrect response has a large effect.

Larger windows provide more stable estimates but may respond too slowly to genuine changes in the student's current learning behaviour.

Recent accuracy is therefore calculated using candidate windows of 3, 5, 10, 20, and 30 previous Base Interactions.

For every current interaction, the current correctness value is excluded before the rolling calculation is performed.

The resulting candidate features are compared in terms of:

- availability,
- variability,
- agreement with broader historical accuracy,
- and disagreement between short and longer recent windows.

No final window is selected until these empirical behaviours are examined.

In [13]:
# Correctness shifted by one interaction so that the
# current answer cannot enter its own historical features.
student_history_df["previous_correct"] = (
    student_history_df
    .groupby("user_id")["correct"]
    .shift(1)
)

candidate_windows = [3, 5, 10, 20, 30]

recent_accuracy_summary = []

for window in candidate_windows:

    feature_name = f"recent_accuracy_{window}"

    student_history_df[feature_name] = (
        student_history_df
        .groupby("user_id")["previous_correct"]
        .transform(
            lambda x: x.rolling(
                window=window,
                min_periods=1
            ).mean()
        )
    )

    values = (
        student_history_df[feature_name]
        .dropna()
    )

    paired = (
        student_history_df[
            [
                feature_name,
                "historical_accuracy"
            ]
        ]
        .dropna()
    )

    correlation = (
        paired[feature_name]
        .corr(
            paired["historical_accuracy"]
        )
    )

    recent_accuracy_summary.append(
        {
            "window": window,
            "available_interactions": len(values),
            "mean": round(values.mean(), 3),
            "std": round(values.std(), 3),
            "correlation_with_historical_accuracy":
                round(correlation, 3)
        }
    )


recent_accuracy_summary = pd.DataFrame(
    recent_accuracy_summary
)

print("Recent-accuracy candidate summary:")
print(
    recent_accuracy_summary.to_string(
        index=False
    )
)

print()


# Compare short and longer windows only where both
# requested windows are completely available.
full_5_and_10 = (
    student_history_df[
        "previous_interaction_count"
    ] >= 10
)

difference_5_10 = (
    student_history_df.loc[
        full_5_and_10,
        "recent_accuracy_5"
    ]
    -
    student_history_df.loc[
        full_5_and_10,
        "recent_accuracy_10"
    ]
).abs()

print("Absolute difference: recent_accuracy_5 vs recent_accuracy_10")
print(
    difference_5_10
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .round(3)
)

print()

print(
    "Difference >= 0.20:",
    (difference_5_10 >= 0.20).sum(),
    "(",
    round(
        (difference_5_10 >= 0.20).mean()
        * 100,
        2
    ),
    "%)"
)

print(
    "Difference >= 0.30:",
    (difference_5_10 >= 0.30).sum(),
    "(",
    round(
        (difference_5_10 >= 0.30).mean()
        * 100,
        2
    ),
    "%)"
)

print()


# Show examples where the 5- and 10-interaction
# views strongly disagree.
comparison_examples = (
    student_history_df.loc[
        full_5_and_10,
        [
            "user_id",
            "order_id",
            "previous_interaction_count",
            "historical_accuracy",
            "recent_accuracy_5",
            "recent_accuracy_10",
            "recent_accuracy_20"
        ]
    ]
    .copy()
)

comparison_examples["difference_5_10"] = (
    comparison_examples["recent_accuracy_5"]
    -
    comparison_examples["recent_accuracy_10"]
).abs()

print("Examples with largest 5-vs-10 differences:")

print(
    comparison_examples
    .sort_values(
        "difference_5_10",
        ascending=False
    )
    .head(15)
    .round(3)
    .to_string(index=False)
)

Recent-accuracy candidate summary:
 window  available_interactions  mean   std  correlation_with_historical_accuracy
      3                  342643 0.644 0.334                                 0.567
      5                  342643 0.643 0.293                                 0.652
     10                  342643 0.644 0.255                                 0.758
     20                  342643 0.644 0.230                                 0.849
     30                  342643 0.645 0.220                                 0.892

Absolute difference: recent_accuracy_5 vs recent_accuracy_10
count    310199.000
mean          0.111
std           0.100
min           0.000
25%           0.000
50%           0.100
75%           0.200
90%           0.200
95%           0.300
99%           0.400
max           0.500
dtype: float64

Difference >= 0.20: 62508 ( 20.15 %)
Difference >= 0.30: 24173 ( 7.79 %)

Examples with largest 5-vs-10 differences:
 user_id  order_id  previous_interaction_count  historical

### 3.3 Short-Term Versus Broader Recent Performance

The comparison between 5-interaction and 10-interaction recent accuracy showed meaningful disagreement for a substantial number of interactions.

Rather than treating this disagreement only as instability, its direction may provide useful information about changes in student performance.

A candidate change signal is therefore defined as:

`recent_accuracy_5 - recent_accuracy_10`

A positive value indicates that performance in the latest five previous interactions is stronger than performance across the latest ten.

A negative value indicates that the latest five interactions are weaker than the broader ten-interaction history.

A value near zero indicates similar short-term and broader recent performance.

This analysis investigates the distribution of this signed difference before deciding whether it should be retained as a learning-state feature.

In [14]:
full_10_history = (
    student_history_df[
        "previous_interaction_count"
    ] >= 10
)

recent_change = (
    student_history_df.loc[
        full_10_history,
        "recent_accuracy_5"
    ]
    -
    student_history_df.loc[
        full_10_history,
        "recent_accuracy_10"
    ]
)

print(
    "Interactions with full 10-interaction history:",
    len(recent_change)
)

print()

print("Signed recent-change summary:")
print(
    recent_change
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .round(3)
)

print()

improving_020 = (
    recent_change >= 0.20
)

declining_020 = (
    recent_change <= -0.20
)

stable_020 = (
    recent_change.abs() < 0.20
)

print("Using ±0.20 only as an analysis threshold:")
print(
    "Improving-like differences:",
    improving_020.sum(),
    "(",
    round(improving_020.mean() * 100, 2),
    "%)"
)

print(
    "Declining-like differences:",
    declining_020.sum(),
    "(",
    round(declining_020.mean() * 100, 2),
    "%)"
)

print(
    "Smaller differences:",
    stable_020.sum(),
    "(",
    round(stable_020.mean() * 100, 2),
    "%)"
)

print()

print(
    "Positive differences:",
    (recent_change > 0).sum(),
    "(",
    round(
        (recent_change > 0).mean() * 100,
        2
    ),
    "%)"
)

print(
    "Negative differences:",
    (recent_change < 0).sum(),
    "(",
    round(
        (recent_change < 0).mean() * 100,
        2
    ),
    "%)"
)

print(
    "Exactly zero:",
    (recent_change == 0).sum(),
    "(",
    round(
        (recent_change == 0).mean() * 100,
        2
    ),
    "%)"
)

print()


student_history_df[
    "recent_accuracy_change_5_vs_10"
] = (
    student_history_df[
        "recent_accuracy_5"
    ]
    -
    student_history_df[
        "recent_accuracy_10"
    ]
)

example_change_rows = (
    student_history_df.loc[
        full_10_history,
        [
            "user_id",
            "order_id",
            "previous_interaction_count",
            "historical_accuracy",
            "recent_accuracy_5",
            "recent_accuracy_10",
            "recent_accuracy_change_5_vs_10"
        ]
    ]
)

print("Largest positive differences:")
print(
    example_change_rows
    .sort_values(
        "recent_accuracy_change_5_vs_10",
        ascending=False
    )
    .head(10)
    .round(3)
    .to_string(index=False)
)

print()

print("Largest negative differences:")
print(
    example_change_rows
    .sort_values(
        "recent_accuracy_change_5_vs_10"
    )
    .head(10)
    .round(3)
    .to_string(index=False)
)

Interactions with full 10-interaction history: 310199

Signed recent-change summary:
count    310199.000
mean         -0.001
std           0.150
min          -0.500
1%           -0.400
5%           -0.200
10%          -0.200
25%          -0.100
50%           0.000
75%           0.100
90%           0.200
95%           0.200
99%           0.300
max           0.500
dtype: float64

Using ±0.20 only as an analysis threshold:
Improving-like differences: 26907 ( 8.67 %)
Declining-like differences: 35601 ( 11.48 %)
Smaller differences: 247691 ( 79.85 %)

Positive differences: 108999 ( 35.14 %)
Negative differences: 108462 ( 34.97 %)
Exactly zero: 92738 ( 29.9 %)

Largest positive differences:
 user_id  order_id  previous_interaction_count  historical_accuracy  recent_accuracy_5  recent_accuracy_10  recent_accuracy_change_5_vs_10
   96260  38144938                          92                0.457                1.0                 0.5                             0.5
   96270  38239478          

### 3.4 Leakage Validation for Recent Accuracy

Recent accuracy was constructed after shifting correctness by one interaction so that the current interaction outcome should not contribute to its own feature values.

A direct validation is performed by independently reconstructing the previous five and previous ten correctness values for selected interactions.

For each checked interaction:

- the current interaction is excluded;
- the previous correctness values are obtained directly from the ordered student history;
- their manually reconstructed means are compared with `recent_accuracy_5` and `recent_accuracy_10`.

This provides explicit evidence that the rolling recent-performance features contain historical information only.

In [15]:
# Select interactions with enough previous history.
validation_rows = (
    student_history_df[
        student_history_df[
            "previous_interaction_count"
        ] >= 10
    ]
    .sample(
        n=10,
        random_state=42
    )
)

validation_results = []

for idx, row in validation_rows.iterrows():

    user_id = row["user_id"]
    order_id = row["order_id"]

    student_history = (
        student_history_df[
            student_history_df["user_id"]
            == user_id
        ]
        .sort_values("order_id")
        .reset_index(drop=True)
    )

    current_position = (
        student_history.index[
            student_history["order_id"]
            == order_id
        ][0]
    )

    previous_rows = (
        student_history.iloc[
            :current_position
        ]
    )

    previous_5 = (
        previous_rows["correct"]
        .tail(5)
        .tolist()
    )

    previous_10 = (
        previous_rows["correct"]
        .tail(10)
        .tolist()
    )

    manual_accuracy_5 = (
        sum(previous_5)
        / len(previous_5)
    )

    manual_accuracy_10 = (
        sum(previous_10)
        / len(previous_10)
    )

    validation_results.append(
        {
            "user_id": user_id,
            "order_id": order_id,
            "current_correct": row["correct"],

            "previous_5":
                "".join(
                    map(str, previous_5)
                ),

            "calculated_accuracy_5":
                row["recent_accuracy_5"],

            "manual_accuracy_5":
                manual_accuracy_5,

            "previous_10":
                "".join(
                    map(str, previous_10)
                ),

            "calculated_accuracy_10":
                row["recent_accuracy_10"],

            "manual_accuracy_10":
                manual_accuracy_10
        }
    )


leakage_validation = pd.DataFrame(
    validation_results
)

leakage_validation[
    "accuracy_5_match"
] = (
    leakage_validation[
        "calculated_accuracy_5"
    ].round(10)
    ==
    leakage_validation[
        "manual_accuracy_5"
    ].round(10)
)

leakage_validation[
    "accuracy_10_match"
] = (
    leakage_validation[
        "calculated_accuracy_10"
    ].round(10)
    ==
    leakage_validation[
        "manual_accuracy_10"
    ].round(10)
)


print("Recent-accuracy leakage validation:")
print(
    leakage_validation
    .to_string(index=False)
)

print()

print(
    "5-interaction calculations matched:",
    leakage_validation[
        "accuracy_5_match"
    ].all()
)

print(
    "10-interaction calculations matched:",
    leakage_validation[
        "accuracy_10_match"
    ].all()
)

print(
    "All leakage-validation checks passed:",
    (
        leakage_validation[
            "accuracy_5_match"
        ].all()
        and
        leakage_validation[
            "accuracy_10_match"
        ].all()
    )
)

Recent-accuracy leakage validation:
 user_id  order_id  current_correct previous_5  calculated_accuracy_5  manual_accuracy_5 previous_10  calculated_accuracy_10  manual_accuracy_10  accuracy_5_match  accuracy_10_match
   79319  30451473                1      01111                    0.8                0.8  0111101111                     0.8                 0.8              True               True
   85949  32039119                1      11100                    0.6                0.6  1110011100                     0.6                 0.6              True               True
   78971  28935301                0      10101                    0.6                0.6  1111010101                     0.7                 0.7              True               True
   96281  38138842                1      00111                    0.6                0.6  1011100111                     0.7                 0.7              True               True
   96263  38216205                1      10011        

## 4. Historical Attempt Behaviour

### 4.1 Attempt Behaviour Within Recent Historical Windows

Correctness describes whether previous interactions were successful, but it does not fully represent the amount of difficulty or effort associated with those interactions.

`attempt_count` provides additional behavioural evidence about repeated attempts.

Phase 3 established that the raw attempt-count distribution is highly right-skewed and contains extreme observations. Therefore, an unbounded arithmetic mean may be disproportionately influenced by a small number of extreme values.

Before constructing the final historical attempt features, attempt behaviour is examined within the same leakage-safe historical framework used for recent accuracy.

For each current interaction, only attempt counts from earlier interactions are considered.

Candidate recent windows of 5 and 10 previous interactions are examined using:

- mean attempt count,
- median attempt count,
- maximum attempt count,
- and the proportion of previous interactions requiring more than one attempt.

No clipping or transformation is applied at this stage.

In [16]:
# Shift attempt_count so that the current interaction cannot
# contribute to its own historical attempt features.
student_history_df["previous_attempt_count"] = (
    student_history_df
    .groupby("user_id")["attempt_count"]
    .shift(1)
)

for window in [5, 10]:

    grouped_previous_attempts = (
        student_history_df
        .groupby("user_id")["previous_attempt_count"]
    )

    student_history_df[
        f"recent_attempt_mean_{window}"
    ] = (
        grouped_previous_attempts
        .transform(
            lambda x: x.rolling(
                window=window,
                min_periods=1
            ).mean()
        )
    )

    student_history_df[
        f"recent_attempt_median_{window}"
    ] = (
        grouped_previous_attempts
        .transform(
            lambda x: x.rolling(
                window=window,
                min_periods=1
            ).median()
        )
    )

    student_history_df[
        f"recent_attempt_max_{window}"
    ] = (
        grouped_previous_attempts
        .transform(
            lambda x: x.rolling(
                window=window,
                min_periods=1
            ).max()
        )
    )

    student_history_df[
        f"recent_multi_attempt_rate_{window}"
    ] = (
        student_history_df[
            "previous_attempt_count"
        ]
        .gt(1)
        .where(
            student_history_df[
                "previous_attempt_count"
            ].notna()
        )
        .groupby(
            student_history_df["user_id"]
        )
        .transform(
            lambda x: x.rolling(
                window=window,
                min_periods=1
            ).mean()
        )
    )


full_10 = (
    student_history_df[
        "previous_interaction_count"
    ] >= 10
)

attempt_features = [
    "recent_attempt_mean_5",
    "recent_attempt_median_5",
    "recent_attempt_max_5",
    "recent_multi_attempt_rate_5",
    "recent_attempt_mean_10",
    "recent_attempt_median_10",
    "recent_attempt_max_10",
    "recent_multi_attempt_rate_10"
]

print(
    "Interactions with full 10-interaction history:",
    full_10.sum()
)

print()

print("Attempt-feature summary for full 10-history interactions:")

print(
    student_history_df.loc[
        full_10,
        attempt_features
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
    .T
    .round(3)
)

print()

print(
    "Maximum recent_attempt_mean_5:",
    student_history_df.loc[
        full_10,
        "recent_attempt_mean_5"
    ].max()
)

print(
    "Maximum recent_attempt_mean_10:",
    student_history_df.loc[
        full_10,
        "recent_attempt_mean_10"
    ].max()
)

print()

print("Rows with largest 10-interaction mean attempt count:")

print(
    student_history_df.loc[
        full_10,
        [
            "user_id",
            "order_id",
            "previous_interaction_count",
            "recent_attempt_mean_10",
            "recent_attempt_median_10",
            "recent_attempt_max_10",
            "recent_multi_attempt_rate_10",
            "recent_accuracy_10"
        ]
    ]
    .sort_values(
        "recent_attempt_mean_10",
        ascending=False
    )
    .head(15)
    .round(3)
    .to_string(index=False)
)

Interactions with full 10-interaction history: 310199

Attempt-feature summary for full 10-history interactions:
                                 count   mean     std  min  50%  75%  90%  \
recent_attempt_mean_5         310199.0  1.601   5.273  0.0  1.2  1.6  2.2   
recent_attempt_median_5       310199.0  1.134   0.522  0.0  1.0  1.0  1.0   
recent_attempt_max_5          310199.0  3.452  26.034  0.0  2.0  3.0  5.0   
recent_multi_attempt_rate_5   310199.0  0.203   0.215  0.0  0.2  0.4  0.4   
recent_attempt_mean_10        310199.0  1.598   3.728  0.0  1.3  1.6  2.2   
recent_attempt_median_10      310199.0  1.078   0.318  0.0  1.0  1.0  1.0   
recent_attempt_max_10         310199.0  5.134  36.361  0.0  3.0  4.0  7.0   
recent_multi_attempt_rate_10  310199.0  0.202   0.171  0.0  0.2  0.3  0.4   

                               95%   99%    99.9%     max  
recent_attempt_mean_5          3.0   6.4   36.081   771.0  
recent_attempt_median_5        2.0   3.0    6.000    45.0  
recent_attemp

### 4.2 Robust Representation of Attempt Count

Recent attempt-count analysis showed that arithmetic means can be strongly distorted by rare extreme observations.

For example, a single interaction with several thousand recorded attempts can dominate the mean of an otherwise ordinary recent-history window.

The raw attempt count should therefore not automatically be used as an unbounded continuous learning-state feature.

Three candidate representations are compared:

1. raw `attempt_count`;
2. attempt count capped at 10;
3. logarithmically transformed attempt count using `log1p`.

The cap of 10 is used only as an analysis candidate based on the Phase 3 distribution, where attempt counts above 10 represented a small minority of interactions. It is not adopted as a final threshold at this stage.

Median recent attempts and multi-attempt rate are also retained as robust candidate representations.

The objective is to preserve meaningful repeated-attempt behaviour while preventing a very small number of extreme values from dominating the student's historical difficulty representation.

In [17]:
import numpy as np

student_history_df["previous_attempt_capped_10"] = student_history_df["previous_attempt_count"].clip(upper=10)
student_history_df["previous_attempt_log1p"] = np.log1p(student_history_df["previous_attempt_count"])

for window in [5, 10]:
    student_history_df[f"recent_attempt_capped_mean_{window}"] = student_history_df.groupby("user_id")["previous_attempt_capped_10"].transform(lambda x: x.rolling(window=window, min_periods=1).mean())
    student_history_df[f"recent_attempt_log_mean_{window}"] = student_history_df.groupby("user_id")["previous_attempt_log1p"].transform(lambda x: x.rolling(window=window, min_periods=1).mean())

full_10 = student_history_df["previous_interaction_count"] >= 10
comparison_features = ["recent_attempt_mean_10", "recent_attempt_capped_mean_10", "recent_attempt_log_mean_10", "recent_attempt_median_10", "recent_multi_attempt_rate_10"]

print("Interactions analysed:", full_10.sum())
print()
print("Robust attempt-feature comparison:")
print(student_history_df.loc[full_10, comparison_features].describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.999]).T.round(3))
print()
print("Correlations between candidate attempt features:")
print(student_history_df.loc[full_10, comparison_features].corr().round(3))
print()
print("Windows containing at least one raw attempt > 10:")
has_extreme_10 = student_history_df["previous_attempt_count"].gt(10).where(student_history_df["previous_attempt_count"].notna()).groupby(student_history_df["user_id"]).transform(lambda x: x.rolling(window=10, min_periods=1).max())
extreme_windows = full_10 & has_extreme_10.eq(1)
print("Interactions:", extreme_windows.sum())
print("Percentage of full 10-history interactions:", round(extreme_windows.sum() / full_10.sum() * 100, 2), "%")
print()
print("Raw vs robust features in extreme windows:")
print(student_history_df.loc[extreme_windows, comparison_features].describe().T.round(3))

Interactions analysed: 310199

Robust attempt-feature comparison:
                                  count   mean    std  min    50%    75%  \
recent_attempt_mean_10         310199.0  1.598  3.728  0.0  1.300  1.600   
recent_attempt_capped_mean_10  310199.0  1.420  0.517  0.0  1.300  1.600   
recent_attempt_log_mean_10     310199.0  0.818  0.154  0.0  0.774  0.884   
recent_attempt_median_10       310199.0  1.078  0.318  0.0  1.000  1.000   
recent_multi_attempt_rate_10   310199.0  0.202  0.171  0.0  0.200  0.300   

                                 90%    95%    99%   99.9%      max  
recent_attempt_mean_10         2.200  2.800  5.900  26.600  387.900  
recent_attempt_capped_mean_10  2.100  2.400  3.300   4.600    7.800  
recent_attempt_log_mean_10     1.009  1.107  1.334   1.686    2.996  
recent_attempt_median_10       1.000  1.500  2.500   4.000   25.500  
recent_multi_attempt_rate_10   0.400  0.500  0.700   0.900    1.000  

Correlations between candidate attempt features:
       

Interactions: 17357
Percentage of full 10-history interactions: 5.6 %

Raw vs robust features in extreme windows:
                                 count   mean     std    min    25%    50%  \
recent_attempt_mean_10         17357.0  5.764  15.070  1.600  2.700  3.400   
recent_attempt_capped_mean_10  17357.0  2.575   0.689  1.500  2.100  2.400   
recent_attempt_log_mean_10     17357.0  1.127   0.207  0.595  0.979  1.088   
recent_attempt_median_10       17357.0  1.306   0.777  0.000  1.000  1.000   
recent_multi_attempt_rate_10   17357.0  0.350   0.180  0.100  0.200  0.300   

                                 75%      max  
recent_attempt_mean_10         4.900  387.900  
recent_attempt_capped_mean_10  2.900    7.800  
recent_attempt_log_mean_10     1.233    2.996  
recent_attempt_median_10       1.500   25.500  
recent_multi_attempt_rate_10   0.500    1.000  


### 4.3 Leakage Validation for Historical Attempt Features

The selected candidate attempt features are based on previous interaction behaviour and must not include the current interaction's `attempt_count`.

A direct validation is therefore performed for sampled interactions with at least ten previous observations.

For each sampled interaction, the previous ten raw attempt counts are independently reconstructed from the ordered student history.

Two historical features are then manually recalculated:

- mean `log1p(attempt_count)` across the previous ten interactions;
- proportion of the previous ten interactions where `attempt_count > 1`.

These manually reconstructed values are compared with:

- `recent_attempt_log_mean_10`;
- `recent_multi_attempt_rate_10`.

The current interaction's attempt count is excluded from all calculations.

In [18]:
attempt_validation_rows = student_history_df[student_history_df["previous_interaction_count"] >= 10].sample(n=10, random_state=43)
attempt_validation_results = []

for idx, row in attempt_validation_rows.iterrows():
    user_id = row["user_id"]
    order_id = row["order_id"]
    student_history = student_history_df[student_history_df["user_id"] == user_id].sort_values("order_id").reset_index(drop=True)
    current_position = student_history.index[student_history["order_id"] == order_id][0]
    previous_10_attempts = student_history.iloc[:current_position]["attempt_count"].tail(10).tolist()
    manual_log_mean = np.log1p(previous_10_attempts).mean()
    manual_multi_attempt_rate = np.mean(np.array(previous_10_attempts) > 1)
    attempt_validation_results.append({
        "user_id": user_id,
        "order_id": order_id,
        "current_attempt_count": row["attempt_count"],
        "previous_10_attempts": ",".join(map(str, previous_10_attempts)),
        "calculated_log_mean": row["recent_attempt_log_mean_10"],
        "manual_log_mean": manual_log_mean,
        "calculated_multi_attempt_rate": row["recent_multi_attempt_rate_10"],
        "manual_multi_attempt_rate": manual_multi_attempt_rate
    })

attempt_leakage_validation = pd.DataFrame(attempt_validation_results)
attempt_leakage_validation["log_mean_match"] = attempt_leakage_validation["calculated_log_mean"].round(10) == attempt_leakage_validation["manual_log_mean"].round(10)
attempt_leakage_validation["multi_attempt_rate_match"] = attempt_leakage_validation["calculated_multi_attempt_rate"].round(10) == attempt_leakage_validation["manual_multi_attempt_rate"].round(10)

print("Attempt-feature leakage validation:")
print(attempt_leakage_validation.to_string(index=False))
print()
print("Log-mean calculations matched:", attempt_leakage_validation["log_mean_match"].all())
print("Multi-attempt-rate calculations matched:", attempt_leakage_validation["multi_attempt_rate_match"].all())
print("All attempt leakage-validation checks passed:", attempt_leakage_validation["log_mean_match"].all() and attempt_leakage_validation["multi_attempt_rate_match"].all())

Attempt-feature leakage validation:
 user_id  order_id  current_attempt_count previous_10_attempts  calculated_log_mean  manual_log_mean  calculated_multi_attempt_rate  manual_multi_attempt_rate  log_mean_match  multi_attempt_rate_match
   78926  23342235                      1  1,1,1,1,4,1,1,1,1,1             0.784776         0.784776                            0.1                        0.1            True                      True
   83249  28152002                      1  1,1,3,1,1,1,1,1,9,5             1.033267         1.033267                            0.3                        0.3            True                      True
   79483  22422247                      1  1,1,1,7,1,3,1,2,1,1             0.941638         0.941638                            0.3                        0.3            True                      True
   78283  27573971                      1  1,1,1,1,1,1,1,1,1,1             0.693147         0.693147                            0.0                        0.0  

## 5. Historical Hint Behaviour

### 5.1 Recent Hint Availability and Usage

Hint behaviour provides additional evidence about whether a student required instructional support during previous interactions.

Phase 3 established that `hint_count` and `hint_total` have different meanings:

- `hint_count` represents the number of hints used;
- `hint_total` represents the number of hints available;
- `hint_total = 0` means normalized hint usage cannot be measured for that interaction.

Therefore, interactions with no available hints must not automatically be interpreted as zero hint dependency.

Historical hint behaviour is first examined using leakage-safe previous-interaction information.

For the previous ten interactions, the analysis measures:

- proportion of interactions where at least one hint was used;
- proportion of interactions where hints were available;
- number of interactions where normalized hint usage was measurable;
- average normalized hint-usage rate only across interactions where `hint_total > 0`.

The current interaction's hint information is excluded.

In [19]:
student_history_df["previous_hint_count"] = student_history_df.groupby("user_id")["hint_count"].shift(1)
student_history_df["previous_hint_total"] = student_history_df.groupby("user_id")["hint_total"].shift(1)
student_history_df["previous_used_hint"] = student_history_df["previous_hint_count"].gt(0).where(student_history_df["previous_hint_count"].notna()).astype(float)
student_history_df["previous_hints_available"] = student_history_df["previous_hint_total"].gt(0).where(student_history_df["previous_hint_total"].notna()).astype(float)
student_history_df["previous_hint_usage_rate"] = np.where(student_history_df["previous_hint_total"] > 0, student_history_df["previous_hint_count"] / student_history_df["previous_hint_total"], np.nan)

window = 10
student_history_df["recent_hint_use_rate_10"] = student_history_df.groupby("user_id")["previous_used_hint"].transform(lambda x: x.rolling(window=window, min_periods=1).mean())
student_history_df["recent_hint_available_rate_10"] = student_history_df.groupby("user_id")["previous_hints_available"].transform(lambda x: x.rolling(window=window, min_periods=1).mean())
student_history_df["recent_hint_measurable_count_10"] = student_history_df["previous_hint_usage_rate"].notna().astype(float).where(student_history_df["previous_hint_count"].notna()).groupby(student_history_df["user_id"]).transform(lambda x: x.rolling(window=window, min_periods=1).sum())
student_history_df["recent_hint_usage_rate_10"] = student_history_df.groupby("user_id")["previous_hint_usage_rate"].transform(lambda x: x.rolling(window=window, min_periods=1).mean())

full_10 = student_history_df["previous_interaction_count"] >= 10
hint_features = ["recent_hint_use_rate_10", "recent_hint_available_rate_10", "recent_hint_measurable_count_10", "recent_hint_usage_rate_10"]

print("Interactions with full 10-interaction history:", full_10.sum())
print()
print("Recent hint-feature summary:")
print(student_history_df.loc[full_10, hint_features].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T.round(3))
print()
print("Full-history interactions with NO measurable hint-usage observation in previous 10:", (student_history_df.loc[full_10, "recent_hint_measurable_count_10"] == 0).sum())
print("Percentage:", round((student_history_df.loc[full_10, "recent_hint_measurable_count_10"] == 0).mean() * 100, 2), "%")
print()
print("Full-history interactions with all 10 previous interactions having measurable hint usage:", (student_history_df.loc[full_10, "recent_hint_measurable_count_10"] == 10).sum())
print("Percentage:", round((student_history_df.loc[full_10, "recent_hint_measurable_count_10"] == 10).mean() * 100, 2), "%")
print()
print("Correlations between recent hint features:")
print(student_history_df.loc[full_10, hint_features].corr().round(3))

Interactions with full 10-interaction history: 310199

Recent hint-feature summary:
                                    count   mean    std  min  25%    50%  \
recent_hint_use_rate_10          310199.0  0.169  0.215  0.0  0.0  0.100   
recent_hint_available_rate_10    310199.0  0.699  0.271  0.0  0.5  0.700   
recent_hint_measurable_count_10  310199.0  6.986  2.708  0.0  5.0  7.000   
recent_hint_usage_rate_10        300866.0  0.205  0.257  0.0  0.0  0.111   

                                   75%   90%   95%   99%   max  
recent_hint_use_rate_10          0.300   0.5   0.6   0.9   1.0  
recent_hint_available_rate_10    0.900   1.0   1.0   1.0   1.0  
recent_hint_measurable_count_10  9.000  10.0  10.0  10.0  10.0  
recent_hint_usage_rate_10        0.325   0.6   0.8   1.0   1.0  

Full-history interactions with NO measurable hint-usage observation in previous 10: 9333
Percentage: 3.01 %

Full-history interactions with all 10 previous interactions having measurable hint usage: 76825
Perc

### 5.2 Reliability of Normalized Historical Hint Usage

Normalized hint usage provides a more meaningful measure of hint dependency than simply counting hint use because it considers how many hints were actually available.

However, the reliability of this feature depends on the number of previous interactions in which hints were available.

For example, a normalized historical hint-usage rate based on only one eligible interaction may be less reliable than the same rate calculated from ten eligible interactions.

The previous ten-interaction windows are therefore grouped according to the number of interactions where normalized hint usage was measurable.

This analysis evaluates whether a minimum evidence requirement or an accompanying availability feature is needed before normalized hint usage is used as a learning-state signal.

In [20]:
full_hint_analysis = student_history_df.loc[full_10, ["user_id", "order_id", "recent_hint_use_rate_10", "recent_hint_available_rate_10", "recent_hint_measurable_count_10", "recent_hint_usage_rate_10", "recent_accuracy_10"]].copy()
full_hint_analysis["hint_evidence_group"] = pd.cut(full_hint_analysis["recent_hint_measurable_count_10"], bins=[-0.1, 0, 2, 5, 9, 10], labels=["0 measurable", "1-2 measurable", "3-5 measurable", "6-9 measurable", "10 measurable"])

print("Hint evidence-group distribution:")
evidence_distribution = full_hint_analysis["hint_evidence_group"].value_counts(sort=False).to_frame("interactions")
evidence_distribution["percentage"] = (evidence_distribution["interactions"] / len(full_hint_analysis) * 100).round(2)
print(evidence_distribution)
print()
print("Behaviour by amount of measurable hint evidence:")
evidence_behaviour = full_hint_analysis.groupby("hint_evidence_group", observed=False).agg(interactions=("order_id", "size"), avg_measurable_count=("recent_hint_measurable_count_10", "mean"), avg_hint_use_rate=("recent_hint_use_rate_10", "mean"), avg_normalized_hint_usage=("recent_hint_usage_rate_10", "mean"), median_normalized_hint_usage=("recent_hint_usage_rate_10", "median"), avg_recent_accuracy=("recent_accuracy_10", "mean")).round(3)
print(evidence_behaviour)
print()
for threshold in [1, 2, 3, 5]:
    low_evidence = (full_hint_analysis["recent_hint_measurable_count_10"] > 0) & (full_hint_analysis["recent_hint_measurable_count_10"] <= threshold)
    print(f"Normalized hint usage based on <= {threshold} measurable interactions:", low_evidence.sum(), "(", round(low_evidence.mean() * 100, 2), "%)")

print()
print("Normalized hint-usage missing:", full_hint_analysis["recent_hint_usage_rate_10"].isna().sum())
print("Missing exactly when measurable count = 0:", (full_hint_analysis["recent_hint_usage_rate_10"].isna() == full_hint_analysis["recent_hint_measurable_count_10"].eq(0)).all())

Hint evidence-group distribution:
                     interactions  percentage
hint_evidence_group                          
0 measurable                 9333        3.01
1-2 measurable              14287        4.61
3-5 measurable              58964       19.01
6-9 measurable             150790       48.61
10 measurable               76825       24.77

Behaviour by amount of measurable hint evidence:
                     interactions  avg_measurable_count  avg_hint_use_rate  \
hint_evidence_group                                                          
0 measurable                 9333                 0.000              0.000   
1-2 measurable              14287                 1.581              0.026   
3-5 measurable              58964                 4.220              0.078   
6-9 measurable             150790                 7.476              0.188   
10 measurable               76825                10.000              0.249   

                     avg_normalized_hint_usage 

### 5.3 Leakage Validation for Historical Hint Features

Historical hint features must represent only behaviour that occurred before the current interaction.

A direct leakage validation is therefore performed on sampled interactions with at least ten previous interactions.

For each sampled interaction, the previous ten `hint_count` and `hint_total` values are independently reconstructed.

The following features are manually recalculated:

- proportion of previous interactions where hints were available;
- average normalized hint usage across only those previous interactions where `hint_total > 0`.

These manually reconstructed values are compared with:

- `recent_hint_available_rate_10`;
- `recent_hint_usage_rate_10`.

The current interaction's hint information is excluded from all calculations.

In [21]:
hint_validation_rows = student_history_df[student_history_df["previous_interaction_count"] >= 10].sample(n=10, random_state=44)
hint_validation_results = []

for idx, row in hint_validation_rows.iterrows():
    user_id = row["user_id"]
    order_id = row["order_id"]
    student_history = student_history_df[student_history_df["user_id"] == user_id].sort_values("order_id").reset_index(drop=True)
    current_position = student_history.index[student_history["order_id"] == order_id][0]
    previous_10 = student_history.iloc[:current_position].tail(10)
    previous_hint_counts = previous_10["hint_count"].tolist()
    previous_hint_totals = previous_10["hint_total"].tolist()
    manual_available_rate = np.mean(np.array(previous_hint_totals) > 0)
    measurable_rates = [hint_count / hint_total for hint_count, hint_total in zip(previous_hint_counts, previous_hint_totals) if hint_total > 0]
    manual_usage_rate = np.mean(measurable_rates) if len(measurable_rates) > 0 else np.nan
    hint_validation_results.append({
        "user_id": user_id,
        "order_id": order_id,
        "current_hint_count": row["hint_count"],
        "current_hint_total": row["hint_total"],
        "previous_10_hint_counts": ",".join(map(str, previous_hint_counts)),
        "previous_10_hint_totals": ",".join(map(str, previous_hint_totals)),
        "calculated_available_rate": row["recent_hint_available_rate_10"],
        "manual_available_rate": manual_available_rate,
        "calculated_usage_rate": row["recent_hint_usage_rate_10"],
        "manual_usage_rate": manual_usage_rate
    })

hint_leakage_validation = pd.DataFrame(hint_validation_results)
hint_leakage_validation["available_rate_match"] = hint_leakage_validation["calculated_available_rate"].round(10) == hint_leakage_validation["manual_available_rate"].round(10)
hint_leakage_validation["usage_rate_match"] = ((hint_leakage_validation["calculated_usage_rate"].round(10) == hint_leakage_validation["manual_usage_rate"].round(10)) | (hint_leakage_validation["calculated_usage_rate"].isna() & hint_leakage_validation["manual_usage_rate"].isna()))

print("Hint-feature leakage validation:")
print(hint_leakage_validation.to_string(index=False))
print()
print("Hint-availability calculations matched:", hint_leakage_validation["available_rate_match"].all())
print("Normalized hint-usage calculations matched:", hint_leakage_validation["usage_rate_match"].all())
print("All hint leakage-validation checks passed:", hint_leakage_validation["available_rate_match"].all() and hint_leakage_validation["usage_rate_match"].all())

Hint-feature leakage validation:
 user_id  order_id  current_hint_count  current_hint_total previous_10_hint_counts previous_10_hint_totals  calculated_available_rate  manual_available_rate  calculated_usage_rate  manual_usage_rate  available_rate_match  usage_rate_match
   86352  29643684                   0                   0     0,0,0,0,0,0,0,0,0,0     0,0,0,0,0,0,0,0,0,0                        0.0                    0.0                    NaN                NaN                  True              True
   79069  31562573                   0                   4     0,0,0,0,0,0,0,0,0,0     2,2,2,2,0,2,2,0,2,4                        0.8                    0.8               0.000000           0.000000                  True              True
   78461  28217685                   0                   4     0,3,0,0,0,0,0,0,0,0     4,4,4,3,4,4,4,4,4,4                        1.0                    1.0               0.075000           0.075000                  True              True
   79744  3

## 6. Historical Response-Time Behaviour

### 6.1 Robust Representation of Recent Response Time

Response time may provide useful information about the amount of time a student required to respond during previous interactions.

However, Phase 3 identified several data-quality characteristics in `ms_first_response`:

- a very small number of negative values;
- zero response times;
- a strongly right-skewed positive distribution;
- and rare extremely long response times.

Therefore, raw response time should not automatically be averaged without considering these characteristics.

For historical feature engineering:

- negative response times are treated as invalid for response-time measurement;
- zero values are retained separately at this stage rather than silently removed;
- positive response times are examined in raw and logarithmically transformed form;
- recent median response time is also evaluated as a robust representation.

All calculations use only interactions occurring before the current interaction.

In [22]:
student_history_df["previous_response_ms"] = student_history_df.groupby("user_id")["ms_first_response"].shift(1)
student_history_df["previous_positive_response_ms"] = student_history_df["previous_response_ms"].where(student_history_df["previous_response_ms"] > 0)
student_history_df["previous_response_log1p"] = np.log1p(student_history_df["previous_positive_response_ms"])

window = 10
student_history_df["recent_response_mean_ms_10"] = student_history_df.groupby("user_id")["previous_positive_response_ms"].transform(lambda x: x.rolling(window=window, min_periods=1).mean())
student_history_df["recent_response_median_ms_10"] = student_history_df.groupby("user_id")["previous_positive_response_ms"].transform(lambda x: x.rolling(window=window, min_periods=1).median())
student_history_df["recent_response_log_mean_10"] = student_history_df.groupby("user_id")["previous_response_log1p"].transform(lambda x: x.rolling(window=window, min_periods=1).mean())
student_history_df["recent_response_measurable_count_10"] = student_history_df["previous_positive_response_ms"].notna().astype(float).where(student_history_df["previous_response_ms"].notna()).groupby(student_history_df["user_id"]).transform(lambda x: x.rolling(window=window, min_periods=1).sum())

full_10 = student_history_df["previous_interaction_count"] >= 10
response_features = ["recent_response_mean_ms_10", "recent_response_median_ms_10", "recent_response_log_mean_10", "recent_response_measurable_count_10"]

print("Interactions with full 10-interaction history:", full_10.sum())
print()
print("Recent response-time feature summary:")
print(student_history_df.loc[full_10, response_features].describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.999]).T.round(3))
print()
print("Full 10-history interactions with all 10 positive response times:", (student_history_df.loc[full_10, "recent_response_measurable_count_10"] == 10).sum())
print("Percentage:", round((student_history_df.loc[full_10, "recent_response_measurable_count_10"] == 10).mean() * 100, 2), "%")
print()
print("Full 10-history interactions with fewer than 10 positive response times:", (student_history_df.loc[full_10, "recent_response_measurable_count_10"] < 10).sum())
print()
print("Correlations between response-time representations:")
print(student_history_df.loc[full_10, ["recent_response_mean_ms_10", "recent_response_median_ms_10", "recent_response_log_mean_10"]].corr().round(3))
print()
print("Rows with largest raw recent response-time means:")
print(student_history_df.loc[full_10, ["user_id", "order_id", "previous_interaction_count", "recent_response_mean_ms_10", "recent_response_median_ms_10", "recent_response_log_mean_10", "recent_response_measurable_count_10", "recent_accuracy_10"]].sort_values("recent_response_mean_ms_10", ascending=False).head(15).round(3).to_string(index=False))

Interactions with full 10-interaction history: 310199

Recent response-time feature summary:
                                        count       mean         std    min  \
recent_response_mean_ms_10           310199.0  44164.974  114362.886  1.571   
recent_response_median_ms_10         310199.0  24542.040   22380.060  1.000   
recent_response_log_mean_10          310199.0      9.810       0.781  0.925   
recent_response_measurable_count_10  310199.0      9.991       0.153  3.000   

                                           50%        75%        90%  \
recent_response_mean_ms_10           29104.000  48548.250  77880.600   
recent_response_median_ms_10         18526.000  31179.500  49320.500   
recent_response_log_mean_10              9.851     10.317     10.741   
recent_response_measurable_count_10     10.000     10.000     10.000   

                                            95%         99%        99.9%  \
recent_response_mean_ms_10           107067.200  252400.572  1177042.872  

### 6.2 Relationship Between Recent Response Time and Learning Behaviour

The response-time distribution alone does not establish that longer response times represent learning difficulty.

A long response may reflect cognitive difficulty, but it may also reflect interruption, inactivity, or other behaviour unrelated to learning.

Therefore, recent response-time features are compared with other historical learning signals rather than being interpreted independently.

The analysis examines relationships between recent response time and:

- recent accuracy;
- recent repeated-attempt behaviour;
- recent normalized hint usage.

Response-time observations are also divided into empirical groups using the distribution of the leakage-safe 10-interaction log-transformed response feature.

These groups are used only for exploratory analysis and are not treated as learning-state thresholds.

In [23]:
response_analysis = student_history_df.loc[full_10, ["user_id", "order_id", "recent_response_log_mean_10", "recent_response_median_ms_10", "recent_accuracy_10", "recent_attempt_log_mean_10", "recent_multi_attempt_rate_10", "recent_hint_usage_rate_10", "recent_hint_available_rate_10"]].copy()

print("Correlations with recent response-time features:")
correlation_columns = ["recent_response_log_mean_10", "recent_response_median_ms_10", "recent_accuracy_10", "recent_attempt_log_mean_10", "recent_multi_attempt_rate_10", "recent_hint_usage_rate_10", "recent_hint_available_rate_10"]
print(response_analysis[correlation_columns].corr().loc[["recent_response_log_mean_10", "recent_response_median_ms_10"], ["recent_accuracy_10", "recent_attempt_log_mean_10", "recent_multi_attempt_rate_10", "recent_hint_usage_rate_10", "recent_hint_available_rate_10"]].round(3))
print()
response_analysis["response_log_quartile"] = pd.qcut(response_analysis["recent_response_log_mean_10"], q=4, labels=["Q1 fastest", "Q2", "Q3", "Q4 slowest"], duplicates="drop")
print("Behaviour by recent response-time quartile:")
response_quartile_summary = response_analysis.groupby("response_log_quartile", observed=False).agg(interactions=("order_id", "size"), avg_log_response=("recent_response_log_mean_10", "mean"), median_response_ms=("recent_response_median_ms_10", "median"), avg_recent_accuracy=("recent_accuracy_10", "mean"), avg_attempt_log=("recent_attempt_log_mean_10", "mean"), avg_multi_attempt_rate=("recent_multi_attempt_rate_10", "mean"), avg_hint_usage_rate=("recent_hint_usage_rate_10", "mean"), avg_hint_available_rate=("recent_hint_available_rate_10", "mean")).round(3)
print(response_quartile_summary)
print()
print("Recent response-time log feature quantile boundaries:")
print(response_analysis["recent_response_log_mean_10"].quantile([0.00, 0.25, 0.50, 0.75, 1.00]).round(3))
print()
response_analysis["recent_response_median_seconds_10"] = response_analysis["recent_response_median_ms_10"] / 1000
print("Median-response seconds by log-response quartile:")
print(response_analysis.groupby("response_log_quartile", observed=False)["recent_response_median_seconds_10"].describe()[["count", "mean", "50%", "75%", "max"]].round(2))

Correlations with recent response-time features:
                              recent_accuracy_10  recent_attempt_log_mean_10  \
recent_response_log_mean_10                0.279                       0.161   
recent_response_median_ms_10               0.135                       0.066   

                              recent_multi_attempt_rate_10  \
recent_response_log_mean_10                          0.061   
recent_response_median_ms_10                         0.031   

                              recent_hint_usage_rate_10  \
recent_response_log_mean_10                      -0.257   
recent_response_median_ms_10                     -0.100   

                              recent_hint_available_rate_10  
recent_response_log_mean_10                          -0.026  
recent_response_median_ms_10                         -0.023  

Behaviour by recent response-time quartile:
                       interactions  avg_log_response  median_response_ms  \
response_log_quartile                

                         count   mean    50%    75%     max
response_log_quartile                                      
Q1 fastest             77550.0   7.37   6.98   9.17  117.83
Q2                     77550.0  15.00  14.42  17.28  123.60
Q3                     77549.0  24.48  23.48  28.07  128.27
Q4 slowest             77550.0  51.31  43.78  59.10  589.71


### 6.3 Leakage Validation for Historical Response-Time Features

Recent response-time analysis showed that response speed should be treated as contextual behavioural evidence rather than a direct indicator of learning difficulty.

Before retaining the robust response-time representations, their historical construction is validated for information leakage.

For sampled interactions with at least ten previous observations, the previous ten response times are independently reconstructed.

Only strictly positive response times are treated as measurable response-time observations, consistent with the feature-engineering procedure.

The following values are manually recalculated:

- median positive response time across the previous ten interactions;
- mean `log1p` positive response time across the previous ten interactions.

These manually reconstructed values are compared with:

- `recent_response_median_ms_10`;
- `recent_response_log_mean_10`.

The current interaction's response time is excluded from all calculations.

In [24]:
response_validation_rows = student_history_df[student_history_df["previous_interaction_count"] >= 10].sample(n=10, random_state=45)
response_validation_results = []

for idx, row in response_validation_rows.iterrows():
    user_id = row["user_id"]
    order_id = row["order_id"]
    student_history = student_history_df[student_history_df["user_id"] == user_id].sort_values("order_id").reset_index(drop=True)
    current_position = student_history.index[student_history["order_id"] == order_id][0]
    previous_10_response = student_history.iloc[:current_position]["ms_first_response"].tail(10).tolist()
    positive_previous_response = [value for value in previous_10_response if value > 0]
    if len(positive_previous_response) > 0:
        manual_median = np.median(positive_previous_response)
        manual_log_mean = np.log1p(positive_previous_response).mean()
    else:
        manual_median = np.nan
        manual_log_mean = np.nan
    response_validation_results.append({
        "user_id": user_id,
        "order_id": order_id,
        "current_response_ms": row["ms_first_response"],
        "previous_10_response_ms": ",".join(map(str, previous_10_response)),
        "positive_measurements": len(positive_previous_response),
        "calculated_median_ms": row["recent_response_median_ms_10"],
        "manual_median_ms": manual_median,
        "calculated_log_mean": row["recent_response_log_mean_10"],
        "manual_log_mean": manual_log_mean
    })

response_leakage_validation = pd.DataFrame(response_validation_results)
response_leakage_validation["median_match"] = ((response_leakage_validation["calculated_median_ms"].round(10) == response_leakage_validation["manual_median_ms"].round(10)) | (response_leakage_validation["calculated_median_ms"].isna() & response_leakage_validation["manual_median_ms"].isna()))
response_leakage_validation["log_mean_match"] = ((response_leakage_validation["calculated_log_mean"].round(10) == response_leakage_validation["manual_log_mean"].round(10)) | (response_leakage_validation["calculated_log_mean"].isna() & response_leakage_validation["manual_log_mean"].isna()))

print("Response-time leakage validation:")
print(response_leakage_validation.to_string(index=False))
print()
print("Median-response calculations matched:", response_leakage_validation["median_match"].all())
print("Log-response calculations matched:", response_leakage_validation["log_mean_match"].all())
print("All response-time leakage-validation checks passed:", response_leakage_validation["median_match"].all() and response_leakage_validation["log_mean_match"].all())

Response-time leakage validation:
 user_id  order_id  current_response_ms                                           previous_10_response_ms  positive_measurements  calculated_median_ms  manual_median_ms  calculated_log_mean  manual_log_mean  median_match  log_mean_match
   96221  38142171                 7532        8145,11864,21862,37132,151540,92444,2995,12355,44850,67347                     10               29497.0           29497.0            10.152008        10.152008          True            True
   84143  26648520                36964       43470,38979,13426,11843,7110,49993,26450,18234,12721,125577                     10               22342.0           22342.0            10.101023        10.101023          True            True
   70684  29217927                52727          22383,38784,23102,19952,19071,26830,4094,8006,6005,12303                     10               19511.5           19511.5             9.600782         9.600782          True            True
   78977  30437030

## 7. Concept-Specific Historical Performance

### 7.1 Student-Skill History Coverage

General student history describes overall learning behaviour, but learning state may differ substantially between concepts.

A student who performs strongly overall may still struggle with a particular skill. Therefore, concept-specific historical performance is investigated using the Concept Interaction Dataset.

Each Concept Interaction represents one unique `order_id + skill_id` pair.

Before constructing concept-specific rolling features, the amount of historical evidence available for each student-skill pair must be examined.

For each Concept Interaction, the number of earlier interactions by the same student on the same `skill_id` is calculated.

The current interaction is not included in this historical count.

This analysis is used to determine whether concept-specific rolling windows are sufficiently supported by the dataset and to identify an appropriate minimum-history strategy.

In [25]:
concept_history_df = concept_df.copy()
concept_history_df = concept_history_df.sort_values(["user_id", "order_id", "skill_id"]).reset_index(drop=True)
concept_history_df["previous_skill_interaction_count"] = concept_history_df.groupby(["user_id", "skill_id"]).cumcount()

print("Concept interactions:", len(concept_history_df))
print("Unique students:", concept_history_df["user_id"].nunique())
print("Unique skills:", concept_history_df["skill_id"].nunique())
print("Unique student-skill pairs:", concept_history_df[["user_id", "skill_id"]].drop_duplicates().shape[0])
print()
print("Minimum previous skill interactions:", concept_history_df["previous_skill_interaction_count"].min())
print("Maximum previous skill interactions:", concept_history_df["previous_skill_interaction_count"].max())
print()
first_skill_interactions = concept_history_df["previous_skill_interaction_count"] == 0
print("Interactions with no previous history for that skill:", first_skill_interactions.sum())
print("Percentage:", round(first_skill_interactions.mean() * 100, 2), "%")
print()
print("Previous skill-history count summary:")
print(concept_history_df["previous_skill_interaction_count"].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))
print()
candidate_skill_windows = [1, 2, 3, 5, 10, 20]
skill_window_coverage = []
total_student_skill_pairs = concept_history_df[["user_id", "skill_id"]].drop_duplicates().shape[0]
for window in candidate_skill_windows:
    interactions_with_full_window = (concept_history_df["previous_skill_interaction_count"] >= window).sum()
    student_skill_pairs_reaching_window = concept_history_df.loc[concept_history_df["previous_skill_interaction_count"] >= window, ["user_id", "skill_id"]].drop_duplicates().shape[0]
    skill_window_coverage.append({"window_size": window, "interactions_with_full_window": interactions_with_full_window, "full_window_percentage": round(interactions_with_full_window / len(concept_history_df) * 100, 2), "student_skill_pairs_reaching_window": student_skill_pairs_reaching_window, "pair_percentage_reaching_window": round(student_skill_pairs_reaching_window / total_student_skill_pairs * 100, 2)})
skill_window_coverage = pd.DataFrame(skill_window_coverage)
print("Candidate concept-history window coverage:")
print(skill_window_coverage.to_string(index=False))
print()
student_skill_max_history = concept_history_df.groupby(["user_id", "skill_id"])["previous_skill_interaction_count"].max()
print("Maximum available history per student-skill pair:")
print(student_skill_max_history.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

Concept interactions: 338001
Unique students: 4163
Unique skills: 123
Unique student-skill pairs: 41982

Minimum previous skill interactions: 0
Maximum previous skill interactions: 289

Interactions with no previous history for that skill: 41982
Percentage: 12.42 %

Previous skill-history count summary:


count    338001.00
mean         10.61
std          16.06
min           0.00
25%           2.00
50%           5.00
75%          13.00
90%          26.00
95%          39.00
99%          79.00
max         289.00
Name: previous_skill_interaction_count, dtype: float64

Candidate concept-history window coverage:
 window_size  interactions_with_full_window  full_window_percentage  student_skill_pairs_reaching_window  pair_percentage_reaching_window
           1                         296019                   87.58                                33381                            79.51
           2                         262638                   77.70                                28215                            67.21
           3                         234423                   69.36                                25374                            60.44
           5                         186916                   55.30                                18928                            45.09
  

### 7.2 Historical and Recent Skill Accuracy

Student-skill history coverage showed that concept-specific histories are substantially shorter than overall student histories.

A fixed 10-interaction concept window would therefore have limited full-window coverage.

To investigate an appropriate representation, leakage-safe concept-specific accuracy features are constructed.

For every Concept Interaction:

- `previous_skill_correct_count` counts earlier correct interactions for the same student and skill;
- `previous_skill_incorrect_count` counts earlier incorrect interactions;
- `historical_skill_accuracy` represents accuracy across all earlier interactions for that skill;
- recent skill accuracy is evaluated using candidate windows of 3, 5, and 10 previous interactions.

The current interaction is excluded from every historical feature.

Smaller windows are included because concept-specific histories are considerably shorter than overall student histories.

In [26]:
concept_history_df["previous_skill_correct"] = concept_history_df.groupby(["user_id", "skill_id"])["correct"].shift(1)
concept_history_df["previous_skill_correct_count"] = concept_history_df["previous_skill_correct"].fillna(0).groupby([concept_history_df["user_id"], concept_history_df["skill_id"]]).cumsum()
concept_history_df["previous_skill_incorrect_count"] = concept_history_df["previous_skill_interaction_count"] - concept_history_df["previous_skill_correct_count"]
concept_history_df["historical_skill_accuracy"] = concept_history_df["previous_skill_correct_count"] / concept_history_df["previous_skill_interaction_count"]
concept_history_df.loc[concept_history_df["previous_skill_interaction_count"] == 0, "historical_skill_accuracy"] = np.nan
for window in [3, 5, 10]:
    concept_history_df[f"recent_skill_accuracy_{window}"] = concept_history_df.groupby(["user_id", "skill_id"])["previous_skill_correct"].transform(lambda x: x.rolling(window=window, min_periods=1).mean())

print("Concept interactions:", len(concept_history_df))
print()
print("Correct + incorrect = previous skill interactions:", ((concept_history_df["previous_skill_correct_count"] + concept_history_df["previous_skill_incorrect_count"]) == concept_history_df["previous_skill_interaction_count"]).all())
print()
print("Historical skill accuracy available:", concept_history_df["historical_skill_accuracy"].notna().sum())
print("Historical skill accuracy unavailable:", concept_history_df["historical_skill_accuracy"].isna().sum())
print("Unavailable exactly on first student-skill interaction:", (concept_history_df["historical_skill_accuracy"].isna() == (concept_history_df["previous_skill_interaction_count"] == 0)).all())
print()
print("Historical skill-accuracy summary:")
print(concept_history_df["historical_skill_accuracy"].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(3))
print()
candidate_skill_accuracy_summary = []
for window in [3, 5, 10]:
    column = f"recent_skill_accuracy_{window}"
    full_window = concept_history_df["previous_skill_interaction_count"] >= window
    candidate_skill_accuracy_summary.append({"window": window, "available_interactions": concept_history_df[column].notna().sum(), "full_window_interactions": full_window.sum(), "full_window_percentage": round(full_window.mean() * 100, 2), "mean": round(concept_history_df[column].mean(), 3), "std": round(concept_history_df[column].std(), 3), "correlation_with_historical_skill_accuracy": round(concept_history_df[[column, "historical_skill_accuracy"]].corr().iloc[0, 1], 3)})
candidate_skill_accuracy_summary = pd.DataFrame(candidate_skill_accuracy_summary)
print("Candidate recent skill-accuracy summary:")
print(candidate_skill_accuracy_summary.to_string(index=False))
print()
full_5_skill_history = concept_history_df["previous_skill_interaction_count"] >= 5
skill_difference_3_5 = (concept_history_df.loc[full_5_skill_history, "recent_skill_accuracy_3"] - concept_history_df.loc[full_5_skill_history, "recent_skill_accuracy_5"]).abs()
print("Interactions with full 5-skill history:", full_5_skill_history.sum())
print()
print("Absolute difference: recent_skill_accuracy_3 vs recent_skill_accuracy_5")
print(skill_difference_3_5.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(3))
print()
print("Difference >= 0.20:", (skill_difference_3_5 >= 0.20).sum(), "(", round((skill_difference_3_5 >= 0.20).mean() * 100, 2), "%)")
print("Difference >= 0.30:", (skill_difference_3_5 >= 0.30).sum(), "(", round((skill_difference_3_5 >= 0.30).mean() * 100, 2), "%)")
print()
print("Examples with largest 3-vs-5 skill differences:")
example_columns = ["user_id", "order_id", "skill_id", "skill_name", "previous_skill_interaction_count", "historical_skill_accuracy", "recent_skill_accuracy_3", "recent_skill_accuracy_5", "recent_skill_accuracy_10"]
examples = concept_history_df.loc[full_5_skill_history, example_columns].assign(difference_3_5=skill_difference_3_5).sort_values("difference_3_5", ascending=False).head(15)
print(examples.round(3).to_string(index=False))

Concept interactions: 338001

Correct + incorrect = previous skill interactions: True

Historical skill accuracy available: 296019
Historical skill accuracy unavailable: 41982
Unavailable exactly on first student-skill interaction: True

Historical skill-accuracy summary:
count    296019.000
mean          0.581
std           0.318
min           0.000
10%           0.000
25%           0.375
50%           0.636
75%           0.826
90%           1.000
95%           1.000
99%           1.000
max           1.000
Name: historical_skill_accuracy, dtype: float64

Candidate recent skill-accuracy summary:
 window  available_interactions  full_window_interactions  full_window_percentage  mean   std  correlation_with_historical_skill_accuracy
      3                  296019                    234423                   69.36 0.605 0.366                                       0.866
      5                  296019                    186916                   55.30 0.595 0.340                            

### 7.3 Recent Concept-Specific Performance Change

Recent skill accuracy describes current concept-specific performance, but learning-state identification also requires evidence about direction of change.

For student-skill pairs with at least five previous interactions, the five most recent historical skill interactions are divided into two non-overlapping periods:

- the older two interactions;
- the most recent three interactions.

The mean correctness of the recent three interactions is compared with the mean correctness of the preceding two interactions.

The resulting feature is:

`recent_skill_accuracy_change_3_vs_previous_2`

A positive value indicates that recent skill performance is better than the immediately preceding historical period, while a negative value indicates deterioration.

This feature is treated as evidence of short-term direction rather than a learning-state label.

The current Concept Interaction is excluded from both periods.

In [27]:
def calculate_skill_change(group):
    previous_correct = group["previous_skill_correct"].reset_index(drop=True)
    recent_3 = previous_correct.rolling(window=3, min_periods=3).mean()
    previous_2 = previous_correct.shift(3).rolling(window=2, min_periods=2).mean()
    return pd.DataFrame({"recent_skill_accuracy_last_3": recent_3.values, "previous_skill_accuracy_prior_2": previous_2.values, "recent_skill_accuracy_change_3_vs_previous_2": (recent_3 - previous_2).values}, index=group.index)

skill_change_features = concept_history_df.groupby(["user_id", "skill_id"], group_keys=False).apply(calculate_skill_change, include_groups=False)
for column in skill_change_features.columns:
    concept_history_df[column] = skill_change_features[column]
skill_change_column = "recent_skill_accuracy_change_3_vs_previous_2"

print("Interactions with concept-change feature available:", concept_history_df[skill_change_column].notna().sum())
print("Percentage of Concept Interactions:", round(concept_history_df[skill_change_column].notna().mean() * 100, 2), "%")
print()
print("Concept-specific change summary:")
print(concept_history_df[skill_change_column].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(3))
print()
available_change = concept_history_df[skill_change_column].notna()
print("Positive concept changes:", (concept_history_df.loc[available_change, skill_change_column] > 0).sum())
print("Negative concept changes:", (concept_history_df.loc[available_change, skill_change_column] < 0).sum())
print("Exactly zero:", (concept_history_df.loc[available_change, skill_change_column] == 0).sum())
print()
for threshold in [0.20, 0.30, 0.50]:
    improving = (concept_history_df.loc[available_change, skill_change_column] >= threshold).sum()
    declining = (concept_history_df.loc[available_change, skill_change_column] <= -threshold).sum()
    total_available = available_change.sum()
    print(f"Using ±{threshold:.2f} as analysis threshold:")
    print("  Improving-like:", improving, "(", round(improving / total_available * 100, 2), "%)")
    print("  Declining-like:", declining, "(", round(declining / total_available * 100, 2), "%)")
    print()
print("Relationship with long-term skill accuracy:")
print(concept_history_df[[skill_change_column, "historical_skill_accuracy", "recent_skill_accuracy_3", "recent_skill_accuracy_5"]].corr().round(3))
print()
print("Largest positive concept changes:")
columns_to_show = ["user_id", "order_id", "skill_id", "skill_name", "previous_skill_interaction_count", "historical_skill_accuracy", "previous_skill_accuracy_prior_2", "recent_skill_accuracy_last_3", skill_change_column]
print(concept_history_df.loc[available_change, columns_to_show].sort_values(skill_change_column, ascending=False).head(10).round(3).to_string(index=False))
print()
print("Largest negative concept changes:")
print(concept_history_df.loc[available_change, columns_to_show].sort_values(skill_change_column, ascending=True).head(10).round(3).to_string(index=False))

Interactions with concept-change feature available: 186916
Percentage of Concept Interactions: 55.3 %

Concept-specific change summary:
count    186916.000
mean          0.030
std           0.406
min          -1.000
1%           -1.000
5%           -0.667
10%          -0.500
25%          -0.333
50%           0.000
75%           0.333
90%           0.500
95%           0.667
99%           1.000
max           1.000
Name: recent_skill_accuracy_change_3_vs_previous_2, dtype: float64

Positive concept changes: 75204
Negative concept changes: 64575
Exactly zero: 47137

Using ±0.20 as analysis threshold:
  Improving-like: 49211 ( 26.33 %)
  Declining-like: 50316 ( 26.92 %)

Using ±0.30 as analysis threshold:
  Improving-like: 49211 ( 26.33 %)
  Declining-like: 50316 ( 26.92 %)

Using ±0.50 as analysis threshold:
  Improving-like: 39187 ( 20.97 %)
  Declining-like: 20697 ( 11.07 %)

Relationship with long-term skill accuracy:
                                              recent_skill_accuracy_c

### 7.4 Leakage Validation for Concept-Specific Performance Change

The concept-specific change feature is intended to describe movement in a student's performance on a particular skill using only historical interactions.

To validate its construction, sampled Concept Interactions with at least five previous interactions on the same skill are independently reconstructed.

For each sampled interaction:

1. the five immediately preceding interactions for the same student and skill are retrieved;
2. the first two of those five form the older period;
3. the final three form the recent period;
4. their mean correctness values are independently calculated;
5. the difference between the recent-three accuracy and older-two accuracy is compared with the engineered feature.

The current Concept Interaction is excluded from all calculations.

This validation checks both the numerical calculation and the absence of current-interaction leakage.

In [28]:
concept_change_validation_rows = concept_history_df[concept_history_df["previous_skill_interaction_count"] >= 5].sample(n=10, random_state=46)
concept_change_validation_results = []

for _, row in concept_change_validation_rows.iterrows():
    user_id = row["user_id"]
    skill_id = row["skill_id"]
    order_id = row["order_id"]
    student_skill_history = concept_history_df[(concept_history_df["user_id"] == user_id) & (concept_history_df["skill_id"] == skill_id)].sort_values("order_id").reset_index(drop=True)
    current_position = student_skill_history.index[student_skill_history["order_id"] == order_id][0]
    previous_5 = student_skill_history.iloc[:current_position]["correct"].tail(5).tolist()
    older_2 = previous_5[:2]
    recent_3 = previous_5[2:]
    manual_prior_2_accuracy = np.mean(older_2)
    manual_recent_3_accuracy = np.mean(recent_3)
    manual_change = manual_recent_3_accuracy - manual_prior_2_accuracy
    concept_change_validation_results.append({
        "user_id": user_id,
        "order_id": order_id,
        "skill_id": skill_id,
        "current_correct": row["correct"],
        "previous_5_correct": "".join(map(str, previous_5)),
        "older_2": "".join(map(str, older_2)),
        "recent_3": "".join(map(str, recent_3)),
        "calculated_prior_2_accuracy": row["previous_skill_accuracy_prior_2"],
        "manual_prior_2_accuracy": manual_prior_2_accuracy,
        "calculated_recent_3_accuracy": row["recent_skill_accuracy_last_3"],
        "manual_recent_3_accuracy": manual_recent_3_accuracy,
        "calculated_change": row["recent_skill_accuracy_change_3_vs_previous_2"],
        "manual_change": manual_change
    })

concept_change_leakage_validation = pd.DataFrame(concept_change_validation_results)
concept_change_leakage_validation["prior_2_match"] = np.isclose(concept_change_leakage_validation["calculated_prior_2_accuracy"], concept_change_leakage_validation["manual_prior_2_accuracy"])
concept_change_leakage_validation["recent_3_match"] = np.isclose(concept_change_leakage_validation["calculated_recent_3_accuracy"], concept_change_leakage_validation["manual_recent_3_accuracy"])
concept_change_leakage_validation["change_match"] = np.isclose(concept_change_leakage_validation["calculated_change"], concept_change_leakage_validation["manual_change"])

print("Concept-change leakage validation:")
print(concept_change_leakage_validation.round(6).to_string(index=False))
print()
print("Prior-2 calculations matched:", concept_change_leakage_validation["prior_2_match"].all())
print("Recent-3 calculations matched:", concept_change_leakage_validation["recent_3_match"].all())
print("Change calculations matched:", concept_change_leakage_validation["change_match"].all())
print("All concept-change leakage-validation checks passed:", concept_change_leakage_validation["prior_2_match"].all() and concept_change_leakage_validation["recent_3_match"].all() and concept_change_leakage_validation["change_match"].all())

Concept-change leakage validation:
 user_id  order_id  skill_id  current_correct previous_5_correct older_2 recent_3  calculated_prior_2_accuracy  manual_prior_2_accuracy  calculated_recent_3_accuracy  manual_recent_3_accuracy  calculated_change  manual_change  prior_2_match  recent_3_match  change_match
   71881  35455830        48                1              01011      01      011                          0.5                      0.5                      0.666667                  0.666667           0.166667       0.166667           True            True          True
   75361  29399414        47                1              01110      01      110                          0.5                      0.5                      0.666667                  0.666667           0.166667       0.166667           True            True          True
   84145  30276521       311                0              00000      00      000                          0.0                      0.0                 

### 7.5 Overall Versus Concept-Specific Historical Performance

Overall student performance and concept-specific performance represent different levels of learning evidence.

A student may perform strongly across interactions in general while showing weak performance on a particular skill. Conversely, a generally weaker student may demonstrate strong performance on a specific concept.

To determine whether concept-specific history contributes information beyond general student history, leakage-safe overall historical features are joined to Concept Interactions using `order_id`.

The analysis compares:

- overall historical accuracy;
- overall recent 10-interaction accuracy;
- historical skill accuracy;
- recent skill accuracy;
- the difference between recent skill performance and recent overall performance.

The comparison is exploratory and is not used to define learning-state labels at this stage.

In [29]:
base_history_for_concept = student_history_df[["order_id", "historical_accuracy", "recent_accuracy_10"]].copy()
concept_comparison_df = concept_history_df.merge(base_history_for_concept, on="order_id", how="left", validate="many_to_one")

print("Concept interactions:", len(concept_comparison_df))
print("Missing joined historical_accuracy:", concept_comparison_df["historical_accuracy"].isna().sum())
print("Missing joined recent_accuracy_10:", concept_comparison_df["recent_accuracy_10"].isna().sum())
print()
concept_comparison_df["skill_vs_overall_recent_accuracy_difference"] = concept_comparison_df["recent_skill_accuracy_5"] - concept_comparison_df["recent_accuracy_10"]
comparison_available = concept_comparison_df["recent_skill_accuracy_5"].notna() & concept_comparison_df["recent_accuracy_10"].notna()
print("Interactions available for skill-vs-overall comparison:", comparison_available.sum())
print("Percentage of Concept Interactions:", round(comparison_available.mean() * 100, 2), "%")
print()
print("Correlations between overall and skill-specific accuracy:")
print(concept_comparison_df.loc[comparison_available, ["historical_accuracy", "recent_accuracy_10", "historical_skill_accuracy", "recent_skill_accuracy_3", "recent_skill_accuracy_5"]].corr().round(3))
print()
difference = concept_comparison_df.loc[comparison_available, "skill_vs_overall_recent_accuracy_difference"]
print("Skill minus overall recent-accuracy difference summary:")
print(difference.describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(3))
print()
for threshold in [0.20, 0.30, 0.50]:
    skill_higher = (difference >= threshold).sum()
    skill_lower = (difference <= -threshold).sum()
    print(f"Absolute comparison threshold ±{threshold:.2f}:")
    print("  Skill performance higher than overall:", skill_higher, "(", round(skill_higher / len(difference) * 100, 2), "%)")
    print("  Skill performance lower than overall:", skill_lower, "(", round(skill_lower / len(difference) * 100, 2), "%)")
    print()
print("Examples where skill performance is much LOWER than overall recent performance:")
example_columns = ["user_id", "order_id", "skill_id", "skill_name", "previous_skill_interaction_count", "historical_accuracy", "recent_accuracy_10", "historical_skill_accuracy", "recent_skill_accuracy_5", "skill_vs_overall_recent_accuracy_difference"]
print(concept_comparison_df.loc[comparison_available, example_columns].sort_values("skill_vs_overall_recent_accuracy_difference", ascending=True).head(15).round(3).to_string(index=False))
print()
print("Examples where skill performance is much HIGHER than overall recent performance:")
print(concept_comparison_df.loc[comparison_available, example_columns].sort_values("skill_vs_overall_recent_accuracy_difference", ascending=False).head(15).round(3).to_string(index=False))

Concept interactions: 338001
Missing joined historical_accuracy: 4940
Missing joined recent_accuracy_10: 4940

Interactions available for skill-vs-overall comparison: 296019
Percentage of Concept Interactions: 87.58 %

Correlations between overall and skill-specific accuracy:
                           historical_accuracy  recent_accuracy_10  \
historical_accuracy                      1.000               0.755   
recent_accuracy_10                       0.755               1.000   
historical_skill_accuracy                0.597               0.650   
recent_skill_accuracy_3                  0.514               0.634   
recent_skill_accuracy_5                  0.553               0.666   

                           historical_skill_accuracy  recent_skill_accuracy_3  \
historical_accuracy                            0.597                    0.514   
recent_accuracy_10                             0.650                    0.634   
historical_skill_accuracy                      1.000       

 user_id  order_id  skill_id                             skill_name  previous_skill_interaction_count  historical_accuracy  recent_accuracy_10  historical_skill_accuracy  recent_skill_accuracy_5  skill_vs_overall_recent_accuracy_difference
   79264  35163532        22               Interior Angles Triangle                                 1                0.690                 0.0                      1.000                      1.0                                          1.0
   90547  35619562       277      Addition and Subtraction Integers                                 1                0.261                 0.0                      1.000                      1.0                                          1.0
   96245  38229052        18          Probability of a Single Event                                 5                0.555                 0.0                      1.000                      1.0                                          1.0
   78919  35229921        13            

### 7.6 Validation of Overall-to-Concept Historical Feature Join

Leakage-safe overall student-history features were joined to Concept Interactions using `order_id`.

Because a single Base Interaction may contain multiple skills, one Base Interaction can correspond to multiple Concept Interactions. Therefore, missing historical values after the join may occur multiple times when the student's first Base Interaction contains multiple skills.

This validation distinguishes expected historical unavailability from merge failure.

For every Concept Interaction, the joined Base Interaction is checked for existence, and missing historical features are compared with the Base Interaction's `previous_interaction_count`.

The expected condition is:

- every Concept Interaction matches a Base Interaction;
- `historical_accuracy` and `recent_accuracy_10` are unavailable exactly when the corresponding Base Interaction has zero previous student interactions.

In [30]:
base_join_validation = student_history_df[["order_id", "previous_interaction_count"]].copy()
concept_join_validation = concept_comparison_df.merge(base_join_validation, on="order_id", how="left", validate="many_to_one")

print("Concept interactions checked:", len(concept_join_validation))
print()
print("Concept interactions with no matching Base order_id:", concept_join_validation["previous_interaction_count"].isna().sum())
print()
zero_overall_history = concept_join_validation["previous_interaction_count"] == 0
missing_historical_accuracy = concept_join_validation["historical_accuracy"].isna()
missing_recent_accuracy = concept_join_validation["recent_accuracy_10"].isna()
print("Concept interactions attached to first Base interaction:", zero_overall_history.sum())
print("Missing historical_accuracy:", missing_historical_accuracy.sum())
print("Missing recent_accuracy_10:", missing_recent_accuracy.sum())
print()
print("historical_accuracy missing exactly when previous_interaction_count = 0:", (missing_historical_accuracy == zero_overall_history).all())
print("recent_accuracy_10 missing exactly when previous_interaction_count = 0:", (missing_recent_accuracy == zero_overall_history).all())
print()
print("Unique students represented among zero-overall-history Concept Interactions:", concept_join_validation.loc[zero_overall_history, "user_id"].nunique())
print()
print("Number of Concept rows generated by each student's first Base Interaction:")
first_interaction_concept_counts = concept_join_validation.loc[zero_overall_history].groupby("user_id").size()
print(first_interaction_concept_counts.value_counts().sort_index())
print()
print("Maximum Concept rows from one student's first Base Interaction:", first_interaction_concept_counts.max())

Concept interactions checked: 338001

Concept interactions with no matching Base order_id: 0

Concept interactions attached to first Base interaction: 4940
Missing historical_accuracy: 4940
Missing recent_accuracy_10: 4940

historical_accuracy missing exactly when previous_interaction_count = 0: True
recent_accuracy_10 missing exactly when previous_interaction_count = 0: True

Unique students represented among zero-overall-history Concept Interactions: 4125

Number of Concept rows generated by each student's first Base Interaction:
1    3369
2     713
3      27
4      16
Name: count, dtype: int64

Maximum Concept rows from one student's first Base Interaction: 4


### 7.7 Concept-History Evidence Strength

Concept-specific learning signals have substantially less history than overall student-level signals.

A five-interaction skill window is a candidate compromise between short-term sensitivity and historical coverage. However, its reliability depends on how much prior evidence exists for the student-skill pair.

This analysis examines concept-history evidence strength and determines how often recent skill features are based on partial versus complete five-interaction histories.

The result will be used to decide how concept-specific features should be represented in the final learning-state feature set.

In [31]:
concept_evidence = concept_history_df.copy()

def skill_evidence_group(n):
    if n == 0:
        return "0 previous"
    elif n == 1:
        return "1 previous"
    elif n == 2:
        return "2 previous"
    elif n < 5:
        return "3-4 previous"
    elif n < 10:
        return "5-9 previous"
    else:
        return "10+ previous"

concept_evidence["skill_evidence_group"] = concept_evidence["previous_skill_interaction_count"].apply(skill_evidence_group)
evidence_summary = concept_evidence.groupby("skill_evidence_group", observed=True).agg(interactions=("order_id", "size"), students=("user_id", "nunique"), student_skill_pairs=("skill_id", "count"), avg_previous_skill_interactions=("previous_skill_interaction_count", "mean"), avg_historical_skill_accuracy=("historical_skill_accuracy", "mean"), avg_recent_skill_accuracy_3=("recent_skill_accuracy_3", "mean"), avg_recent_skill_accuracy_5=("recent_skill_accuracy_5", "mean"))
evidence_summary["percentage"] = evidence_summary["interactions"] / len(concept_evidence) * 100
group_order = ["0 previous", "1 previous", "2 previous", "3-4 previous", "5-9 previous", "10+ previous"]
evidence_summary = evidence_summary.reindex(group_order)
print("Concept-history evidence strength:")
print(evidence_summary.round(3))
print()
full_five = concept_evidence["previous_skill_interaction_count"] >= 5
print("Interactions with full 5-interaction skill history:", full_five.sum())
print("Percentage:", round(full_five.mean() * 100, 2), "%")
print()
partial_five = (concept_evidence["previous_skill_interaction_count"] > 0) & (concept_evidence["previous_skill_interaction_count"] < 5)
print("Interactions with partial 1-4 skill history:", partial_five.sum())
print("Percentage:", round(partial_five.mean() * 100, 2), "%")
print()
no_skill_history = concept_evidence["previous_skill_interaction_count"] == 0
print("Interactions with no previous skill history:", no_skill_history.sum())
print("Percentage:", round(no_skill_history.mean() * 100, 2), "%")
print()
print("Full + partial + no history covers all interactions:", (full_five.sum() + partial_five.sum() + no_skill_history.sum()) == len(concept_evidence))

Concept-history evidence strength:
                      interactions  students  student_skill_pairs  \
skill_evidence_group                                                
0 previous                   41982      4163                41982   
1 previous                   33381      3951                33381   
2 previous                   28215      3754                28215   
3-4 previous                 47507      3625                47507   
5-9 previous                 73920      3104                73920   
10+ previous                112996      2160               112996   

                      avg_previous_skill_interactions  \
skill_evidence_group                                    
0 previous                                      0.000   
1 previous                                      1.000   
2 previous                                      2.000   
3-4 previous                                    3.466   
5-9 previous                                    6.745   
10+ previous 

### 7.8 Final Concept-History Feature Selection

The previous analyses established that concept-specific history provides useful information that is different from overall student performance.

However, concept-history depth varies substantially across interactions. Therefore, the final representation should preserve both the student's concept-specific performance and the amount of evidence supporting that performance.

This step reviews the candidate concept-history features together before selecting the final concept-level signals for the learning-state feature set.

In [32]:
concept_candidates = concept_history_df.copy()
candidate_columns = ["previous_skill_interaction_count", "historical_skill_accuracy", "recent_skill_accuracy_3", "recent_skill_accuracy_5", "recent_skill_accuracy_change_3_vs_previous_2"]
print("CONCEPT FEATURE AVAILABILITY")
print("=" * 60)
availability_rows = []
for col in candidate_columns:
    available = concept_candidates[col].notna().sum()
    availability_rows.append({"feature": col, "available": available, "missing": len(concept_candidates) - available, "availability_percentage": available / len(concept_candidates) * 100})
availability_df = pd.DataFrame(availability_rows)
print(availability_df.round(2).to_string(index=False))
print()
accuracy_features = ["historical_skill_accuracy", "recent_skill_accuracy_3", "recent_skill_accuracy_5"]
print("CORRELATIONS AMONG CONCEPT ACCURACY FEATURES")
print("=" * 60)
print(concept_candidates[accuracy_features].corr().round(3))
print()
print("FEATURE AVAILABILITY BY SKILL-HISTORY DEPTH")
print("=" * 60)
history_groups = pd.cut(concept_candidates["previous_skill_interaction_count"], bins=[-1, 0, 1, 2, 4, 9, float("inf")], labels=["0", "1", "2", "3-4", "5-9", "10+"])
for col in candidate_columns[1:]:
    temp = concept_candidates[col].notna().groupby(history_groups, observed=True).agg(["sum", "count"])
    temp["availability_percentage"] = temp["sum"] / temp["count"] * 100
    print()
    print(col)
    print(temp.round(2))
print()
print("BASIC RANGE VALIDATION")
print("=" * 60)
for col in ["historical_skill_accuracy", "recent_skill_accuracy_3", "recent_skill_accuracy_5"]:
    valid = concept_candidates[col].dropna()
    print(col, "within [0,1]:", valid.between(0, 1).all())
change_valid = concept_candidates["recent_skill_accuracy_change_3_vs_previous_2"].dropna()
print("Concept-change feature within [-1,1]:", change_valid.between(-1, 1).all())

CONCEPT FEATURE AVAILABILITY
                                     feature  available  missing  availability_percentage
            previous_skill_interaction_count     338001        0                   100.00
                   historical_skill_accuracy     296019    41982                    87.58
                     recent_skill_accuracy_3     296019    41982                    87.58
                     recent_skill_accuracy_5     296019    41982                    87.58
recent_skill_accuracy_change_3_vs_previous_2     186916   151085                    55.30

CORRELATIONS AMONG CONCEPT ACCURACY FEATURES
                           historical_skill_accuracy  recent_skill_accuracy_3  \
historical_skill_accuracy                      1.000                    0.866   
recent_skill_accuracy_3                        0.866                    1.000   
recent_skill_accuracy_5                        0.932                    0.931   

                           recent_skill_accuracy_5  
histori

### 7.9 Unified Overall and Concept-Level Feature Table

The learning-state component requires both overall student behaviour and concept-specific learning evidence.

The validated overall historical features are therefore joined to the Concept Interaction Dataset using `order_id`.

Each resulting row represents a student-concept interaction and contains only historical information available before the current interaction.

The unified table preserves:

- overall student-history evidence;
- recent overall accuracy;
- recent accuracy change;
- robust attempt behaviour;
- recent hint behaviour;
- recent response-time behaviour;
- concept-history depth;
- recent concept accuracy;
- recent concept-specific change.

The current interaction outcome is retained only for later validation and target-development purposes and is not used as a historical predictor.

In [33]:
overall_feature_columns = ["order_id", "previous_interaction_count", "historical_accuracy", "recent_accuracy_10", "recent_accuracy_change_5_vs_10", "recent_attempt_log_mean_10", "recent_multi_attempt_rate_10", "recent_hint_usage_rate_10", "recent_hint_available_rate_10", "recent_response_log_mean_10", "recent_response_median_ms_10"]
overall_features = student_history_df[overall_feature_columns].copy()
concept_feature_columns = ["order_id", "user_id", "assignment_id", "problem_id", "skill_id", "skill_name", "correct", "previous_skill_interaction_count", "recent_skill_accuracy_5", "recent_skill_accuracy_change_3_vs_previous_2"]
unified_feature_df = concept_history_df[concept_feature_columns].merge(overall_features, on="order_id", how="left", validate="many_to_one")

print("UNIFIED FEATURE TABLE")
print("=" * 60)
print("Rows:", len(unified_feature_df))
print("Columns:", unified_feature_df.shape[1])
print("Unique students:", unified_feature_df["user_id"].nunique())
print("Unique skills:", unified_feature_df["skill_id"].nunique())
print("Unique order_id:", unified_feature_df["order_id"].nunique())
print()
print("Missing-value summary:")
print(unified_feature_df.isna().sum().sort_values(ascending=False))
print()
print("Rows with no matching overall feature record:", unified_feature_df["previous_interaction_count"].isna().sum())
print()
print("Current correct retained:", "correct" in unified_feature_df.columns)
print("Current correct included in predictor list:", "correct" in [col for col in unified_feature_df.columns if col not in ["order_id", "user_id", "assignment_id", "problem_id", "skill_id", "skill_name"] and col != "correct"])
print()
print("First 5 rows:")
print(unified_feature_df.head().to_string(index=False))

UNIFIED FEATURE TABLE
Rows: 338001
Columns: 20
Unique students: 4163
Unique skills: 123
Unique order_id: 283105

Missing-value summary:
recent_skill_accuracy_change_3_vs_previous_2    151085
recent_skill_accuracy_5                          41982
recent_hint_usage_rate_10                        16020
skill_name                                       12364
recent_multi_attempt_rate_10                      4940
recent_hint_available_rate_10                     4940
recent_response_log_mean_10                       4940
recent_response_median_ms_10                      4940
recent_accuracy_change_5_vs_10                    4940
recent_attempt_log_mean_10                        4940
recent_accuracy_10                                4940
historical_accuracy                               4940
problem_id                                           0
assignment_id                                        0
user_id                                              0
order_id                               

### 7.10 Cold-Start and Historical Evidence Indicators

Historical learning features are naturally unavailable when insufficient prior evidence exists.

These missing values are meaningful and must not automatically be interpreted as poor performance.

For example:

- a missing recent skill accuracy may mean that the student has never previously encountered the skill;
- a missing concept-change value means that fewer than five previous interactions exist for that student-skill pair;
- missing overall historical features occur on the student's first Base Interaction;
- normalized hint usage may be unavailable when none of the recent interactions provided measurable hint opportunities.

Explicit evidence indicators are therefore created so that later models can distinguish true low performance from unavailable historical evidence.

In [34]:
feature_ready_df = unified_feature_df.copy()
feature_ready_df["has_overall_history"] = (feature_ready_df["previous_interaction_count"] > 0).astype(int)
feature_ready_df["has_skill_history"] = (feature_ready_df["previous_skill_interaction_count"] > 0).astype(int)
feature_ready_df["has_full_skill_window_5"] = (feature_ready_df["previous_skill_interaction_count"] >= 5).astype(int)
feature_ready_df["has_skill_change_evidence"] = feature_ready_df["recent_skill_accuracy_change_3_vs_previous_2"].notna().astype(int)
feature_ready_df["has_recent_hint_usage_evidence"] = feature_ready_df["recent_hint_usage_rate_10"].notna().astype(int)

print("COLD-START / EVIDENCE INDICATORS")
print("=" * 60)
indicator_columns = ["has_overall_history", "has_skill_history", "has_full_skill_window_5", "has_skill_change_evidence", "has_recent_hint_usage_evidence"]
for col in indicator_columns:
    counts = feature_ready_df[col].value_counts().sort_index()
    print()
    print(col)
    print(counts)
    print("Percentage with evidence:", round(feature_ready_df[col].mean() * 100, 2), "%")
print()
print("CONSISTENCY CHECKS")
print("=" * 60)
check_1 = (feature_ready_df["has_overall_history"] == feature_ready_df["historical_accuracy"].notna().astype(int)).all()
print("Overall-history indicator matches historical_accuracy availability:", check_1)
check_2 = (feature_ready_df["has_skill_history"] == feature_ready_df["recent_skill_accuracy_5"].notna().astype(int)).all()
print("Skill-history indicator matches recent_skill_accuracy_5 availability:", check_2)
check_3 = (feature_ready_df["has_full_skill_window_5"] == feature_ready_df["has_skill_change_evidence"]).all()
print("Full 5-skill-history indicator matches change-feature availability:", check_3)
check_4 = (feature_ready_df["has_recent_hint_usage_evidence"] == feature_ready_df["recent_hint_usage_rate_10"].notna().astype(int)).all()
print("Hint-evidence indicator matches hint-usage availability:", check_4)
print()
print("Joint overall / skill cold-start situations:")
print(pd.crosstab(feature_ready_df["has_overall_history"], feature_ready_df["has_skill_history"], rownames=["Overall history"], colnames=["Skill history"], margins=True))
print()
print("Rows with skill history but NO overall history:", ((feature_ready_df["has_skill_history"] == 1) & (feature_ready_df["has_overall_history"] == 0)).sum())
print("Rows with overall history but NO skill history:", ((feature_ready_df["has_overall_history"] == 1) & (feature_ready_df["has_skill_history"] == 0)).sum())

COLD-START / EVIDENCE INDICATORS

has_overall_history
has_overall_history
0      4940
1    333061
Name: count, dtype: int64
Percentage with evidence: 98.54 %

has_skill_history
has_skill_history
0     41982
1    296019
Name: count, dtype: int64
Percentage with evidence: 87.58 %

has_full_skill_window_5
has_full_skill_window_5
0    151085
1    186916
Name: count, dtype: int64
Percentage with evidence: 55.3 %

has_skill_change_evidence
has_skill_change_evidence
0    151085
1    186916
Name: count, dtype: int64
Percentage with evidence: 55.3 %

has_recent_hint_usage_evidence
has_recent_hint_usage_evidence
0     16020
1    321981
Name: count, dtype: int64
Percentage with evidence: 95.26 %

CONSISTENCY CHECKS
Overall-history indicator matches historical_accuracy availability: True
Skill-history indicator matches recent_skill_accuracy_5 availability: True
Full 5-skill-history indicator matches change-feature availability: True
Hint-evidence indicator matches hint-usage availability: True

Jo

### 7.11 Final Predictor Redundancy Review

The validated historical features are now assembled into a proposed final predictor set.

Before the feature set is locked, correlations are examined to identify strongly redundant numerical signals.

The goal is not to remove every correlated feature automatically. Two features may be correlated while still representing different learning behaviours. Instead, this review identifies potentially redundant predictors so that the final selection can be justified explicitly.

Identifier fields and the current interaction outcome are excluded from this predictor review.

In [35]:
proposed_predictors = ["previous_interaction_count", "previous_skill_interaction_count", "recent_accuracy_10", "recent_accuracy_change_5_vs_10", "recent_attempt_log_mean_10", "recent_multi_attempt_rate_10", "recent_hint_usage_rate_10", "recent_hint_available_rate_10", "recent_response_log_mean_10", "recent_response_median_ms_10", "recent_skill_accuracy_5", "recent_skill_accuracy_change_3_vs_previous_2", "has_overall_history", "has_skill_history", "has_full_skill_window_5", "has_recent_hint_usage_evidence"]
print("PROPOSED FINAL PREDICTORS")
print("=" * 70)
for i, col in enumerate(proposed_predictors, start=1):
    print(f"{i:2d}. {col}")
print()
print("Number of proposed predictors:", len(proposed_predictors))
corr_matrix = feature_ready_df[proposed_predictors].corr()
print()
print("HIGH ABSOLUTE CORRELATIONS (>= 0.80)")
print("=" * 70)
high_corr_pairs = []
for i in range(len(proposed_predictors)):
    for j in range(i + 1, len(proposed_predictors)):
        col_1 = proposed_predictors[i]
        col_2 = proposed_predictors[j]
        corr = corr_matrix.loc[col_1, col_2]
        if pd.notna(corr) and abs(corr) >= 0.80:
            high_corr_pairs.append({"feature_1": col_1, "feature_2": col_2, "correlation": corr})
high_corr_df = pd.DataFrame(high_corr_pairs)
if len(high_corr_df) == 0:
    print("No predictor pairs with |correlation| >= 0.80")
else:
    high_corr_df["absolute_correlation"] = high_corr_df["correlation"].abs()
    high_corr_df = high_corr_df.sort_values("absolute_correlation", ascending=False)
    print(high_corr_df[["feature_1", "feature_2", "correlation"]].round(3).to_string(index=False))
print()
print("PREDICTOR MISSINGNESS")
print("=" * 70)
missing_review = pd.DataFrame({"missing": feature_ready_df[proposed_predictors].isna().sum(), "percentage": feature_ready_df[proposed_predictors].isna().mean() * 100})
print(missing_review.sort_values("percentage", ascending=False).round(2))
print()
print("PREDICTOR RANGE SUMMARY")
print("=" * 70)
print(feature_ready_df[proposed_predictors].describe().T[["min", "mean", "50%", "max"]].round(3))

PROPOSED FINAL PREDICTORS
 1. previous_interaction_count
 2. previous_skill_interaction_count
 3. recent_accuracy_10
 4. recent_accuracy_change_5_vs_10
 5. recent_attempt_log_mean_10
 6. recent_multi_attempt_rate_10
 7. recent_hint_usage_rate_10
 8. recent_hint_available_rate_10
 9. recent_response_log_mean_10
10. recent_response_median_ms_10
11. recent_skill_accuracy_5
12. recent_skill_accuracy_change_3_vs_previous_2
13. has_overall_history
14. has_skill_history
15. has_full_skill_window_5
16. has_recent_hint_usage_evidence

Number of proposed predictors: 16



HIGH ABSOLUTE CORRELATIONS (>= 0.80)
                 feature_1                    feature_2  correlation
recent_attempt_log_mean_10 recent_multi_attempt_rate_10        0.823

PREDICTOR MISSINGNESS
                                              missing  percentage
recent_skill_accuracy_change_3_vs_previous_2   151085       44.70
recent_skill_accuracy_5                         41982       12.42
recent_hint_usage_rate_10                       16020        4.74
recent_accuracy_change_5_vs_10                   4940        1.46
recent_multi_attempt_rate_10                     4940        1.46
recent_hint_available_rate_10                    4940        1.46
recent_attempt_log_mean_10                       4940        1.46
recent_accuracy_10                               4940        1.46
recent_response_median_ms_10                     4940        1.46
recent_response_log_mean_10                      4940        1.46
previous_skill_interaction_count                    0        0.00
previous_

                                                min       mean        50%  \
previous_interaction_count                    0.000    201.332    109.000   
previous_skill_interaction_count              0.000     10.614      5.000   
recent_accuracy_10                            0.000      0.662      0.700   
recent_accuracy_change_5_vs_10               -0.500      0.004      0.000   
recent_attempt_log_mean_10                    0.000      0.823      0.774   
recent_multi_attempt_rate_10                  0.000      0.204      0.200   
recent_hint_usage_rate_10                     0.000      0.193      0.100   
recent_hint_available_rate_10                 0.000      0.716      0.800   
recent_response_log_mean_10                   0.693      9.880      9.905   
recent_response_median_ms_10                  1.000  27055.170  19531.000   
recent_skill_accuracy_5                       0.000      0.595      0.600   
recent_skill_accuracy_change_3_vs_previous_2 -1.000      0.030      0.000   

### 7.12 Final Feature Leakage and Integrity Validation

Before exporting the engineered feature dataset, the complete candidate predictor set is subjected to a final integrity and leakage review.

The validation confirms that:

- the unified dataset retains the expected Concept Interaction row count;
- interaction and concept identifiers remain structurally valid;
- the current interaction outcome is not included among the predictors;
- all proposed predictors exist;
- historical evidence indicators are internally consistent;
- bounded rate and accuracy features remain within their valid domains;
- concept-change and overall-change features remain within their expected ranges;
- count features are non-negative;
- missing historical values correspond to explicitly represented evidence limitations.

Passing these checks confirms that the engineered dataset is structurally ready for export and later learning-state development.

In [36]:
final_predictors = ["previous_interaction_count", "previous_skill_interaction_count", "recent_accuracy_10", "recent_accuracy_change_5_vs_10", "recent_attempt_log_mean_10", "recent_multi_attempt_rate_10", "recent_hint_usage_rate_10", "recent_hint_available_rate_10", "recent_response_log_mean_10", "recent_response_median_ms_10", "recent_skill_accuracy_5", "recent_skill_accuracy_change_3_vs_previous_2", "has_overall_history", "has_skill_history", "has_full_skill_window_5", "has_recent_hint_usage_evidence"]
validation_results = []
def add_check(name, passed):
    validation_results.append({"validation": name, "passed": bool(passed)})
add_check("Rows = 338001", len(feature_ready_df) == 338001)
add_check("Students = 4163", feature_ready_df["user_id"].nunique() == 4163)
add_check("Skills = 123", feature_ready_df["skill_id"].nunique() == 123)
add_check("order_id + skill_id unique", not feature_ready_df[["order_id", "skill_id"]].duplicated().any())
add_check("All 16 final predictors exist", all(col in feature_ready_df.columns for col in final_predictors))
add_check("Current correct is NOT a predictor", "correct" not in final_predictors)
current_interaction_fields = {"correct", "attempt_count", "hint_count", "hint_total", "ms_first_response", "bottom_hint", "first_action"}
add_check("No direct current-interaction behavioural field is a predictor", len(current_interaction_fields.intersection(final_predictors)) == 0)
add_check("previous_interaction_count non-negative", (feature_ready_df["previous_interaction_count"] >= 0).all())
add_check("previous_skill_interaction_count non-negative", (feature_ready_df["previous_skill_interaction_count"] >= 0).all())
bounded_01_features = ["recent_accuracy_10", "recent_multi_attempt_rate_10", "recent_hint_usage_rate_10", "recent_hint_available_rate_10", "recent_skill_accuracy_5"]
for col in bounded_01_features:
    values = feature_ready_df[col].dropna()
    add_check(f"{col} within [0,1]", values.between(0, 1).all())
overall_change = feature_ready_df["recent_accuracy_change_5_vs_10"].dropna()
add_check("recent_accuracy_change_5_vs_10 within [-0.5,0.5]", overall_change.between(-0.5, 0.5).all())
skill_change = feature_ready_df["recent_skill_accuracy_change_3_vs_previous_2"].dropna()
add_check("skill change within [-1,1]", skill_change.between(-1, 1).all())
add_check("Overall evidence indicator consistent", (feature_ready_df["has_overall_history"] == feature_ready_df["recent_accuracy_10"].notna().astype(int)).all())
add_check("Skill evidence indicator consistent", (feature_ready_df["has_skill_history"] == feature_ready_df["recent_skill_accuracy_5"].notna().astype(int)).all())
add_check("Full skill-window indicator consistent", (feature_ready_df["has_full_skill_window_5"] == feature_ready_df["recent_skill_accuracy_change_3_vs_previous_2"].notna().astype(int)).all())
add_check("Hint evidence indicator consistent", (feature_ready_df["has_recent_hint_usage_evidence"] == feature_ready_df["recent_hint_usage_rate_10"].notna().astype(int)).all())
indicator_columns = ["has_overall_history", "has_skill_history", "has_full_skill_window_5", "has_recent_hint_usage_evidence"]
for col in indicator_columns:
    add_check(f"{col} contains only 0/1", set(feature_ready_df[col].dropna().unique()).issubset({0, 1}))
validation_df = pd.DataFrame(validation_results)
print("FINAL FEATURE ENGINEERING VALIDATION")
print("=" * 75)
print(validation_df.to_string(index=False))
print()
print("Checks passed:", validation_df["passed"].sum(), "/", len(validation_df))
print()
all_passed = validation_df["passed"].all()
print("FINAL RESULT:", "PASS" if all_passed else "FAIL")

FINAL FEATURE ENGINEERING VALIDATION
                                                    validation  passed
                                                 Rows = 338001    True
                                               Students = 4163    True
                                                  Skills = 123    True
                                    order_id + skill_id unique    True
                                 All 16 final predictors exist    True
                            Current correct is NOT a predictor    True
No direct current-interaction behavioural field is a predictor    True
                       previous_interaction_count non-negative    True
                 previous_skill_interaction_count non-negative    True
                               recent_accuracy_10 within [0,1]    True
                     recent_multi_attempt_rate_10 within [0,1]    True
                        recent_hint_usage_rate_10 within [0,1]    True
                    recent_hint_availabl

### 7.13 Export Final Engineered Learning-State Feature Dataset

All historical feature engineering and leakage validation checks have passed.

The final dataset contains one row per student-concept interaction and combines:

- interaction and concept identifiers;
- the current interaction outcome for later target-development and evaluation;
- 16 validated historical candidate predictors;
- explicit evidence indicators for cold-start and limited-history situations.

Historical missing values are intentionally preserved at this stage because they represent meaningful absence of prior evidence rather than observed zero performance.

The exported dataset will serve as the feature-engineered input for the next phase of learning-state development.

In [37]:
from pathlib import Path

identifier_columns = ["order_id", "user_id", "assignment_id", "problem_id", "skill_id", "skill_name"]
outcome_columns = ["correct"]
final_predictors = ["previous_interaction_count", "previous_skill_interaction_count", "recent_accuracy_10", "recent_accuracy_change_5_vs_10", "recent_attempt_log_mean_10", "recent_multi_attempt_rate_10", "recent_hint_usage_rate_10", "recent_hint_available_rate_10", "recent_response_log_mean_10", "recent_response_median_ms_10", "recent_skill_accuracy_5", "recent_skill_accuracy_change_3_vs_previous_2", "has_overall_history", "has_skill_history", "has_full_skill_window_5", "has_recent_hint_usage_evidence"]
final_columns = identifier_columns + outcome_columns + final_predictors
final_feature_df = feature_ready_df[final_columns].copy()
print("FINAL ENGINEERED FEATURE DATASET")
print("=" * 70)
print("Shape:", final_feature_df.shape)
print("Rows:", len(final_feature_df))
print("Columns:", final_feature_df.shape[1])
print("Unique students:", final_feature_df["user_id"].nunique())
print("Unique skills:", final_feature_df["skill_id"].nunique())
print("Unique order_id:", final_feature_df["order_id"].nunique())
print("Duplicate order_id + skill_id:", final_feature_df[["order_id", "skill_id"]].duplicated().sum())
print()
print("Number of final predictors:", len(final_predictors))
print("Current correct is a predictor:", "correct" in final_predictors)
print()
print("Missing values in final dataset:")
print(final_feature_df.isna().sum().sort_values(ascending=False))
print()
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "learning_state_features.csv"
final_feature_df.to_csv(output_path, index=False)
print("Saved to:", output_path)
print("File exists:", output_path.exists())
print("File size (MB):", round(output_path.stat().st_size / (1024 ** 2), 2))
reloaded_feature_df = pd.read_csv(output_path, low_memory=False)
print()
print("RELOAD VALIDATION")
print("=" * 70)
print("Reloaded shape:", reloaded_feature_df.shape)
print("Shape preserved:", reloaded_feature_df.shape == final_feature_df.shape)
print("Column order preserved:", list(reloaded_feature_df.columns) == list(final_feature_df.columns))
print("Duplicate order_id + skill_id after reload:", reloaded_feature_df[["order_id", "skill_id"]].duplicated().sum())
print()
print("FINAL EXPORT RESULT:", "PASS" if (reloaded_feature_df.shape == final_feature_df.shape and list(reloaded_feature_df.columns) == list(final_feature_df.columns) and reloaded_feature_df[["order_id", "skill_id"]].duplicated().sum() == 0) else "FAIL")

FINAL ENGINEERED FEATURE DATASET
Shape: (338001, 23)
Rows: 338001
Columns: 23
Unique students: 4163
Unique skills: 123
Unique order_id: 283105
Duplicate order_id + skill_id: 0

Number of final predictors: 16
Current correct is a predictor: False

Missing values in final dataset:
recent_skill_accuracy_change_3_vs_previous_2    151085
recent_skill_accuracy_5                          41982
recent_hint_usage_rate_10                        16020
skill_name                                       12364
recent_accuracy_change_5_vs_10                    4940
recent_hint_available_rate_10                     4940
recent_multi_attempt_rate_10                      4940
recent_response_median_ms_10                      4940
recent_response_log_mean_10                       4940
recent_attempt_log_mean_10                        4940
recent_accuracy_10                                4940
order_id                                             0
user_id                                              0
probl

Saved to: D:\TEA_CUPS\RESEARCH_01\student_personalization_memory\data\processed\learning_state_features.csv
File exists: True
File size (MB): 52.58



RELOAD VALIDATION
Reloaded shape: (338001, 23)
Shape preserved: True
Column order preserved: True
Duplicate order_id + skill_id after reload: 0

FINAL EXPORT RESULT: PASS


In [38]:
import pandas as pd

feature_path = "../data/processed/learning_state_features.csv"
df = pd.read_csv(feature_path, low_memory=False)

print("=" * 70)
print("PHASE 5 — STEP 1: CORE PERFORMANCE DISTRIBUTIONS")
print("=" * 70)
print("\nDataset shape:", df.shape)
features = ["recent_accuracy_10", "recent_skill_accuracy_5"]
print("\nCORE PERFORMANCE SUMMARY")
print("=" * 70)
print(df[features].describe(percentiles=[0.01, 0.05, 0.10, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.75, 0.80, 0.90, 0.95, 0.99]).round(3).T)
print("\nMISSINGNESS")
print("=" * 70)
for col in features:
    missing = df[col].isna().sum()
    available = df[col].notna().sum()
    print(f"\n{col}")
    print("Available:", available)
    print("Missing:", missing)
    print("Available %:", round(available / len(df) * 100, 2))
print("\nRECENT OVERALL ACCURACY — VALUE DISTRIBUTION")
print("=" * 70)
print(df["recent_accuracy_10"].value_counts(dropna=False).sort_index())
print("\nRECENT SKILL ACCURACY — VALUE DISTRIBUTION")
print("=" * 70)
print(df["recent_skill_accuracy_5"].value_counts(dropna=False).sort_index())
print("\nCORRELATION WHEN BOTH ARE AVAILABLE")
print("=" * 70)
both = df[["recent_accuracy_10", "recent_skill_accuracy_5"]].dropna()
print("Interactions:", len(both))
print(both.corr().round(3))

PHASE 5 — STEP 1: CORE PERFORMANCE DISTRIBUTIONS

Dataset shape: (338001, 23)

CORE PERFORMANCE SUMMARY
                            count   mean    std  min  ...  90%  95%  99%  max
recent_accuracy_10       333061.0  0.662  0.253  0.0  ...  1.0  1.0  1.0  1.0
recent_skill_accuracy_5  296019.0  0.595  0.340  0.0  ...  1.0  1.0  1.0  1.0

[2 rows x 20 columns]

MISSINGNESS

recent_accuracy_10
Available: 333061
Missing: 4940
Available %: 98.54

recent_skill_accuracy_5
Available: 296019
Missing: 41982
Available %: 87.58

RECENT OVERALL ACCURACY — VALUE DISTRIBUTION
recent_accuracy_10
0.000000    11955
0.100000     5799
0.111111       75
0.125000      121
0.142857      161
0.166667      230
0.200000     8528
0.222222      163
0.250000      656
0.285714      179
0.300000    12736
0.333333     1127
0.375000      213
0.400000    19649
0.428571      304
0.444444      272
0.500000    31168
0.555556      340
0.571429      467
0.600000    40277
0.625000      477
0.666667     2301
0.700000    50376

In [39]:
print("=" * 70)
print("PHASE 5 — STEP 2: OVERALL VS CONCEPT PERFORMANCE")
print("=" * 70)
analysis = df[["recent_accuracy_10", "recent_skill_accuracy_5", "previous_skill_interaction_count"]].dropna(subset=["recent_accuracy_10", "recent_skill_accuracy_5"]).copy()
analysis["skill_performance_band"] = pd.cut(analysis["recent_skill_accuracy_5"], bins=[-0.001, 0.4, 0.8, 1.001], labels=["Low (<=0.40)", "Middle (>0.40 to <0.80)", "High (>=0.80)"], right=False)
print("\nINTERACTIONS BY CONCEPT PERFORMANCE BAND")
print("=" * 70)
band_counts = analysis["skill_performance_band"].value_counts().sort_index()
band_table = pd.DataFrame({"interactions": band_counts, "percentage": (band_counts / len(analysis) * 100).round(2)})
print(band_table)
print("\nOVERALL PERFORMANCE WITHIN EACH CONCEPT BAND")
print("=" * 70)
print(analysis.groupby("skill_performance_band", observed=True).agg(interactions=("recent_accuracy_10", "size"), mean_overall_accuracy=("recent_accuracy_10", "mean"), median_overall_accuracy=("recent_accuracy_10", "median"), mean_skill_accuracy=("recent_skill_accuracy_5", "mean"), median_skill_accuracy=("recent_skill_accuracy_5", "median"), avg_previous_skill_interactions=("previous_skill_interaction_count", "mean")).round(3))
print("\nOVERALL ACCURACY DISTRIBUTION WITHIN EACH CONCEPT BAND")
print("=" * 70)
print(analysis.groupby("skill_performance_band", observed=True)["recent_accuracy_10"].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]).round(3))
print("\nIMPORTANT DISAGREEMENT CASES")
print("=" * 70)
low_skill_high_overall = analysis[(analysis["recent_skill_accuracy_5"] <= 0.4) & (analysis["recent_accuracy_10"] >= 0.8)]
high_skill_low_overall = analysis[(analysis["recent_skill_accuracy_5"] >= 0.8) & (analysis["recent_accuracy_10"] <= 0.4)]
print("\nLow concept performance (<=0.40) but high overall performance (>=0.80):", len(low_skill_high_overall))
print("Percentage of analysed interactions:", round(len(low_skill_high_overall) / len(analysis) * 100, 2), "%")
print("\nHigh concept performance (>=0.80) but low overall performance (<=0.40):", len(high_skill_low_overall))
print("Percentage of analysed interactions:", round(len(high_skill_low_overall) / len(analysis) * 100, 2), "%")
print("\nExamples: LOW concept but HIGH overall")
print("=" * 70)
print(df.loc[low_skill_high_overall.head(10).index, ["user_id", "order_id", "skill_id", "skill_name", "previous_skill_interaction_count", "recent_accuracy_10", "recent_skill_accuracy_5"]].to_string(index=False))
print("\nExamples: HIGH concept but LOW overall")
print("=" * 70)
print(df.loc[high_skill_low_overall.head(10).index, ["user_id", "order_id", "skill_id", "skill_name", "previous_skill_interaction_count", "recent_accuracy_10", "recent_skill_accuracy_5"]].to_string(index=False))

PHASE 5 — STEP 2: OVERALL VS CONCEPT PERFORMANCE

INTERACTIONS BY CONCEPT PERFORMANCE BAND
                         interactions  percentage
skill_performance_band                           
Low (<=0.40)                    68922       23.28
Middle (>0.40 to <0.80)        101979       34.45
High (>=0.80)                  125118       42.27

OVERALL PERFORMANCE WITHIN EACH CONCEPT BAND
                         interactions  ...  avg_previous_skill_interactions
skill_performance_band                 ...                                 
Low (<=0.40)                    68922  ...                           10.096
Middle (>0.40 to <0.80)        101979  ...                           13.265
High (>=0.80)                  125118  ...                           12.299

[3 rows x 6 columns]

OVERALL ACCURACY DISTRIBUTION WITHIN EACH CONCEPT BAND
                            count   mean    std  min  ...  50%  75%  90%  max
skill_performance_band                                ...                    

In [40]:
import pandas as pd

feature_path = "data/processed/learning_state_features.csv"
df = pd.read_csv(feature_path, low_memory=False)

print("=" * 75)
print("PHASE 5 â€” STEP 3: TREND SIGNAL ANALYSIS")
print("=" * 75)

trend_features = [
    "recent_accuracy_change_5_vs_10",
    "recent_skill_accuracy_change_3_vs_previous_2"
]

print("\nTREND FEATURE SUMMARY")
print("=" * 75)
print(df[trend_features].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(3).T)

print("\nTREND FEATURE AVAILABILITY")
print("=" * 75)
for col in trend_features:
    available = df[col].notna().sum()
    missing = df[col].isna().sum()
    print(f"\n{col}")
    print("Available:", available)
    print("Missing:", missing)
    print("Available %:", round(available / len(df) * 100, 2))

print("\nOVERALL TREND VALUE DISTRIBUTION")
print("=" * 75)
print(df["recent_accuracy_change_5_vs_10"].value_counts(dropna=False).sort_index())

print("\nCONCEPT TREND VALUE DISTRIBUTION")
print("=" * 75)
print(df["recent_skill_accuracy_change_3_vs_previous_2"].value_counts(dropna=False).sort_index())

def trend_direction(x):
    if pd.isna(x):
        return "Unavailable"
    if x >= 0.20:
        return "Positive"
    if x <= -0.20:
        return "Negative"
    return "Small / Stable"

df_step3 = df.copy()
df_step3["overall_trend_direction"] = df_step3["recent_accuracy_change_5_vs_10"].apply(trend_direction)
df_step3["skill_trend_direction"] = df_step3["recent_skill_accuracy_change_3_vs_previous_2"].apply(trend_direction)

print("\nEXPLORATORY OVERALL TREND DIRECTIONS")
print("=" * 75)
overall_counts = df_step3["overall_trend_direction"].value_counts()
print(pd.DataFrame({"interactions": overall_counts, "percentage": (overall_counts / len(df_step3) * 100).round(2)}))

print("\nEXPLORATORY CONCEPT TREND DIRECTIONS")
print("=" * 75)
skill_counts = df_step3["skill_trend_direction"].value_counts()
print(pd.DataFrame({"interactions": skill_counts, "percentage": (skill_counts / len(df_step3) * 100).round(2)}))

print("\nWHEN BOTH TREND SIGNALS ARE AVAILABLE")
print("=" * 75)
both_trends = df_step3[df_step3["recent_accuracy_change_5_vs_10"].notna() & df_step3["recent_skill_accuracy_change_3_vs_previous_2"].notna()].copy()
print("Interactions:", len(both_trends))
print("Percentage:", round(len(both_trends) / len(df_step3) * 100, 2), "%")

print("\nOVERALL VS CONCEPT TREND DIRECTION")
print("=" * 75)
cross = pd.crosstab(both_trends["overall_trend_direction"], both_trends["skill_trend_direction"], margins=True)
print(cross)
same_direction = (both_trends["overall_trend_direction"] == both_trends["skill_trend_direction"]).sum()
print("\nExact exploratory direction agreement:", same_direction, "(", round(same_direction / len(both_trends) * 100, 2), "%)")

print("\nTREND BY RECENT CONCEPT PERFORMANCE")
print("=" * 75)
both_trends["skill_performance_band"] = pd.cut(both_trends["recent_skill_accuracy_5"], bins=[-0.001, 0.4, 0.8, 1.001], labels=["Low", "Middle", "High"], right=False)
print(pd.crosstab(both_trends["skill_performance_band"], both_trends["skill_trend_direction"], normalize="index").mul(100).round(2))



PHASE 5 — STEP 3: TREND SIGNAL ANALYSIS

TREND FEATURE SUMMARY
                                                 count   mean  ...    99%  max
recent_accuracy_change_5_vs_10                333061.0  0.004  ...  0.333  0.5
recent_skill_accuracy_change_3_vs_previous_2  186916.0  0.030  ...  1.000  1.0

[2 rows x 14 columns]

TREND FEATURE AVAILABILITY

recent_accuracy_change_5_vs_10
Available: 333061
Missing: 4940
Available %: 98.54

recent_skill_accuracy_change_3_vs_previous_2
Available: 186916
Missing: 151085
Available %: 55.3

OVERALL TREND VALUE DISTRIBUTION
recent_accuracy_change_5_vs_10
-0.500000     533
-0.444444       6
-0.400000    1255
-0.400000    1816
-0.375000       4
             ... 
 0.375000      36
 0.400000    2784
 0.444444      15
 0.500000     432
 NaN         4940
Name: count, Length: 84, dtype: int64

CONCEPT TREND VALUE DISTRIBUTION
recent_skill_accuracy_change_3_vs_previous_2
-1.000000      2881
-0.666667     11742
-0.500000      6074
-0.333333     29619
-0.16666

In [41]:
import pandas as pd
import numpy as np

feature_path = "data/processed/learning_state_features.csv"
df = pd.read_csv(feature_path, low_memory=False)

print("=" * 75)
print("PHASE 5 â€” STEP 4: ATTEMPT BEHAVIOUR ANALYSIS")
print("=" * 75)

attempt_features = ["recent_attempt_log_mean_10", "recent_multi_attempt_rate_10"]
performance_features = ["recent_accuracy_10", "recent_skill_accuracy_5"]

print("\nATTEMPT FEATURE SUMMARY")
print("=" * 75)
print(df[attempt_features].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(3))

print("\nATTEMPT FEATURE AVAILABILITY")
print("=" * 75)
for col in attempt_features:
    available = df[col].notna().sum()
    missing = df[col].isna().sum()
    print(f"\n{col}")
    print("Available:", available)
    print("Missing:", missing)
    print("Available %:", round(available / len(df) * 100, 2))

analysis_cols = attempt_features + performance_features
print("\nCORRELATIONS WITH PERFORMANCE")
print("=" * 75)
print(df[analysis_cols].corr().round(3))

attempt_analysis = df[df["recent_skill_accuracy_5"].notna()].copy()
attempt_analysis["skill_performance_band"] = pd.cut(
    attempt_analysis["recent_skill_accuracy_5"], bins=[-np.inf, 0.40, 0.80, np.inf],
    labels=["Low (<=0.40)", "Middle (>0.40 to <0.80)", "High (>=0.80)"], right=False
)
attempt_analysis.loc[attempt_analysis["recent_skill_accuracy_5"] <= 0.40, "skill_performance_band"] = "Low (<=0.40)"
attempt_analysis.loc[(attempt_analysis["recent_skill_accuracy_5"] > 0.40) & (attempt_analysis["recent_skill_accuracy_5"] < 0.80), "skill_performance_band"] = "Middle (>0.40 to <0.80)"
attempt_analysis.loc[attempt_analysis["recent_skill_accuracy_5"] >= 0.80, "skill_performance_band"] = "High (>=0.80)"

print("\nATTEMPT BEHAVIOUR BY RECENT CONCEPT PERFORMANCE")
print("=" * 75)
attempt_by_skill = attempt_analysis.groupby("skill_performance_band", observed=True).agg(
    interactions=("order_id", "size"), avg_recent_skill_accuracy=("recent_skill_accuracy_5", "mean"),
    avg_attempt_log=("recent_attempt_log_mean_10", "mean"), median_attempt_log=("recent_attempt_log_mean_10", "median"),
    avg_multi_attempt_rate=("recent_multi_attempt_rate_10", "mean"), median_multi_attempt_rate=("recent_multi_attempt_rate_10", "median")
).round(3)
print(attempt_by_skill)

overall_attempt_analysis = df[df["recent_accuracy_10"].notna()].copy()
overall_attempt_analysis["overall_performance_band"] = "Middle (>0.40 to <0.80)"
overall_attempt_analysis.loc[overall_attempt_analysis["recent_accuracy_10"] <= 0.40, "overall_performance_band"] = "Low (<=0.40)"
overall_attempt_analysis.loc[overall_attempt_analysis["recent_accuracy_10"] >= 0.80, "overall_performance_band"] = "High (>=0.80)"

print("\nATTEMPT BEHAVIOUR BY RECENT OVERALL PERFORMANCE")
print("=" * 75)
attempt_by_overall = overall_attempt_analysis.groupby("overall_performance_band").agg(
    interactions=("order_id", "size"), avg_recent_accuracy=("recent_accuracy_10", "mean"),
    avg_attempt_log=("recent_attempt_log_mean_10", "mean"), median_attempt_log=("recent_attempt_log_mean_10", "median"),
    avg_multi_attempt_rate=("recent_multi_attempt_rate_10", "mean"), median_multi_attempt_rate=("recent_multi_attempt_rate_10", "median")
).round(3)
print(attempt_by_overall)

quartile_data = df[df["recent_multi_attempt_rate_10"].notna()].copy()
quartile_data["multi_attempt_quartile"] = pd.qcut(quartile_data["recent_multi_attempt_rate_10"], q=4, duplicates="drop")
print("\nPERFORMANCE BY MULTI-ATTEMPT-RATE QUARTILE")
print("=" * 75)
quartile_summary = quartile_data.groupby("multi_attempt_quartile", observed=True).agg(
    interactions=("order_id", "size"), avg_multi_attempt_rate=("recent_multi_attempt_rate_10", "mean"),
    avg_recent_accuracy=("recent_accuracy_10", "mean"), avg_recent_skill_accuracy=("recent_skill_accuracy_5", "mean")
).round(3)
print(quartile_summary)

rate_data = df[df["recent_multi_attempt_rate_10"].notna()].copy()
rate_data["multi_attempt_group"] = pd.cut(
    rate_data["recent_multi_attempt_rate_10"], bins=[-0.001, 0.0, 0.20, 0.40, 0.60, 1.0],
    labels=["0%", ">0% to 20%", ">20% to 40%", ">40% to 60%", ">60%"], include_lowest=True
)
print("\nPERFORMANCE BY INTERPRETABLE MULTI-ATTEMPT GROUP")
print("=" * 75)
rate_summary = rate_data.groupby("multi_attempt_group", observed=True).agg(
    interactions=("order_id", "size"), avg_multi_attempt_rate=("recent_multi_attempt_rate_10", "mean"),
    avg_recent_accuracy=("recent_accuracy_10", "mean"), avg_recent_skill_accuracy=("recent_skill_accuracy_5", "mean")
).round(3)
print(rate_summary)

print("\n" + "=" * 75)
print("PHASE 5 â€” STEP 4 ANALYSIS COMPLETE")
print("=" * 75)



PHASE 5 — STEP 4: ATTEMPT BEHAVIOUR ANALYSIS

ATTEMPT FEATURE SUMMARY
       recent_attempt_log_mean_10  recent_multi_attempt_rate_10
count                  333061.000                    333061.000
mean                        0.823                         0.204
std                         0.171                         0.184
min                         0.000                         0.000
10%                         0.693                         0.000
25%                         0.693                         0.100
50%                         0.774                         0.200
75%                         0.884                         0.300
90%                         1.023                         0.500
95%                         1.130                         0.556
99%                         1.401                         0.800
max                         6.412                         1.000

ATTEMPT FEATURE AVAILABILITY

recent_attempt_log_mean_10
Available: 333061
Missing: 4940
Availabl

In [42]:
import pandas as pd
import numpy as np

feature_path = "data/processed/learning_state_features.csv"
df = pd.read_csv(feature_path, low_memory=False)

print("=" * 75)
print("PHASE 5 â€” STEP 5: HINT BEHAVIOUR ANALYSIS")
print("=" * 75)

hint_features = ["recent_hint_usage_rate_10", "recent_hint_available_rate_10"]
performance_features = ["recent_accuracy_10", "recent_skill_accuracy_5"]

print("\nHINT FEATURE SUMMARY")
print("=" * 75)
print(df[hint_features].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(3))

print("\nHINT FEATURE AVAILABILITY")
print("=" * 75)
for col in hint_features:
    available = df[col].notna().sum()
    missing = df[col].isna().sum()
    print(f"\n{col}")
    print("Available:", available)
    print("Missing:", missing)
    print("Available %:", round(available / len(df) * 100, 2))

print("\nHINT EVIDENCE INDICATOR")
print("=" * 75)
print(df["has_recent_hint_usage_evidence"].value_counts().sort_index())
print("Percentage with hint-usage evidence:", round(df["has_recent_hint_usage_evidence"].mean() * 100, 2), "%")

analysis_cols = hint_features + performance_features
print("\nCORRELATIONS WITH PERFORMANCE")
print("=" * 75)
print(df[analysis_cols].corr().round(3))

skill_hint = df[df["recent_skill_accuracy_5"].notna()].copy()
skill_hint["skill_performance_band"] = "Middle (>0.40 to <0.80)"
skill_hint.loc[skill_hint["recent_skill_accuracy_5"] <= 0.40, "skill_performance_band"] = "Low (<=0.40)"
skill_hint.loc[skill_hint["recent_skill_accuracy_5"] >= 0.80, "skill_performance_band"] = "High (>=0.80)"

print("\nHINT BEHAVIOUR BY RECENT CONCEPT PERFORMANCE")
print("=" * 75)
skill_hint_summary = skill_hint.groupby("skill_performance_band").agg(
    interactions=("order_id", "size"), avg_recent_skill_accuracy=("recent_skill_accuracy_5", "mean"),
    hint_usage_evidence_rate=("has_recent_hint_usage_evidence", "mean"), avg_hint_usage_rate=("recent_hint_usage_rate_10", "mean"),
    median_hint_usage_rate=("recent_hint_usage_rate_10", "median"), avg_hint_available_rate=("recent_hint_available_rate_10", "mean")
).round(3)
print(skill_hint_summary)

overall_hint = df[df["recent_accuracy_10"].notna()].copy()
overall_hint["overall_performance_band"] = "Middle (>0.40 to <0.80)"
overall_hint.loc[overall_hint["recent_accuracy_10"] <= 0.40, "overall_performance_band"] = "Low (<=0.40)"
overall_hint.loc[overall_hint["recent_accuracy_10"] >= 0.80, "overall_performance_band"] = "High (>=0.80)"

print("\nHINT BEHAVIOUR BY RECENT OVERALL PERFORMANCE")
print("=" * 75)
overall_hint_summary = overall_hint.groupby("overall_performance_band").agg(
    interactions=("order_id", "size"), avg_recent_accuracy=("recent_accuracy_10", "mean"),
    hint_usage_evidence_rate=("has_recent_hint_usage_evidence", "mean"), avg_hint_usage_rate=("recent_hint_usage_rate_10", "mean"),
    median_hint_usage_rate=("recent_hint_usage_rate_10", "median"), avg_hint_available_rate=("recent_hint_available_rate_10", "mean")
).round(3)
print(overall_hint_summary)

hint_group_data = df[df["recent_hint_usage_rate_10"].notna()].copy()
hint_group_data["hint_usage_group"] = pd.cut(
    hint_group_data["recent_hint_usage_rate_10"], bins=[-0.001, 0.0, 0.20, 0.40, 0.60, 1.0],
    labels=["0%", ">0% to 20%", ">20% to 40%", ">40% to 60%", ">60%"], include_lowest=True
)
print("\nPERFORMANCE BY INTERPRETABLE HINT-USAGE GROUP")
print("=" * 75)
hint_group_summary = hint_group_data.groupby("hint_usage_group", observed=True).agg(
    interactions=("order_id", "size"), avg_hint_usage_rate=("recent_hint_usage_rate_10", "mean"),
    avg_hint_available_rate=("recent_hint_available_rate_10", "mean"), avg_recent_accuracy=("recent_accuracy_10", "mean"),
    avg_recent_skill_accuracy=("recent_skill_accuracy_5", "mean")
).round(3)
print(hint_group_summary)

availability_data = df[df["recent_hint_available_rate_10"].notna()].copy()
availability_data["hint_availability_group"] = pd.cut(
    availability_data["recent_hint_available_rate_10"], bins=[-0.001, 0.20, 0.40, 0.60, 0.80, 1.0],
    labels=["<=20%", ">20% to 40%", ">40% to 60%", ">60% to 80%", ">80%"], include_lowest=True
)
print("\nPERFORMANCE BY HINT-EVIDENCE AVAILABILITY")
print("=" * 75)
availability_summary = availability_data.groupby("hint_availability_group", observed=True).agg(
    interactions=("order_id", "size"), avg_hint_available_rate=("recent_hint_available_rate_10", "mean"),
    avg_hint_usage_rate=("recent_hint_usage_rate_10", "mean"), avg_recent_accuracy=("recent_accuracy_10", "mean"),
    avg_recent_skill_accuracy=("recent_skill_accuracy_5", "mean")
).round(3)
print(availability_summary)

zero_usage = (df["recent_hint_usage_rate_10"] == 0) & (df["has_recent_hint_usage_evidence"] == 1)
missing_evidence = df["has_recent_hint_usage_evidence"] == 0

print("\nZERO HINT USAGE VS NO HINT EVIDENCE")
print("=" * 75)
print("Measured zero hint usage:", int(zero_usage.sum()))
print("Percentage of all interactions:", round(zero_usage.mean() * 100, 2), "%")
print("\nNo measurable hint-usage evidence:", int(missing_evidence.sum()))
print("Percentage of all interactions:", round(missing_evidence.mean() * 100, 2), "%")
print("\nZero measured usage and missing evidence are distinct:", bool((zero_usage & missing_evidence).sum() == 0))

comparison_rows = []
for label, mask in [("Measured zero hint usage", zero_usage), ("No measurable hint evidence", missing_evidence)]:
    subset = df.loc[mask]
    comparison_rows.append({
        "group": label, "interactions": len(subset), "avg_recent_accuracy": subset["recent_accuracy_10"].mean(),
        "avg_recent_skill_accuracy": subset["recent_skill_accuracy_5"].mean(),
        "avg_multi_attempt_rate": subset["recent_multi_attempt_rate_10"].mean()
    })
comparison = pd.DataFrame(comparison_rows)

print("\nPERFORMANCE: ZERO HINT USAGE VS NO HINT EVIDENCE")
print("=" * 75)
print(comparison.round(3).to_string(index=False))

print("\n" + "=" * 75)
print("PHASE 5 â€” STEP 5 ANALYSIS COMPLETE")
print("=" * 75)



PHASE 5 — STEP 5: HINT BEHAVIOUR ANALYSIS

HINT FEATURE SUMMARY
       recent_hint_usage_rate_10  recent_hint_available_rate_10
count                 321981.000                     333061.000
mean                       0.193                          0.716
std                        0.253                          0.278
min                        0.000                          0.000
10%                        0.000                          0.300
25%                        0.000                          0.500
50%                        0.100                          0.800
75%                        0.300                          1.000
90%                        0.571                          1.000
95%                        0.778                          1.000
99%                        1.000                          1.000
max                        1.000                          1.000

HINT FEATURE AVAILABILITY

recent_hint_usage_rate_10
Available: 321981
Missing: 16020
Available %: 95.2

In [43]:
import pandas as pd
import numpy as np

feature_path = "data/processed/learning_state_features.csv"
df = pd.read_csv(feature_path, low_memory=False)

print("=" * 75)
print("PHASE 5 â€” STEP 6: RESPONSE-TIME BEHAVIOUR ANALYSIS")
print("=" * 75)

response_features = ["recent_response_log_mean_10", "recent_response_median_ms_10"]
print("\nRESPONSE-TIME FEATURE SUMMARY")
print("=" * 75)
print(df[response_features].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(3))

print("\nRESPONSE-TIME FEATURE AVAILABILITY")
print("=" * 75)
for col in response_features:
    available = df[col].notna().sum()
    missing = df[col].isna().sum()
    print(f"\n{col}")
    print("Available:", available)
    print("Missing:", missing)
    print("Available %:", round(available / len(df) * 100, 2))

analysis_cols = ["recent_response_log_mean_10", "recent_response_median_ms_10", "recent_accuracy_10", "recent_skill_accuracy_5", "recent_multi_attempt_rate_10", "recent_hint_usage_rate_10"]
print("\nCORRELATIONS WITH LEARNING SIGNALS")
print("=" * 75)
print(df[analysis_cols].corr().round(3))

skill_response = df[df["recent_skill_accuracy_5"].notna()].copy()
skill_response["skill_performance_band"] = "Middle (>0.40 to <0.80)"
skill_response.loc[skill_response["recent_skill_accuracy_5"] <= 0.40, "skill_performance_band"] = "Low (<=0.40)"
skill_response.loc[skill_response["recent_skill_accuracy_5"] >= 0.80, "skill_performance_band"] = "High (>=0.80)"
print("\nRESPONSE TIME BY RECENT CONCEPT PERFORMANCE")
print("=" * 75)
skill_response_summary = skill_response.groupby("skill_performance_band").agg(
    interactions=("order_id", "size"), avg_recent_skill_accuracy=("recent_skill_accuracy_5", "mean"),
    avg_response_log=("recent_response_log_mean_10", "mean"), median_response_log=("recent_response_log_mean_10", "median"),
    avg_response_median_ms=("recent_response_median_ms_10", "mean"), median_response_median_ms=("recent_response_median_ms_10", "median")
).round(3)
print(skill_response_summary)

overall_response = df[df["recent_accuracy_10"].notna()].copy()
overall_response["overall_performance_band"] = "Middle (>0.40 to <0.80)"
overall_response.loc[overall_response["recent_accuracy_10"] <= 0.40, "overall_performance_band"] = "Low (<=0.40)"
overall_response.loc[overall_response["recent_accuracy_10"] >= 0.80, "overall_performance_band"] = "High (>=0.80)"
print("\nRESPONSE TIME BY RECENT OVERALL PERFORMANCE")
print("=" * 75)
overall_response_summary = overall_response.groupby("overall_performance_band").agg(
    interactions=("order_id", "size"), avg_recent_accuracy=("recent_accuracy_10", "mean"),
    avg_response_log=("recent_response_log_mean_10", "mean"), median_response_log=("recent_response_log_mean_10", "median"),
    avg_response_median_ms=("recent_response_median_ms_10", "mean"), median_response_median_ms=("recent_response_median_ms_10", "median")
).round(3)
print(overall_response_summary)

response_q = df[df["recent_response_log_mean_10"].notna()].copy()
response_q["response_log_quartile"] = pd.qcut(response_q["recent_response_log_mean_10"], q=4, labels=["Q1 fastest", "Q2", "Q3", "Q4 slowest"], duplicates="drop")
print("\nLEARNING SIGNALS BY RESPONSE-TIME QUARTILE")
print("=" * 75)
response_quartile_summary = response_q.groupby("response_log_quartile", observed=True).agg(
    interactions=("order_id", "size"), avg_response_log=("recent_response_log_mean_10", "mean"),
    median_response_ms=("recent_response_median_ms_10", "median"), avg_recent_accuracy=("recent_accuracy_10", "mean"),
    avg_recent_skill_accuracy=("recent_skill_accuracy_5", "mean"), avg_multi_attempt_rate=("recent_multi_attempt_rate_10", "mean"),
    avg_hint_usage_rate=("recent_hint_usage_rate_10", "mean")
).round(3)
print(response_quartile_summary)

response_seconds = df[df["recent_response_median_ms_10"].notna()].copy()
response_seconds["recent_response_median_seconds"] = response_seconds["recent_response_median_ms_10"] / 1000
print("\nRECENT MEDIAN RESPONSE TIME IN SECONDS")
print("=" * 75)
print(response_seconds["recent_response_median_seconds"].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

response_seconds["response_time_group"] = pd.cut(
    response_seconds["recent_response_median_seconds"], bins=[0, 10, 20, 30, 60, np.inf],
    labels=["<=10 sec", ">10 to 20 sec", ">20 to 30 sec", ">30 to 60 sec", ">60 sec"], include_lowest=True
)
print("\nLEARNING SIGNALS BY INTERPRETABLE RESPONSE-TIME GROUP")
print("=" * 75)
response_group_summary = response_seconds.groupby("response_time_group", observed=True).agg(
    interactions=("order_id", "size"), avg_median_response_seconds=("recent_response_median_seconds", "mean"),
    avg_recent_accuracy=("recent_accuracy_10", "mean"), avg_recent_skill_accuracy=("recent_skill_accuracy_5", "mean"),
    avg_multi_attempt_rate=("recent_multi_attempt_rate_10", "mean"), avg_hint_usage_rate=("recent_hint_usage_rate_10", "mean")
).round(3)
print(response_group_summary)

within_band = df[df["recent_skill_accuracy_5"].notna() & df["recent_response_log_mean_10"].notna()].copy()
within_band["skill_performance_band"] = "Middle (>0.40 to <0.80)"
within_band.loc[within_band["recent_skill_accuracy_5"] <= 0.40, "skill_performance_band"] = "Low (<=0.40)"
within_band.loc[within_band["recent_skill_accuracy_5"] >= 0.80, "skill_performance_band"] = "High (>=0.80)"
within_band["response_half"] = pd.qcut(within_band["recent_response_log_mean_10"], q=2, labels=["Faster half", "Slower half"])
print("\nFAST VS SLOW WITHIN CONCEPT-PERFORMANCE BANDS")
print("=" * 75)
within_band_summary = within_band.groupby(["skill_performance_band", "response_half"], observed=True).agg(
    interactions=("order_id", "size"), avg_recent_skill_accuracy=("recent_skill_accuracy_5", "mean"),
    avg_response_log=("recent_response_log_mean_10", "mean"), avg_multi_attempt_rate=("recent_multi_attempt_rate_10", "mean"),
    avg_hint_usage_rate=("recent_hint_usage_rate_10", "mean")
).round(3)
print(within_band_summary)

print("\n" + "=" * 75)
print("PHASE 5 â€” STEP 6 ANALYSIS COMPLETE")
print("=" * 75)



PHASE 5 — STEP 6: RESPONSE-TIME BEHAVIOUR ANALYSIS

RESPONSE-TIME FEATURE SUMMARY
       recent_response_log_mean_10  recent_response_median_ms_10
count                   333061.000                  3.330610e+05
mean                         9.880                  2.705517e+04
std                          0.791                  3.933031e+04
min                          0.693                  1.000000e+00
10%                          8.944                  6.678500e+03
25%                          9.416                  1.123450e+04
50%                          9.905                  1.953100e+04
75%                         10.382                  3.339100e+04
90%                         10.827                  5.400500e+04
95%                         11.096                  7.219000e+04
99%                         11.651                  1.294358e+05
max                         15.142                  1.442909e+07

RESPONSE-TIME FEATURE AVAILABILITY

recent_response_log_mean_10
Availabl

In [44]:
import pandas as pd
import numpy as np

feature_path = "data/processed/learning_state_features.csv"
df = pd.read_csv(feature_path, low_memory=False)

print("=" * 80)
print("PHASE 5 â€” STEP 7: COMBINED EVIDENCE PATTERN ANALYSIS")
print("=" * 80)

combined = df[df["recent_skill_accuracy_5"].notna() & df["recent_accuracy_10"].notna() & df["recent_multi_attempt_rate_10"].notna()].copy()
print("\nRows available for combined analysis:", len(combined))
print("Percentage of full dataset:", round(len(combined) / len(df) * 100, 2), "%")

combined["skill_band"] = "Middle"
combined.loc[combined["recent_skill_accuracy_5"] <= 0.40, "skill_band"] = "Low"
combined.loc[combined["recent_skill_accuracy_5"] >= 0.80, "skill_band"] = "High"
combined["overall_band"] = "Middle"
combined.loc[combined["recent_accuracy_10"] <= 0.40, "overall_band"] = "Low"
combined.loc[combined["recent_accuracy_10"] >= 0.80, "overall_band"] = "High"

combined["attempt_support_band"] = "Moderate"
combined.loc[combined["recent_multi_attempt_rate_10"] <= 0.20, "attempt_support_band"] = "Low"
combined.loc[combined["recent_multi_attempt_rate_10"] > 0.40, "attempt_support_band"] = "High"
combined["hint_support_band"] = "Unavailable"
hint_available = combined["recent_hint_usage_rate_10"].notna()
combined.loc[hint_available, "hint_support_band"] = "Moderate"
combined.loc[hint_available & (combined["recent_hint_usage_rate_10"] <= 0.20), "hint_support_band"] = "Low"
combined.loc[hint_available & (combined["recent_hint_usage_rate_10"] > 0.40), "hint_support_band"] = "High"

print("\nSKILL PERFORMANCE Ã— MULTI-ATTEMPT SUPPORT")
print("=" * 80)
print(pd.crosstab(combined["skill_band"], combined["attempt_support_band"], normalize="index").mul(100).round(2))
print("\nSKILL PERFORMANCE Ã— HINT SUPPORT")
print("=" * 80)
print(pd.crosstab(combined["skill_band"], combined["hint_support_band"], normalize="index").mul(100).round(2))

low_skill = combined["recent_skill_accuracy_5"] <= 0.40
high_skill = combined["recent_skill_accuracy_5"] >= 0.80
high_attempt_support = combined["recent_multi_attempt_rate_10"] > 0.40
low_attempt_support = combined["recent_multi_attempt_rate_10"] <= 0.20
high_hint_support = combined["recent_hint_usage_rate_10"] > 0.40
low_hint_support = combined["recent_hint_usage_rate_10"] <= 0.20
hint_measurable = combined["recent_hint_usage_rate_10"].notna()
strong_support_need_pattern = low_skill & high_attempt_support & high_hint_support
strong_independence_pattern = high_skill & low_attempt_support & low_hint_support & hint_measurable

print("\nSTRONG COMBINED PATTERNS")
print("=" * 80)
print("Low skill + high attempts + high hints:", strong_support_need_pattern.sum())
print("Percentage of analysed rows:", round(strong_support_need_pattern.mean() * 100, 2), "%")
print()
print("High skill + low attempts + low hints:", strong_independence_pattern.sum())
print("Percentage of analysed rows:", round(strong_independence_pattern.mean() * 100, 2), "%")

profile_rows = []
for label, mask in [("Low skill + high attempts + high hints", strong_support_need_pattern), ("High skill + low attempts + low hints", strong_independence_pattern)]:
    subset = combined.loc[mask]
    profile_rows.append({
        "profile": label, "interactions": len(subset), "avg_skill_accuracy": subset["recent_skill_accuracy_5"].mean(),
        "avg_overall_accuracy": subset["recent_accuracy_10"].mean(), "avg_multi_attempt_rate": subset["recent_multi_attempt_rate_10"].mean(),
        "avg_hint_usage_rate": subset["recent_hint_usage_rate_10"].mean(), "avg_overall_change": subset["recent_accuracy_change_5_vs_10"].mean(),
        "avg_skill_change": subset["recent_skill_accuracy_change_3_vs_previous_2"].mean()
    })
profile_summary = pd.DataFrame(profile_rows)
print("\nSTRONG PROFILE SUMMARY")
print("=" * 80)
print(profile_summary.round(3).to_string(index=False))

middle_skill = combined[(combined["recent_skill_accuracy_5"] > 0.40) & (combined["recent_skill_accuracy_5"] < 0.80)].copy()
print("\nMIDDLE CONCEPT-PERFORMANCE GROUP")
print("=" * 80)
print("Interactions:", len(middle_skill))
print(middle_skill[["recent_skill_accuracy_5", "recent_accuracy_10", "recent_multi_attempt_rate_10", "recent_hint_usage_rate_10", "recent_accuracy_change_5_vs_10", "recent_skill_accuracy_change_3_vs_previous_2"]].describe().round(3))

middle_skill["support_pattern"] = "Mixed / Moderate"
middle_skill.loc[(middle_skill["recent_multi_attempt_rate_10"] <= 0.20) & (middle_skill["recent_hint_usage_rate_10"] <= 0.20), "support_pattern"] = "Low support need"
middle_skill.loc[(middle_skill["recent_multi_attempt_rate_10"] > 0.40) | (middle_skill["recent_hint_usage_rate_10"] > 0.40), "support_pattern"] = "High support need"
print("\nMIDDLE PERFORMANCE BY SUPPORT PATTERN")
print("=" * 80)
middle_support_summary = middle_skill.groupby("support_pattern").agg(
    interactions=("order_id", "size"), avg_skill_accuracy=("recent_skill_accuracy_5", "mean"), avg_overall_accuracy=("recent_accuracy_10", "mean"),
    avg_multi_attempt_rate=("recent_multi_attempt_rate_10", "mean"), avg_hint_usage_rate=("recent_hint_usage_rate_10", "mean"),
    avg_overall_change=("recent_accuracy_change_5_vs_10", "mean"), avg_skill_change=("recent_skill_accuracy_change_3_vs_previous_2", "mean")
).round(3)
print(middle_support_summary)

low_skill_rows = combined[combined["recent_skill_accuracy_5"] <= 0.40].copy()
low_skill_rows["skill_trend_group"] = "Trend unavailable / small"
low_skill_rows.loc[low_skill_rows["recent_skill_accuracy_change_3_vs_previous_2"] >= 0.20, "skill_trend_group"] = "Positive concept trend"
low_skill_rows.loc[low_skill_rows["recent_skill_accuracy_change_3_vs_previous_2"] <= -0.20, "skill_trend_group"] = "Negative concept trend"
print("\nLOW CONCEPT PERFORMANCE BY TREND")
print("=" * 80)
low_trend_summary = low_skill_rows.groupby("skill_trend_group").agg(
    interactions=("order_id", "size"), avg_skill_accuracy=("recent_skill_accuracy_5", "mean"), avg_overall_accuracy=("recent_accuracy_10", "mean"),
    avg_multi_attempt_rate=("recent_multi_attempt_rate_10", "mean"), avg_hint_usage_rate=("recent_hint_usage_rate_10", "mean")
).round(3)
print(low_trend_summary)

high_skill_rows = combined[combined["recent_skill_accuracy_5"] >= 0.80].copy()
high_skill_rows["skill_trend_group"] = "Trend unavailable / small"
high_skill_rows.loc[high_skill_rows["recent_skill_accuracy_change_3_vs_previous_2"] >= 0.20, "skill_trend_group"] = "Positive concept trend"
high_skill_rows.loc[high_skill_rows["recent_skill_accuracy_change_3_vs_previous_2"] <= -0.20, "skill_trend_group"] = "Negative concept trend"
print("\nHIGH CONCEPT PERFORMANCE BY TREND")
print("=" * 80)
high_trend_summary = high_skill_rows.groupby("skill_trend_group").agg(
    interactions=("order_id", "size"), avg_skill_accuracy=("recent_skill_accuracy_5", "mean"), avg_overall_accuracy=("recent_accuracy_10", "mean"),
    avg_multi_attempt_rate=("recent_multi_attempt_rate_10", "mean"), avg_hint_usage_rate=("recent_hint_usage_rate_10", "mean")
).round(3)
print(high_trend_summary)

print("\n" + "=" * 80)
print("PHASE 5 â€” STEP 7 ANALYSIS COMPLETE")
print("=" * 80)



PHASE 5 — STEP 7: COMBINED EVIDENCE PATTERN ANALYSIS

Rows available for combined analysis: 296019
Percentage of full dataset: 87.58 %

SKILL PERFORMANCE × MULTI-ATTEMPT SUPPORT
attempt_support_band   High    Low  Moderate
skill_band                                  
High                   2.15  84.95     12.89
Low                   23.37  43.00     33.62
Middle                 8.13  61.67     30.20

SKILL PERFORMANCE × HINT SUPPORT
hint_support_band   High    Low  Moderate  Unavailable
skill_band                                            
High                4.82  79.97      9.81         5.40
Low                37.64  38.69     22.32         1.35
Middle             10.07  68.15     19.24         2.54

STRONG COMBINED PATTERNS
Low skill + high attempts + high hints: 9400
Percentage of analysed rows: 3.18 %

High skill + low attempts + low hints: 88636
Percentage of analysed rows: 29.94 %

STRONG PROFILE SUMMARY
                               profile  interactions  avg_skill_accuracy  

In [45]:
import pandas as pd
import numpy as np

feature_path = "data/processed/learning_state_features.csv"
df = pd.read_csv(feature_path, low_memory=False)
analysis_df = df.copy()

print("=" * 80)
print("PHASE 5 â€” STEP 8: EVIDENCE STRENGTH & COLD-START ANALYSIS")
print("=" * 80)

def assign_evidence_level(row):
    if row["previous_interaction_count"] == 0:
        return "1. Cold start"
    elif row["previous_skill_interaction_count"] == 0:
        return "2. Overall history only"
    elif row["previous_skill_interaction_count"] < 5:
        return "3. Partial skill history"
    return "4. Full skill evidence"

analysis_df["evidence_level"] = analysis_df.apply(assign_evidence_level, axis=1)
evidence_order = ["1. Cold start", "2. Overall history only", "3. Partial skill history", "4. Full skill evidence"]
analysis_df["evidence_level"] = pd.Categorical(analysis_df["evidence_level"], categories=evidence_order, ordered=True)

print("\nEVIDENCE-LEVEL DISTRIBUTION")
print("=" * 80)
evidence_counts = analysis_df["evidence_level"].value_counts(sort=False).rename("interactions").to_frame()
evidence_counts["percentage"] = (evidence_counts["interactions"] / len(analysis_df) * 100).round(2)
print(evidence_counts)
print("\nTotal interactions:", len(analysis_df))
print("All interactions assigned exactly one evidence level:", evidence_counts["interactions"].sum() == len(analysis_df))

print("\nHISTORY DEPTH BY EVIDENCE LEVEL")
print("=" * 80)
history_summary = analysis_df.groupby("evidence_level", observed=False).agg(
    interactions=("order_id", "size"), students=("user_id", "nunique"), avg_previous_interactions=("previous_interaction_count", "mean"),
    median_previous_interactions=("previous_interaction_count", "median"), avg_previous_skill_interactions=("previous_skill_interaction_count", "mean"),
    median_previous_skill_interactions=("previous_skill_interaction_count", "median")
).round(2)
print(history_summary)

print("\nFEATURE AVAILABILITY BY EVIDENCE LEVEL (%)")
print("=" * 80)
availability_features = ["recent_accuracy_10", "recent_accuracy_change_5_vs_10", "recent_attempt_log_mean_10", "recent_multi_attempt_rate_10", "recent_hint_usage_rate_10", "recent_hint_available_rate_10", "recent_response_log_mean_10", "recent_response_median_ms_10", "recent_skill_accuracy_5", "recent_skill_accuracy_change_3_vs_previous_2"]
availability_table = {}
for feature in availability_features:
    availability_table[feature] = analysis_df.groupby("evidence_level", observed=False)[feature].apply(lambda x: x.notna().mean() * 100)
availability_table = pd.DataFrame(availability_table).round(2)
print(availability_table)

print("\nPERFORMANCE BY EVIDENCE LEVEL")
print("=" * 80)
performance_summary = analysis_df.groupby("evidence_level", observed=False).agg(
    interactions=("order_id", "size"), avg_recent_accuracy=("recent_accuracy_10", "mean"), avg_recent_skill_accuracy=("recent_skill_accuracy_5", "mean"),
    avg_overall_change=("recent_accuracy_change_5_vs_10", "mean"), avg_skill_change=("recent_skill_accuracy_change_3_vs_previous_2", "mean")
).round(3)
print(performance_summary)

print("\nSUPPORT BEHAVIOUR BY EVIDENCE LEVEL")
print("=" * 80)
support_summary = analysis_df.groupby("evidence_level", observed=False).agg(
    interactions=("order_id", "size"), avg_attempt_log=("recent_attempt_log_mean_10", "mean"), avg_multi_attempt_rate=("recent_multi_attempt_rate_10", "mean"),
    avg_hint_usage_rate=("recent_hint_usage_rate_10", "mean"), avg_hint_available_rate=("recent_hint_available_rate_10", "mean"),
    avg_response_log=("recent_response_log_mean_10", "mean"), median_response_ms=("recent_response_median_ms_10", "median")
).round(3)
print(support_summary)

print("\nCOLD-START STRUCTURE")
print("=" * 80)
cold_start = analysis_df[analysis_df["evidence_level"] == "1. Cold start"]
print("Cold-start interactions:", len(cold_start))
print("Unique students:", cold_start["user_id"].nunique())
print("All have previous_interaction_count = 0:", (cold_start["previous_interaction_count"] == 0).all())
print("All have previous_skill_interaction_count = 0:", (cold_start["previous_skill_interaction_count"] == 0).all())
print("Recent overall accuracy unavailable:", cold_start["recent_accuracy_10"].isna().all())
print("Recent skill accuracy unavailable:", cold_start["recent_skill_accuracy_5"].isna().all())
print("Skill trend unavailable:", cold_start["recent_skill_accuracy_change_3_vs_previous_2"].isna().all())

print("\nOVERALL-HISTORY-ONLY STRUCTURE")
print("=" * 80)
overall_only = analysis_df[analysis_df["evidence_level"] == "2. Overall history only"]
print("Interactions:", len(overall_only))
print("Unique students:", overall_only["user_id"].nunique())
print("All have previous overall history:", (overall_only["previous_interaction_count"] > 0).all())
print("All have zero previous skill history:", (overall_only["previous_skill_interaction_count"] == 0).all())
print("Recent overall accuracy available:", overall_only["recent_accuracy_10"].notna().mean().round(4))
print("Recent skill accuracy unavailable:", overall_only["recent_skill_accuracy_5"].isna().all())

print("\nPARTIAL SKILL-HISTORY STRUCTURE")
print("=" * 80)
partial_skill = analysis_df[analysis_df["evidence_level"] == "3. Partial skill history"]
print("Interactions:", len(partial_skill))
print("Previous skill-history range:", partial_skill["previous_skill_interaction_count"].min(), "to", partial_skill["previous_skill_interaction_count"].max())
print("Recent skill accuracy availability %:", round(partial_skill["recent_skill_accuracy_5"].notna().mean() * 100, 2))
print("Skill-change availability %:", round(partial_skill["recent_skill_accuracy_change_3_vs_previous_2"].notna().mean() * 100, 2))

print("\nFULL SKILL-EVIDENCE STRUCTURE")
print("=" * 80)
full_skill = analysis_df[analysis_df["evidence_level"] == "4. Full skill evidence"]
print("Interactions:", len(full_skill))
print("Minimum previous skill interactions:", full_skill["previous_skill_interaction_count"].min())
print("Recent skill accuracy availability %:", round(full_skill["recent_skill_accuracy_5"].notna().mean() * 100, 2))
print("Skill-change availability %:", round(full_skill["recent_skill_accuracy_change_3_vs_previous_2"].notna().mean() * 100, 2))

print("\nEVIDENCE CAPABILITY MATRIX")
print("=" * 80)
capability_matrix = pd.DataFrame({
    "Evidence level": evidence_order, "Overall history": ["No", "Yes", "Yes", "Yes"], "Skill history": ["No", "No", "Partial", "Yes"],
    "Overall performance evidence": ["No", "Yes", "Yes", "Yes"], "Skill performance evidence": ["No", "No", "Yes", "Yes"],
    "Skill trend evidence": ["No", "No", "No", "Yes"]
})
print(capability_matrix.to_string(index=False))

print("\n" + "=" * 80)
print("PHASE 5 â€” STEP 8 ANALYSIS COMPLETE")
print("=" * 80)



PHASE 5 — STEP 8: EVIDENCE STRENGTH & COLD-START ANALYSIS

EVIDENCE-LEVEL DISTRIBUTION
                          interactions  percentage
evidence_level                                    
1. Cold start                     4940        1.46
2. Overall history only          37042       10.96
3. Partial skill history        109103       32.28
4. Full skill evidence          186916       55.30

Total interactions: 338001
All interactions assigned exactly one evidence level: True

HISTORY DEPTH BY EVIDENCE LEVEL
                          interactions  ...  median_previous_skill_interactions
evidence_level                          ...                                    
1. Cold start                     4940  ...                                 0.0
2. Overall history only          37042  ...                                 0.0
3. Partial skill history        109103  ...                                 2.0
4. Full skill evidence          186916  ...                                12.0

[4 row

In [46]:
import pandas as pd
import numpy as np

feature_path = "data/processed/learning_state_features.csv"
df = pd.read_csv(feature_path, low_memory=False)
analysis_df = df.copy()

print("=" * 80)
print("PHASE 5 â€” STEP 9: CANDIDATE 3-STATE DEFINITION ANALYSIS")
print("=" * 80)

def evidence_level(row):
    if row["previous_interaction_count"] == 0:
        return "COLD_START"
    elif row["previous_skill_interaction_count"] == 0:
        return "OVERALL_ONLY"
    elif row["previous_skill_interaction_count"] < 5:
        return "PARTIAL_SKILL"
    return "FULL_SKILL"

analysis_df["evidence_level"] = analysis_df.apply(evidence_level, axis=1)
analysis_df["primary_performance"] = np.where(analysis_df["has_skill_history"] == 1, analysis_df["recent_skill_accuracy_5"], analysis_df["recent_accuracy_10"])
analysis_df.loc[analysis_df["evidence_level"] == "COLD_START", "primary_performance"] = np.nan

print("\nPRIMARY PERFORMANCE SOURCE")
print("=" * 80)
analysis_df["performance_source"] = np.select(
    [analysis_df["evidence_level"] == "COLD_START", analysis_df["has_skill_history"] == 1],
    ["Unavailable", "Skill"], default="Overall"
)
print(analysis_df["performance_source"].value_counts(dropna=False))

def base_state(value):
    if pd.isna(value): return "UNAVAILABLE"
    elif value <= 0.40: return "NEEDS_SUPPORT"
    elif value >= 0.80: return "STRONG"
    return "DEVELOPING"

analysis_df["base_state"] = analysis_df["primary_performance"].apply(base_state)
print("\nBASE PERFORMANCE STATE DISTRIBUTION")
print("=" * 80)
base_counts = analysis_df["base_state"].value_counts()
base_table = pd.DataFrame({"interactions": base_counts, "percentage": (base_counts / len(analysis_df) * 100).round(2)})
print(base_table)

def support_pattern(row):
    attempt = row["recent_multi_attempt_rate_10"]
    hint = row["recent_hint_usage_rate_10"]
    if pd.isna(attempt): return "UNAVAILABLE"
    high_attempt = attempt > 0.40
    low_attempt = attempt <= 0.20
    if pd.isna(hint):
        if high_attempt: return "HIGH_SUPPORT_SIGNAL"
        elif low_attempt: return "LOW_SUPPORT_SIGNAL"
        return "MIXED_SUPPORT_SIGNAL"
    high_hint = hint > 0.40
    low_hint = hint <= 0.20
    if high_attempt and high_hint: return "HIGH_SUPPORT_SIGNAL"
    elif low_attempt and low_hint: return "LOW_SUPPORT_SIGNAL"
    return "MIXED_SUPPORT_SIGNAL"

analysis_df["support_pattern"] = analysis_df.apply(support_pattern, axis=1)
print("\nSUPPORT-PATTERN DISTRIBUTION")
print("=" * 80)
support_counts = analysis_df["support_pattern"].value_counts()
support_table = pd.DataFrame({"interactions": support_counts, "percentage": (support_counts / len(analysis_df) * 100).round(2)})
print(support_table)

analysis_df["selected_trend"] = np.where(analysis_df["has_full_skill_window_5"] == 1, analysis_df["recent_skill_accuracy_change_3_vs_previous_2"], analysis_df["recent_accuracy_change_5_vs_10"])
def trend_direction(value):
    if pd.isna(value): return "UNAVAILABLE"
    elif value >= 0.20: return "POSITIVE"
    elif value <= -0.20: return "NEGATIVE"
    return "SMALL_STABLE"

analysis_df["selected_trend_direction"] = analysis_df["selected_trend"].apply(trend_direction)
print("\nSELECTED TREND DISTRIBUTION")
print("=" * 80)
trend_counts = analysis_df["selected_trend_direction"].value_counts()
trend_table = pd.DataFrame({"interactions": trend_counts, "percentage": (trend_counts / len(analysis_df) * 100).round(2)})
print(trend_table)

print("\nBASE STATE Ã— SUPPORT PATTERN")
print("=" * 80)
print(pd.crosstab(analysis_df["base_state"], analysis_df["support_pattern"]))
print("\nROW PERCENTAGES")
print("=" * 80)
print(pd.crosstab(analysis_df["base_state"], analysis_df["support_pattern"], normalize="index").mul(100).round(2))

print("\nBASE STATE Ã— SELECTED TREND")
print("=" * 80)
print(pd.crosstab(analysis_df["base_state"], analysis_df["selected_trend_direction"]))
print("\nROW PERCENTAGES")
print("=" * 80)
print(pd.crosstab(analysis_df["base_state"], analysis_df["selected_trend_direction"], normalize="index").mul(100).round(2))

high_conf_support = (analysis_df["base_state"] == "NEEDS_SUPPORT") & (analysis_df["support_pattern"] == "HIGH_SUPPORT_SIGNAL")
high_conf_strong = (analysis_df["base_state"] == "STRONG") & (analysis_df["support_pattern"] == "LOW_SUPPORT_SIGNAL")
developing_positive = (analysis_df["base_state"] == "DEVELOPING") & (analysis_df["selected_trend_direction"] == "POSITIVE")
developing_negative = (analysis_df["base_state"] == "DEVELOPING") & (analysis_df["selected_trend_direction"] == "NEGATIVE")
print("\nHIGH-CONFIDENCE / IMPORTANT PROFILES")
print("=" * 80)
profile_table = pd.DataFrame({"profile": ["Needs support + high support signal", "Strong + low support signal", "Developing + positive trend", "Developing + negative trend"], "interactions": [high_conf_support.sum(), high_conf_strong.sum(), developing_positive.sum(), developing_negative.sum()]})
profile_table["percentage_of_all"] = (profile_table["interactions"] / len(analysis_df) * 100).round(2)
print(profile_table.to_string(index=False))

strong_but_high_support = (analysis_df["base_state"] == "STRONG") & (analysis_df["support_pattern"] == "HIGH_SUPPORT_SIGNAL")
low_but_low_support = (analysis_df["base_state"] == "NEEDS_SUPPORT") & (analysis_df["support_pattern"] == "LOW_SUPPORT_SIGNAL")
print("\nCONTRADICTORY PROFILES")
print("=" * 80)
contradiction_table = pd.DataFrame({"profile": ["Strong performance + high support signal", "Low performance + low support signal"], "interactions": [strong_but_high_support.sum(), low_but_low_support.sum()]})
contradiction_table["percentage_of_all"] = (contradiction_table["interactions"] / len(analysis_df) * 100).round(2)
print(contradiction_table.to_string(index=False))

print("\nBASE STATE BY EVIDENCE LEVEL")
print("=" * 80)
print(pd.crosstab(analysis_df["evidence_level"], analysis_df["base_state"]))
print("\nROW PERCENTAGES")
print("=" * 80)
print(pd.crosstab(analysis_df["evidence_level"], analysis_df["base_state"], normalize="index").mul(100).round(2))

print("\nLEARNING SIGNAL SUMMARY BY BASE STATE")
print("=" * 80)
state_summary = analysis_df.groupby("base_state").agg(
    interactions=("order_id", "size"), avg_primary_performance=("primary_performance", "mean"), avg_overall_accuracy=("recent_accuracy_10", "mean"),
    avg_skill_accuracy=("recent_skill_accuracy_5", "mean"), avg_multi_attempt_rate=("recent_multi_attempt_rate_10", "mean"),
    avg_hint_usage_rate=("recent_hint_usage_rate_10", "mean"), avg_selected_trend=("selected_trend", "mean"), median_response_ms=("recent_response_median_ms_10", "median")
).round(3)
print(state_summary)

print("\nSANITY CHECKS")
print("=" * 80)
print("Cold-start rows are UNAVAILABLE:", (analysis_df.loc[analysis_df["evidence_level"] == "COLD_START", "base_state"] == "UNAVAILABLE").all())
print("Non-cold-start rows have a base state:", (analysis_df.loc[analysis_df["evidence_level"] != "COLD_START", "base_state"] != "UNAVAILABLE").all())
print("All rows accounted for:", len(analysis_df) == analysis_df["base_state"].value_counts().sum())

print("\n" + "=" * 80)
print("PHASE 5 â€” STEP 9 ANALYSIS COMPLETE")
print("=" * 80)



PHASE 5 — STEP 9: CANDIDATE 3-STATE DEFINITION ANALYSIS

PRIMARY PERFORMANCE SOURCE
performance_source
Skill          296019
Overall         37042
Unavailable      4940
Name: count, dtype: int64

BASE PERFORMANCE STATE DISTRIBUTION
               interactions  percentage
base_state                             
STRONG               145457       43.03
NEEDS_SUPPORT        100645       29.78
DEVELOPING            86959       25.73
UNAVAILABLE            4940        1.46

SUPPORT-PATTERN DISTRIBUTION
                      interactions  percentage
support_pattern                               
LOW_SUPPORT_SIGNAL          172421       51.01
MIXED_SUPPORT_SIGNAL        147746       43.71
HIGH_SUPPORT_SIGNAL          12894        3.81
UNAVAILABLE                   4940        1.46

SELECTED TREND DISTRIBUTION
                          interactions  percentage
selected_trend_direction                          
SMALL_STABLE                    209577       62.00
NEGATIVE                         6

In [47]:
import pandas as pd
import numpy as np
df=pd.read_csv("data/processed/learning_state_features.csv",low_memory=False); a=df.copy()
def ev(r):
 if r.previous_interaction_count==0:return "COLD_START"
 if r.previous_skill_interaction_count==0:return "OVERALL_ONLY"
 if r.previous_skill_interaction_count<5:return "PARTIAL_SKILL"
 return "FULL_SKILL"
a["evidence_level"]=a.apply(ev,axis=1)
a["primary_performance"]=np.where(a.has_skill_history==1,a.recent_skill_accuracy_5,a.recent_accuracy_10);a.loc[a.evidence_level=="COLD_START","primary_performance"]=np.nan
a["performance_source"]=np.select([a.evidence_level=="COLD_START",a.has_skill_history==1],["Unavailable","Skill"],default="Overall")
def sp(r):
 x,h=r.recent_multi_attempt_rate_10,r.recent_hint_usage_rate_10
 if pd.isna(x):return "UNAVAILABLE"
 hi,lo=x>.4,x<=.2
 if pd.isna(h):return "HIGH_SUPPORT_SIGNAL" if hi else "LOW_SUPPORT_SIGNAL" if lo else "MIXED_SUPPORT_SIGNAL"
 return "HIGH_SUPPORT_SIGNAL" if hi and h>.4 else "LOW_SUPPORT_SIGNAL" if lo and h<=.2 else "MIXED_SUPPORT_SIGNAL"
a["support_pattern"]=a.apply(sp,axis=1);a["selected_trend"]=np.where(a.has_full_skill_window_5==1,a.recent_skill_accuracy_change_3_vs_previous_2,a.recent_accuracy_change_5_vs_10)
a["trend_direction"]=a.selected_trend.apply(lambda x:"UNAVAILABLE" if pd.isna(x) else "POSITIVE" if x>=.2 else "NEGATIVE" if x<=-.2 else "SMALL_STABLE")
v=a[a.primary_performance.notna()].copy()
def reg(x):
 return "A. Very low <=0.20" if x<=.2 else "B. Low >0.20 to 0.40" if x<=.4 else "C. Lower-middle >0.40 to <0.60" if x<.6 else "D. Upper-middle 0.60 to <0.80" if x<.8 else "E. High 0.80 to <1.00" if x<1 else "F. Perfect 1.00"
order=["A. Very low <=0.20","B. Low >0.20 to 0.40","C. Lower-middle >0.40 to <0.60","D. Upper-middle 0.60 to <0.80","E. High 0.80 to <1.00","F. Perfect 1.00"]
v["performance_region"]=pd.Categorical(v.primary_performance.apply(reg),categories=order,ordered=True)
print("="*80,"\nPHASE 5 â€” STEP 10: BOUNDARY & CONTRADICTION VALIDATION\n","="*80,sep="")
print("\nRows with performance evidence:",len(v));print("Percentage of full dataset:",round(len(v)/len(a)*100,2),"%")
print("\nPRIMARY PERFORMANCE VALUE DISTRIBUTION\n"+"="*80);print(v.primary_performance.value_counts().sort_index())
print("\nPERFORMANCE-REGION DISTRIBUTION\n"+"="*80);rc=v.performance_region.value_counts(sort=False).rename("interactions").to_frame();rc["percentage"]=(rc.interactions/len(v)*100).round(2);print(rc)
print("\nBEHAVIOUR ACROSS PERFORMANCE REGIONS\n"+"="*80);print(v.groupby("performance_region",observed=False).agg(interactions=("order_id","size"),avg_primary_performance=("primary_performance","mean"),avg_multi_attempt_rate=("recent_multi_attempt_rate_10","mean"),median_multi_attempt_rate=("recent_multi_attempt_rate_10","median"),avg_hint_usage_rate=("recent_hint_usage_rate_10","mean"),median_hint_usage_rate=("recent_hint_usage_rate_10","median"),avg_selected_trend=("selected_trend","mean"),avg_response_median_ms=("recent_response_median_ms_10","mean")).round(3))
print("\nSUPPORT PATTERN WITHIN PERFORMANCE REGIONS (%)\n"+"="*80);print(pd.crosstab(v.performance_region,v.support_pattern,normalize="index").mul(100).round(2))
print("\nTREND WITHIN PERFORMANCE REGIONS (%)\n"+"="*80);print(pd.crosstab(v.performance_region,v.trend_direction,normalize="index").mul(100).round(2))
def boundary(lo,hi,bins,labels,title):
 b=v[v.primary_performance.between(lo,hi,inclusive="both")].copy();b["boundary_group"]=pd.cut(b.primary_performance,bins=bins,labels=labels,right=True)
 print("\n"+title+"\n"+"="*80);print(b.groupby("boundary_group",observed=False).agg(interactions=("order_id","size"),avg_performance=("primary_performance","mean"),avg_attempt_rate=("recent_multi_attempt_rate_10","mean"),avg_hint_rate=("recent_hint_usage_rate_10","mean"),avg_trend=("selected_trend","mean")).round(3));print("\nSUPPORT PATTERN AROUND "+title[:4]+" (%)\n"+"="*80);print(pd.crosstab(b.boundary_group,b.support_pattern,normalize="index").mul(100).round(2))
boundary(.3,.6,[-np.inf,.4,.59,np.inf],["<=0.40",">0.40 to <0.60",">=0.60"],"0.40 BOUNDARY ANALYSIS")
boundary(.6,1,[-np.inf,.79,.99,np.inf],["<0.80","0.80 to <1.00","1.00"],"0.80 BOUNDARY ANALYSIS")
ll=v[(v.primary_performance<=.4)&(v.support_pattern=="LOW_SUPPORT_SIGNAL")];sh=v[(v.primary_performance>=.8)&(v.support_pattern=="HIGH_SUPPORT_SIGNAL")]
for title,s in [("LOW PERFORMANCE + LOW SUPPORT ANALYSIS",ll),("STRONG PERFORMANCE + HIGH SUPPORT ANALYSIS",sh)]:
 print("\n"+title+"\n"+"="*80);print("Interactions:",len(s));print("Percentage of available interactions:",round(len(s)/len(v)*100,2),"%\n\nPERFORMANCE SUMMARY");print(s[["primary_performance","recent_accuracy_10","recent_skill_accuracy_5","recent_multi_attempt_rate_10","recent_hint_usage_rate_10","selected_trend"]].describe(percentiles=[.1,.25,.5,.75,.9]).round(3));print("\nTREND DISTRIBUTION");print(s.trend_direction.value_counts(normalize=True).mul(100).round(2));print("\nEVIDENCE-LEVEL DISTRIBUTION");print(s.evidence_level.value_counts(normalize=True).mul(100).round(2))
print("\nPERFORMANCE SOURCE COMPARISON\n"+"="*80);print(v.groupby("performance_source").agg(interactions=("order_id","size"),avg_primary_performance=("primary_performance","mean"),avg_attempt_rate=("recent_multi_attempt_rate_10","mean"),avg_hint_rate=("recent_hint_usage_rate_10","mean"),avg_trend=("selected_trend","mean")).round(3));print("\nPERFORMANCE REGIONS BY SOURCE (%)\n"+"="*80);print(pd.crosstab(v.performance_source,v.performance_region,normalize="index").mul(100).round(2))
d=pd.DataFrame({"diagnostic_group":["Primary performance <=0.40","Primary performance >0.40 and <0.80","Primary performance >=0.80","Low performance + low support","Strong performance + high support"],"interactions":[(v.primary_performance<=.4).sum(),((v.primary_performance>.4)&(v.primary_performance<.8)).sum(),(v.primary_performance>=.8).sum(),len(ll),len(sh)]});d["percentage_of_available"]=(d.interactions/len(v)*100).round(2)
print("\nFINAL DIAGNOSTIC COUNTS\n"+"="*80);print(d.to_string(index=False));print("\nSANITY CHECKS\n"+"="*80);print("Available + cold start = full dataset:",len(v)+(a.evidence_level=="COLD_START").sum()==len(a));print("Performance regions cover all available rows:",v.performance_region.notna().all());print("Low-support contradiction obeys <=0.40:",(ll.primary_performance<=.4).all());print("Strong/high-support contradiction obeys >=0.80:",(sh.primary_performance>=.8).all());print("\n"+"="*80+"\nPHASE 5 â€” STEP 10 ANALYSIS COMPLETE\n"+"="*80)



PHASE 5 — STEP 10: BOUNDARY & CONTRADICTION VALIDATION

Rows with performance evidence: 333061
Percentage of full dataset: 98.54 %

PRIMARY PERFORMANCE VALUE DISTRIBUTION
primary_performance
0.000000    45287
0.100000      417
0.111111        9
0.125000       13
0.142857       34
0.166667       30
0.200000    16667
0.222222       14
0.250000     3060
0.285714       27
0.300000      914
0.333333     5679
0.375000       26
0.400000    28468
0.428571       51
0.444444       37
0.500000    18492
0.555556       39
0.571429       73
0.600000    46721
0.625000       60
0.666667     9384
0.700000     4829
0.714286      110
0.750000     7076
0.777778       87
0.800000    59989
0.833333      117
0.857143      109
0.875000      102
0.888889       94
0.900000     6355
1.000000    78691
Name: count, dtype: int64

PERFORMANCE-REGION DISTRIBUTION
                                interactions  percentage
performance_region                                      
A. Very low <=0.20                     624